<a href="https://colab.research.google.com/github/artkula/ML-retreat-tekmek-2026/blob/main/3-LLM/LLM_clinical_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

heading.svg

## Why This Matters

**Real-world challenge**: Medical data arrives from diverse sources: hospital systems in different countries, legacy databases, sensor networks, manual entry forms. Each uses different units (kg vs lbs), languages (English vs German), date formats (DD/MM vs MM/DD), and quality standards. Manual standardization is error-prone, time-consuming, and does not scale.

**Traditional limitations**: Rule-based systems break when encountering new formats. Statistical methods cannot understand that "Größe: 180" means height in German or that "6'2"" needs conversion to centimeters. They see syntax, not semantics.

**Key insight**: LLMs combine pattern recognition with domain knowledge. They understand that a 25-year-old "basketball player" with weight "85 kg" likely has height around 190cm, while an "elderly patient" with the same weight suggests 170cm. This contextual reasoning transforms data preprocessing from rigid rules to intelligent interpretation.

**Learning objective**: Master the engineering skills to build production-ready medical data pipelines that leverage LLM intelligence while maintaining reliability, security, and cost efficiency.

## Conceptual Foundation

**The preprocessing challenge**:

Traditional approach: $\text{Raw Data} \xrightarrow{\text{rules}} \text{Clean Data}$

LLM-enhanced approach: $\text{Raw Data} \xrightarrow{\text{context + domain knowledge}} \text{Intelligent Standardization}$

---

**Medical data standardization complexity**:

- **Unit conversions**: Imperial ↔ metric with context awareness (6'2" vs 6.2 lbs)
- **Date parsing**: Cultural formats with ambiguity resolution (is 01/02/2024 Jan 2nd or Feb 1st?)
- **Language normalization**: Multilingual medical terms ("Größe"→height, "peso"→weight)
- **Value validation**: Physiological constraints (height 50–250 cm, weight 20–300 kg, the bounds enforced in Section 6)
- **Missing data recovery**: Context-aware imputation using patient demographics and medical history

**Why LLMs excel here**:
- **Semantic understanding**: Recognize medical concepts across languages and formats
- **Context integration**: Use patient demographics to validate and improve standardization
- **Uncertainty handling**: Flag ambiguous cases rather than making silent errors
- **Domain knowledge**: Apply medical training data to make clinically-informed decisions

---

## Our Learning Strategy

**Progressive complexity**:
1. **Security foundations**: API key management and data protection
2. **Structured output**: JSON schemas for reliable parsing
3. **Batch processing**: Efficient, robust API interactions with error handling
4. **Classical baseline**: Statistical methods for comparison and fallback
5. **LLM enhancement**: Context-aware imputation and standardization
6. **Production deployment**: Complete pipeline with monitoring and validation

**Engineering principles**:
- **Reliability first**: Classical methods as fallback when LLMs fail
- **Cost optimization**: Use expensive LLM processing strategically
- **Validation driven**: Every transformation validated against medical constraints
- **Audit ready**: Complete logging for regulatory compliance

---

## Learning Objectives

By completing this notebook, you will be able to:

- **Deploy secure LLM pipelines** with proper API key management and data protection practices
- **Engineer robust batch processing** with exponential backoff, error handling, and monitoring
- **Implement intelligent standardization** that combines LLM context awareness with classical validation
- **Build cost-effective hybrid systems** that optimize the accuracy/cost tradeoff for different use cases
- **Create production-ready pipelines** with comprehensive logging, validation, and regulatory compliance
- **Evaluate method performance** using masking experiments and clinical validation protocols

## Before We Code: Prediction Exercise

**What should we expect to see?**
- LLMs significantly outperform rule-based systems on heterogeneous, multilingual data
- Classical methods provide faster, cheaper baseline for simple transformations
- Hybrid approaches achieve best cost/performance balance
- Context richness directly correlates with LLM imputation quality

**What challenges will we encounter?**
- **API reliability**: Network failures, rate limits, model availability
- **Cost scaling**: Processing large datasets with expensive LLM calls
- **Quality control**: Validating LLM outputs against medical constraints
- **Privacy concerns**: Sending sensitive medical data to external APIs

**How will we validate success?**
- **Masking experiments**: Hide known values, test imputation accuracy
- **Clinical review**: Medical experts evaluate plausibility of standardized data
- **Population validation**: Aggregate results match known health statistics
- **Cost-benefit analysis**: Quantify accuracy improvements vs computational costs

## Success Criteria

By the end of this notebook, you should have:

- No hard-coded API keys (secure credential management)
- A masking experiment that reports MAE and $R^2$ for every method on the same held-out values
- A per-batch audit trail (batch id, size, attempts, timestamp) for every LLM transformation
- A batched, parallel pipeline whose runtime and cost scale predictably to larger datasets
- Imputed values that stay inside the physiological bounds checked in Section 6
- A documented fallback path for every step that depends on the API

> **Scope.** This notebook runs on a small synthetic dataset (50 records in FAST mode).
> It teaches the engineering patterns. It is not by itself evidence that the pipeline is
> fit for clinical use: a real deployment additionally needs governed real data, clinical
> review, and production monitoring.


In [ ]:
#@title Install missing packages (runs on Colab and on your own computer; installs only what is missing)
import importlib.util, subprocess, sys

REQUIRED = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'sklearn': 'scikit-learn', 'ipywidgets': 'ipywidgets', 'jsonschema': 'jsonschema', 'dateutil': 'python-dateutil', 'tqdm': 'tqdm', 'google.genai': 'google-genai', 'gspread': 'gspread'}  # module to import: package to install

def _missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:
        return True

missing = [pkg for mod, pkg in REQUIRED.items() if _missing(mod)]
if missing:
    print("Installing:", " ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Done.")
else:
    print("All required packages are installed.")


In [ ]:
#@title Section 0: Core Setup and Environment Configuration

# This section sets up the environment and imports all necessary libraries

import os, sys, time, json, warnings, hashlib, random
from dataclasses import dataclass
from typing import Dict, Any, Optional, List, Tuple, Generator, TypedDict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML, clear_output

# Configure visualization settings
plt.rcParams.update({"figure.dpi": 110})
sns.set_theme(style="whitegrid", palette="muted")

from tqdm.auto import tqdm

# Import sklearn components for classical imputation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # Required for IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_absolute_error, r2_score

# JSON validation
from jsonschema import validate, Draft202012Validator

# Detect execution environment (Colab, Jupyter, or script)
def detect_environment():
    """Detect if running in Colab, Jupyter, or as a script."""
    try:
        import google.colab
        return "colab"
    except Exception:
        return "jupyter" if "ipykernel" in sys.modules else "script"

ENVIRONMENT = detect_environment()

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
warnings.filterwarnings("ignore")

print(f"🔍 Environment: {ENVIRONMENT}")

# Configuration flags
FAST = bool(int(os.environ.get("FAST", "1")))       # Use smaller datasets for quick testing
DRY_RUN = bool(int(os.environ.get("DRY_RUN", "0"))) # Skip API calls for testing

print("⚡ FAST mode" if FAST else "🐢 FULL mode", "| 🎭 DRY_RUN:", DRY_RUN)

# Model configuration constants
CTX_WINDOW_TOKENS = 1_000_000  # Context window size for batch processing
MAX_OUTPUT_TOKENS = 2048 if FAST else 8192
SAMPLE_SIZE = 12 if FAST else None  # Limit samples in FAST mode

# APIs and Key Management

**What is an API?**  
An **API (Application Programming Interface)** is a standardized way for programs to talk to each other.
  
In this notebook, it lets us send data to a **Large Language Model (LLM)** service (Gemini) and receive results. The same code can be linked to different LLMs simply by switching the API endpoint and credentials.

**Real-world problem**: Access requires a secret **API key** (like a password). If the key is hard-coded in a notebook, it can be accidentally shared in version control or screenshots, creating a serious security risk.

**Traditional approach**: Store keys in environment variables or config files instead of code.

**Key insight**: Modern platforms provide **secure secret management**:
- Colab → `userdata` store  
- Local Jupyter → `.env` files or OS environment variables  
- Cloud platforms → built-in secret vaults  

## Our learning strategy

**Progressive approach**:  
1. Detect execution environment automatically  
2. Use platform-specific secure storage  
3. Provide clear fallback instructions  
4. Validate API connectivity before use  

## Before we code: prediction exercise
- What do we expect to happen when connecting through the API?  
- What challenges could arise when handling sensitive medical data securely?


In [ ]:
#@title # Section 1: Secure API Key Management and Model Initialization

#@markdown **Which models to offer.** "Automatic" puts every model that can be reached into the model list below,
#@markdown so you pick the model interactively. Three sources: Gemini (secret `GOOGLE_API_KEY`); open models on
#@markdown Ollama Cloud (secret `OLLAMA_API_KEY`, created at ollama.com/settings/keys; the Free plan includes a
#@markdown starter amount of usage); and the local model on the KTH GPU server, which answers only on the KTH
#@markdown network (a laptop on campus or on the KTH VPN), so it does not appear on Colab.
LLM_PROVIDER = "Automatic"  #@param ["Automatic", "Gemini API (Google)", "Ollama Cloud API", "KTH GPU server (local model)"]
FRED_URL = "http://fred.hallf.kth.se:11435"  #@param {type:"string"}
FRED_MODEL = "qwen3.8:27b"  #@param {type:"string"}
#@markdown Models to offer from Ollama Cloud, separated by commas (the names listed at ollama.com/api/tags):
OLLAMA_CLOUD_MODELS = "gemma4:31b, gpt-oss:120b"  #@param {type:"string"}
import os as _os
LLM_PROVIDER = _os.getenv("LLM_PROVIDER", LLM_PROVIDER)
USE_GEMINI = LLM_PROVIDER not in ("KTH GPU server (local model)", "Ollama Cloud API")
USE_FRED = LLM_PROVIDER not in ("Gemini API (Google)", "Ollama Cloud API")
USE_OLLAMA = LLM_PROVIDER not in ("Gemini API (Google)", "KTH GPU server (local model)")

import os, sys, json, warnings
from typing import Dict, Any, Optional, List

try:
    ENVIRONMENT
except NameError:
    ENVIRONMENT = "jupyter"
try:
    DRY_RUN
except NameError:
    DRY_RUN = False

warnings.filterwarnings("ignore")

def get_api_key(name="GEMINI_API_KEY") -> Optional[str]:
    key = None
    if ENVIRONMENT == "colab":
        try:
            from google.colab import userdata  # type: ignore
            for key_name in [name, "GOOGLE_API_KEY", "GEMINI_API_KEY"]:
                try:
                    key = userdata.get(key_name)
                    if key:
                        print(f"✅ Found API key: {key_name}")
                        break
                except Exception:
                    continue
        except Exception:
            pass
    if not key:
        for key_name in [name, "GOOGLE_API_KEY", "GEMINI_API_KEY"]:
            key = os.getenv(key_name)
            if key:
                print(f"✅ Found API key in environment: {key_name}")
                break
    return key

API_KEY = get_api_key() if USE_GEMINI else None
if USE_GEMINI and not API_KEY:
    if USE_FRED or USE_OLLAMA:
        print("ℹ️ No Gemini key found: Gemini is not offered.")
        USE_GEMINI = False
    else:
        raise AssertionError("No API key found. In Colab → 🔑 Secrets → add GOOGLE_API_KEY or GEMINI_API_KEY.")

# Try v2, then v1
MODE = None
client_v2 = None
genai_v1 = None

if USE_GEMINI:
    try:
        from google import genai as genai_v2  # new SDK
        client_v2 = genai_v2.Client(api_key=API_KEY)
        MODE = "v2"
        print("✅ Using google.genai (v2)")
    except Exception as e:
        print(f"⚠️ V2 SDK failed: {e}")
        try:
            import google.generativeai as _genai_v1  # older SDK
            _genai_v1.configure(api_key=API_KEY)
            genai_v1 = _genai_v1
            MODE = "v1"
            print("✅ Falling back to google.generativeai (v1)")
        except Exception as e2:
            MODE = None
            print(f"❌ Both SDKs failed: {e2}")

def list_models_safe() -> List[Dict[str, Any]]:
    """
    Returns [{'name','display_name','supported_generation_methods'}, ...]
    """
    try:
        if MODE == "v2" and client_v2:
            out = []
            for m in client_v2.models.list():
                out.append({
                    "name": m.name,
                    "display_name": getattr(m, "display_name", m.name),
                    "supported_generation_methods": getattr(m, "supported_generation_methods", []),
                })
            return out
        elif MODE == "v1" and genai_v1:
            out = []
            for m in genai_v1.list_models():
                out.append({
                    "name": m.name,
                    "display_name": getattr(m, "display_name", m.name),
                    "supported_generation_methods": getattr(m, "supported_generation_methods", []),
                })
            return out
        else:
            return []
    except Exception as e:
        print(f"⚠️ list_models failed: {e}")
        return []

AVAILABLE_MODELS = list_models_safe()

def filter_generative_models(models_list: List[Dict[str, Any]]) -> List[str]:
    """
    Keep likely text-generation models. Drop image/audio/speech/vision/thinking/live/dialog/embedding.
    """
    keep = []
    for m in models_list:
        name = m["name"]
        lname = name.lower()
        if any(x in lname for x in ["embedding", "image", "imagen", "audio", "speech", "tts", "veo", "vision", "thinking", "live", "dialog"]):
            continue
        methods = m.get("supported_generation_methods", []) or []
        if not methods or ("generateContent" in methods):
            keep.append(name)
    return keep

AVAILABLE_GENERATIVE = filter_generative_models(AVAILABLE_MODELS)

print(f"📋 Found {len(AVAILABLE_MODELS)} total models")
print(f"✅ {len(AVAILABLE_GENERATIVE)} text models available:")
for model_name in AVAILABLE_GENERATIVE[:6]:
    print(f"   • {model_name}")
if len(AVAILABLE_GENERATIVE) > 6:
    print(f"   ... and {len(AVAILABLE_GENERATIVE) - 6} more")

# ---------------- Ollama servers: the KTH GPU server (fred) and Ollama Cloud, the same interface as Gemini ----------------
import urllib.request, urllib.error

def _ollama_post(base_url, path, body, api_key=None, timeout=600):
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = "Bearer " + api_key
    req = urllib.request.Request(base_url.rstrip("/") + path, data=json.dumps(body).encode("utf-8"), headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))

def _fred_reachable():
    try:
        with urllib.request.urlopen(FRED_URL.rstrip("/") + "/api/version", timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

class _FredText:
    def __init__(self, text):
        self.text = text

def ollama_chat(base_url, api_key, model_name, prompt, temperature=0.2, max_output_tokens=2048, mime=None, schema=None,
                who="The model server"):
    """One chat request to an Ollama server. JSON mode uses the same schema as Gemini (Ollama 'format')."""
    body = {"model": model_name, "stream": False, "think": False,
            "messages": [{"role": "user", "content": prompt}],
            "options": {"temperature": temperature, "num_predict": max_output_tokens}}
    if mime == "application/json":
        # plain "json" mode lets the model return only one object; allow a list as Gemini does
        if schema is not None:
            body["format"] = schema
        elif "json array" in prompt.lower():
            body["format"] = {"type": "array"}                 # the prompt asks for a list: require one
        else:
            body["format"] = {"type": ["array", "object"]}
    out = None
    for _attempt in range(3):
        try:
            out = _ollama_post(base_url, "/api/chat", body, api_key)
            break
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", "replace").strip()[:300]
            if e.code in (401, 403, 429):                     # key not accepted, or the usage limit of the plan is reached
                raise RuntimeError(f"{who}: HTTP {e.code}. {detail}") from None
            if "think" in body and "think" in detail.lower():  # a model without a thinking switch
                del body["think"]
            elif isinstance(body.get("format"), dict):        # schema not accepted: fall back to plain JSON mode
                body["format"] = "json"
            else:
                raise RuntimeError(f"{who}: HTTP {e.code}. {detail}") from None
    if out is None:
        raise RuntimeError(f"{who}: the request was not accepted.")
    text = out.get("message", {}).get("content", "")
    if os.getenv("FRED_DEBUG"):                              # optional: keep the raw replies for checking
        with open("fred_replies.log", "a", encoding="utf-8") as f:
            f.write(json.dumps({"model": model_name, "format": body.get("format"), "done_reason": out.get("done_reason"),
                                "eval_count": out.get("eval_count"), "text": text}, ensure_ascii=False) + "\n")
    if mime == "application/json":
        # In JSON mode an open model tends to wrap a requested list as {"records": [...]}.
        # Gemini returns the bare list, which is what the rest of the notebook expects.
        try:
            obj = json.loads(text)
            if isinstance(obj, dict) and len(obj) == 1 and isinstance(next(iter(obj.values())), list):
                text = json.dumps(next(iter(obj.values())), ensure_ascii=False)
        except Exception:
            pass
    return _FredText(text)

def fred_generate(model_name, prompt, temperature=0.2, max_output_tokens=2048, mime=None, schema=None):
    """The local model on the KTH GPU server."""
    return ollama_chat(FRED_URL, None, model_name, prompt, temperature, max_output_tokens, mime, schema, who="KTH GPU server")

FRED_OK = USE_FRED and _fred_reachable()
if USE_FRED:
    if FRED_OK:
        print(f"✅ KTH GPU server reachable: {FRED_URL}, model {FRED_MODEL} (one request at a time)")
    else:
        print(f"⚠️ KTH GPU server not reachable at {FRED_URL}. It answers only on the KTH network or VPN; on Colab use Gemini or Ollama Cloud.")

# ---------------- Ollama Cloud: open models through an API, with a key ----------------
OLLAMA_CLOUD_URL = os.getenv("OLLAMA_CLOUD_URL", "https://ollama.com")

def get_ollama_key() -> Optional[str]:
    key = None
    if ENVIRONMENT == "colab":
        try:
            from google.colab import userdata  # type: ignore
            key = userdata.get("OLLAMA_API_KEY")
        except Exception:
            key = None                                        # no such secret, or access not granted
    return key or os.getenv("OLLAMA_API_KEY")

OLLAMA_KEY = get_ollama_key() if USE_OLLAMA else None
OLLAMA_MODELS = [m.strip() for m in OLLAMA_CLOUD_MODELS.split(",") if m.strip()]
OLLAMA_OK = bool(USE_OLLAMA and OLLAMA_KEY and OLLAMA_MODELS)
if USE_OLLAMA:
    if OLLAMA_OK:
        print(f"✅ Ollama Cloud key found: {', '.join(OLLAMA_MODELS)} (one request at a time)")
    elif LLM_PROVIDER == "Ollama Cloud API":
        raise AssertionError("No Ollama key found. In Colab → 🔑 Secrets → add OLLAMA_API_KEY (from ollama.com/settings/keys).")
    else:
        print("ℹ️ No OLLAMA_API_KEY found: Ollama Cloud is not offered.")

def ollama_cloud_generate(model_name, prompt, temperature=0.2, max_output_tokens=2048, mime=None, schema=None):
    """An open model on Ollama Cloud. The data leave the computer: synthetic or public data only."""
    return ollama_chat(OLLAMA_CLOUD_URL, OLLAMA_KEY, model_name, prompt, temperature, max_output_tokens, mime, schema, who="Ollama Cloud")

class UnifiedModelAdapter:
    """
    Normalizes generate_content across SDKs.
    Supports JSON mode if response_mime_type='application/json' (v2).
    """

    def __init__(self, model_id: str):
        self.model_id = model_id

    def generate_content(self, prompt: str, generation_config: Dict[str, Any]):
        temperature = float(generation_config.get("temperature", 0.2))
        max_output_tokens = int(generation_config.get("max_output_tokens", 2048))
        mime = generation_config.get("response_mime_type")
        schema = generation_config.get("response_schema")

        if self.model_id.startswith("fred/"):
            return fred_generate(self.model_id[len("fred/"):], prompt, temperature, max_output_tokens, mime, schema)
        if self.model_id.startswith("ollama/"):
            return ollama_cloud_generate(self.model_id[len("ollama/"):], prompt, temperature, max_output_tokens, mime, schema)

        if MODE == "v2" and client_v2:
            cfg = {"temperature": temperature, "max_output_tokens": max_output_tokens}
            if mime == "application/json":
                cfg["response_mime_type"] = "application/json"
                if schema is not None:
                    cfg["response_schema"] = schema
            resp = client_v2.models.generate_content(
                model=self.model_id,
                contents=prompt,
                config=cfg
            )
            if hasattr(resp, "text"):
                return resp
            class _Wrap:
                def __init__(self, r):
                    txt = ""
                    try:
                        c0 = getattr(r, "candidates", [])[0]
                        parts = getattr(getattr(c0, "content", None), "parts", None)
                        if parts:
                            txt = "".join(getattr(p, "text", "") or "" for p in parts)
                    except Exception:
                        txt = str(r)
                    self.text = txt
            return _Wrap(resp)

        elif MODE == "v1" and genai_v1:
            gm = genai_v1.GenerativeModel(self.model_id)
            gen_cfg = {"temperature": temperature, "max_output_tokens": max_output_tokens}
            resp = gm.generate_content(prompt, generation_config=gen_cfg)
            return resp

        else:
            raise RuntimeError("No active SDK client. Cannot generate content.")



# Model Selection and Configuration

## Why this matters

**Real-world problem**: Different models have different capabilities, costs, and speed. Choosing the right one is crucial for efficiency and reproducibility.

Sometimes you may want a **hosted API model** (Gemini, GPT, Claude), and other times you may prefer to run a **local server** with an open-source model (e.g., LLaMA, Mistral, Falcon) for privacy, cost control, or offline use.

**Trade-offs (hosted Gemini families, as offered to this notebook)**:
- **Gemini Flash Lite** (`gemini-flash-lite-latest`) → Cheapest and fastest, well suited to bulk standardization
- **Gemini Flash** (`gemini-flash-latest`) → Balanced speed and reasoning for everyday work
- **Gemini Pro** (`gemini-pro-latest`) → Strongest reasoning, highest cost, best reserved for difficult cases

Prefer the rolling `-latest` aliases over pinned version numbers. Google retires numbered
releases, and a hard-coded id such as `gemini-2.5-flash-lite` eventually returns a 404 on use
even while the models endpoint still lists it.

**Open-source options** (run on your own hardware or server):  
- **Small models (1-7B parameters)** → Lightweight, fast, can run on a laptop GPU/CPU, but weaker reasoning  
- **Medium models (13-30B parameters)** → Better reasoning, higher memory needs, suitable for servers  
- **Large models (65B+ parameters)** → State-of-the-art reasoning, but require high-end GPUs/TPUs and more engineering effort  

**Open models through an API (Ollama Cloud)**: Ollama also hosts open models (Gemma, gpt-oss, Mistral, DeepSeek and others) behind the same interface as a local Ollama server. The Free plan includes a starter amount of usage, one request at a time. In this notebook: create a key at ollama.com/settings/keys, store it as the secret `OLLAMA_API_KEY`, and the models appear in the list below. As with any cloud API the data leave your computer: fine for the synthetic records used here, not for data that must stay in house.  

---

### Have you heard about tokens?

- **1 token ≈ 4 characters of English text** (on average)  
- This is roughly **¾ of a word**  
- **1,000 tokens ≈ 750 words ≈ 1.5-2 book pages** (depending on formatting)  
- A **1M token context window** therefore holds roughly **750,000 words**, or about **1,500–2,000 printed pages**  

👉 So when a model advertises a 1–2M token context window, it can take in the equivalent of several full-length books, or a few hundred research papers, in a single request.

---

### Why input and output tokens cost differently

- **Input tokens** → The model *reads* them in one pass (the *prefill* stage). All input tokens are processed together, so the hardware stays busy and the cost per token is low.  
- **Output tokens** → The model *generates* them one at a time (the *decode* stage). This is more expensive because:  
  - Each new token needs its own forward pass through the whole network.  
  - The steps are sequential, so they cannot be batched the way input tokens are.  
  - Decoding is limited mostly by memory bandwidth rather than raw arithmetic, which leaves hardware under-used.  

**Rule of thumb**:  
- Input tokens are **cheaper** (feeding long context is relatively efficient).  
- Output tokens are **more expensive** because generation is computationally heavier and slower.  

---


### Running models locally vs. via API

While APIs give you easy access to powerful hosted models, **running models locally** provides a different set of advantages:  
- **Full control of the pipeline** → No dependency on an external provider  
- **Better data privacy** → Sensitive data never leaves your system  
- **Custom fine-tuning** → You can adapt the model to your domain or task  
- **Specialized deployment** → Optimized inference on GPUs, CPUs, or edge devices  

👉 In short: APIs offer **convenience and scale**, while local models offer **control and privacy**. The best choice depends on your constraints and objectives.  

---

## Conceptual foundation

**Token economics** (Gemini API paid tier, August 2026, price per 1M tokens):

| Model | Input | Output |
|---|---|---|
| Gemini 2.5 Flash Lite | \$0.10–\$0.30 | \$0.40 |
| Gemini 3.5 Flash Lite | \$0.30 | \$2.50 |
| Gemini 3.5 Flash | \$1.50 | \$9.00 |
| Gemini 3.7 Flash | \$0.75–\$1.50 | \$3.75–\$7.50 |
| Gemini 2.5 Pro | \$1.25–\$2.50 | \$10.00–\$15.00 |
| Gemini 3.1 Pro (preview) | \$2.00–\$4.00 | \$12.00–\$18.00 |

Ranges depend on the input type (text and image against audio) or on prompt length
(up to 200k tokens against longer prompts). Prices change, so treat the table as an
order-of-magnitude guide and check the pricing page before budgeting a large run.

- Context window = maximum tokens a model can process at once  
- **Batching strategy**: N records × M tokens < context_window  
- Use ~80% of the window as a safety margin to avoid truncation  

**Thinking and parameters**:  
- More parameters → deeper reasoning and context handling, but slower and more expensive  
- Fewer parameters → faster response, lower memory use, but limited reasoning depth  
- Some models are optimized for **speed** (Flash), others for **reasoning** (Pro), and some open-source models try to balance both (for example a 7B model against a 70B model of the same family).

**Model settings (important knobs)**:  
- **Temperature** → Controls randomness in generation.  
  - *Low (0.1–0.3)*: More deterministic, consistent outputs (good for data cleaning, standardization).  
  - *Medium (0.5–0.7)*: Balanced creativity and reliability.  
  - *High (0.8–1.0+)*: More diverse, creative outputs, but less predictable.  
- **Top-k / Top-p (nucleus sampling)** → Further control over how many candidate tokens are considered.  
- **Max output tokens** → Limits length of the generated response.  
- **Stop sequences** → Ensure the model does not “run on” past the desired output.

👉 In practice:  
- For **data preprocessing and medical standardization**, we want **low temperature** (near-deterministic, stable results).  
- For **exploration or ideation**, a **higher temperature** can be useful.  

---

## Strategy

**Selection criteria**:  
1. Data standardization → Use fast models (Flash or small open-source) with low temperature  
2. Complex reasoning → Use Pro or larger parameter models with moderate temperature  
3. Large batches → Choose models with larger context windows  
4. Offline / secure deployment → Run an open-source model locally  

## API vs. Local Models — Comparison

| Aspect              | Hosted API Models (Gemini, GPT, Claude) | Local / Open-Source Models (LLaMA, Mistral, Falcon) |
|---------------------|------------------------------------------|-----------------------------------------------------|
| **Setup**           | No installation, ready to use via API   | Requires local server, GPU/CPU setup, dependencies  |
| **Cost**            | Pay per token (subscription or usage)   | Free to run, but hardware and energy cost matter    |
| **Performance**     | Optimized, scalable infrastructure      | Depends on your hardware (GPU VRAM, RAM, CPUs)      |
| **Capabilities**    | State-of-the-art reasoning, large context windows | Varies by model size (1B–70B+ parameters) |
| **Privacy**         | Data leaves your environment (needs trust in provider) | Full data control, no external sharing              |
| **Flexibility**     | Limited to vendor’s models/configuration | Can fine-tune, customize prompts, extend pipelines  |
| **Speed**           | Low latency, high throughput in cloud   | Smaller models fast, larger models slower locally   |
| **Use cases**       | Quick experiments, large-scale processing | Sensitive data, offline work, domain-specific fine-tuning |

👉 **Rule of thumb**:  
- Use **API models** for scale, ease, and cutting-edge performance.  
- Use **local open-source models** for privacy, offline use, or when you want to fine-tune the system.  

---

## Before we code: prediction exercise
- What do we expect to see when comparing small vs. large models?  
- How will temperature affect reproducibility vs. diversity of outputs?  
- What challenges might arise with real medical data (e.g., privacy, context length)?  
- How would you validate results in practice across different model families?

In [ ]:
#@title Section 2: Model Configuration and Selection

import re
import ipywidgets as widgets
from IPython.display import display, clear_output
from typing import Dict, Any, List, Optional

# ---------------- Shortlisting logic ----------------

# Families we actually want to show, in priority order.
# Rolling aliases come first so the shortlist does not go stale when Google
# retires a numbered release. Numbered families follow as explicit fallbacks.
BASES = [
    "gemini-flash-lite-latest",
    "gemini-flash-latest",
    "gemini-pro-latest",
    "gemini-3.5-flash-lite",
    "gemini-3.5-flash",
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "gemini-2.5-flash-lite",
]

def shortlist_models(model_ids: List[str], bases=BASES, max_options: int = 8) -> List[str]:
    """
    From the messy catalog, keep at most one 'latest-looking' entry per base family.
    Heuristic: pick lexicographically last match for each base (usually newer suffix).
    """
    picked = []
    for base in bases:
        matches = [m for m in model_ids if m.split("/")[-1].startswith(base)]
        if matches:
            exact = [m for m in matches if m.split("/")[-1] == base]
            pick = exact[0] if exact else sorted(matches)[-1]
            if pick not in picked:
                picked.append(pick)  # prefer exact base id, else 'latest'
    # Fallback if nothing matched (unlikely): keep first few generative IDs
    if not picked:
        picked = sorted(model_ids)[:max_options]
    # Enforce cap
    return picked[:max_options]

# Build a tidy list from AVAILABLE_GENERATIVE
LIMITED_IDS = shortlist_models(AVAILABLE_GENERATIVE) if USE_GEMINI else []
if OLLAMA_OK:
    LIMITED_IDS = [f"ollama/{m}" for m in OLLAMA_MODELS] + LIMITED_IDS   # open models through the Ollama Cloud API
if USE_FRED and FRED_OK:
    LIMITED_IDS = [f"fred/{FRED_MODEL}"] + LIMITED_IDS   # the local model is one more option in the list

# --------------- Catalog with friendly names ---------------

def _display_name_from_id(mid: str) -> str:
    # "models/gemini-2.5-flash-lite-preview-06-17" → "Gemini 2.5 Flash Lite (Preview 06-17)"
    if mid.startswith("fred/"):
        return f"KTH GPU server (local): {mid[len('fred/'):]}"
    if mid.startswith("ollama/"):
        return f"Ollama Cloud (API): {mid[len('ollama/'):]}"
    tail = mid.split("/")[-1]
    name = tail.replace("gemini-", "Gemini ").replace("-", " ")
    name = re.sub(r"\bpro\b", "Pro", name, flags=re.I)
    name = re.sub(r"\bflash\b", "Flash", name, flags=re.I)
    name = name.replace("Lite", "Lite")
    name = re.sub(r"\( ?preview ?\)", "(Preview)", name, flags=re.I)
    return name.title().replace("Preview", "(Preview)")

def build_model_catalog(model_names: List[str]) -> Dict[str, Dict[str, Any]]:
    catalog = {}
    for mid in model_names:
        pretty = _display_name_from_id(mid)
        desc = "Text generation model discovered at runtime."
        if "(Preview)" in pretty:
            desc = "Preview model; capabilities may change."
        catalog[mid] = {
            "id": mid,
            "name": pretty,
            "description": desc,
            # cost hints are optional; not blocking
            "cost_per_1m_input": None,
            "cost_per_1m_output": None,
        }
    return catalog

MODEL_CATALOG = build_model_catalog(LIMITED_IDS)

# ---------------- Default selection: preselect Lite ----------------

def pick_default_lite(catalog: Dict[str, Dict[str, Any]]) -> Optional[str]:
    # Prefer 2.5 Lite, then 2.0 Lite, then Flash, then Pro.
    prefs = [
        "models/gemini-flash-lite-latest",
        "models/gemini-flash-latest",
        "models/gemini-3.5-flash-lite",
        "models/gemini-3.5-flash",
        "models/gemini-2.5-flash",
        "models/gemini-pro-latest",
        "models/gemini-2.5-pro",
    ]
    keys = list(catalog.keys())
    for pref in prefs:
        candidates = [k for k in keys if k.startswith(pref)]
        if candidates:
            return sorted(candidates)[-1]
    return keys[0] if keys else None

SELECTED_MODEL = pick_default_lite(MODEL_CATALOG)

# ---------------- Widgets ----------------

model = None
model_dropdown = widgets.Dropdown(
    options=[(cfg["name"], mid) for mid, cfg in MODEL_CATALOG.items()],
    value=SELECTED_MODEL,
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='520px')
)
init_button = widgets.Button(description="Initialize Model", button_style='success')
info_output = widgets.Output()
status_output = widgets.Output()

def _render_info(mid: str):
    cfg = MODEL_CATALOG.get(mid, {"name": mid, "description": "n/a"})
    with info_output:
        clear_output(wait=True)
        print(f"📊 {cfg['name']}")
        print(f"   {cfg.get('description','')}")

def on_model_change(change):
    global SELECTED_MODEL
    SELECTED_MODEL = change["new"]
    _render_info(SELECTED_MODEL)

def _health_check(adapter: "UnifiedModelAdapter") -> str:
    # The classic sanity ping you wanted:
    prompt = "Reply with exactly: Hello, how can I help you?"
    try:
        resp = adapter.generate_content(prompt, {"temperature": 0.1, "max_output_tokens": 256})
        txt = (getattr(resp, "text", "") or "").strip()
        return txt[:80]
    except Exception as e:
        return f"[health-check failed: {e}]"

def initialize_model_handler(_):
    global model, SELECTED_MODEL
    with status_output:
        clear_output(wait=True)
        print(f"⚙️ Initializing with selected model: {SELECTED_MODEL}...")
        try:
            if DRY_RUN:
                print("🎭 DRY_RUN=True — skipping SDK initialization.")
                model = None
            else:
                # Some model ids are listed by the API but rejected on use
                # (for example "no longer available to new users"), so every
                # candidate is health-checked and the first working one is kept.
                model = None
                ok = False
                candidates = [SELECTED_MODEL] + [m for m in MODEL_CATALOG if m != SELECTED_MODEL]
                for candidate in candidates:
                    trial = UnifiedModelAdapter(candidate)
                    check = _health_check(trial)
                    if check.startswith("[health-check failed"):
                        print(f"⚠️ {candidate} is not usable: {check}")
                        continue
                    model = trial
                    SELECTED_MODEL = candidate
                    ok = check.startswith("Hello, how can I help you?")
                    if ok:
                        print(f"✅ Adapter ready: {candidate}. Health check: {check}")
                    else:
                        print(f"⚠️ {candidate} responded, but not with the expected phrase:\n   {check}")
                    break
                if model is None:
                    print("❌ None of the shortlisted models accepted a request.")
                else:
                    if model_dropdown.value != SELECTED_MODEL:
                        model_dropdown.value = SELECTED_MODEL
                    print(f"\nModel status: {'✅ Ready' if ok else '⚠️ Degraded (permissions/quota or preview quirks)'}")
        except Exception as e:
            model = None
            print(f"❌ Failed to initialize model {SELECTED_MODEL}: {e}")

model_dropdown.observe(on_model_change, names='value')
init_button.on_click(initialize_model_handler)

print("📦 Select a model and click Initialize:")
display(widgets.VBox([model_dropdown, info_output, init_button, status_output]))

_render_info(SELECTED_MODEL)

# Auto-initialize so that "Run all" yields a working model without a manual click.
# The button above stays available for switching models interactively.
if (MODE or FRED_OK or OLLAMA_OK) and not DRY_RUN:
    initialize_model_handler(None)

# Data Loading with Google Sheets Integration

## Why this matters

**Real-world problem**: Medical data often lives in spreadsheets.  
Manual export/import to CSV or Excel is error-prone — different software may use **different delimiters, encodings, or date formats**, leading to corrupted or misaligned data.

**Key insight**: Direct Sheets integration enables collaborative data preparation while maintaining version control and avoids many manual mismatches.  
But when falling back to CSV/Excel, we must be ready to handle format differences robustly.

## Conceptual foundation

**Data formats**:  
- **Google Sheets** → Cloud-native, collaborative, version-controlled  
- **CSV** → Universal, but delimiter varies (`","`, `";"`, `"\t"`), and encodings differ (`UTF-8`, `Latin-1`, etc.)  
- **Excel** → Rich formatting, multiple sheets, proprietary  

**Typical mismatches**:  
- **Delimiters** → Some regions use commas (`,`), others semicolons (`;`) or tabs (`\t`)  
- **Decimal separators** → `1.75` vs `1,75`  
- **Encodings** → `UTF-8` vs legacy encodings (`cp1252`, `iso-8859-1`)  
- **Date formats** → `DD/MM/YYYY` vs `MM/DD/YYYY` vs `YYYY-MM-DD`  

**Handling strategy**:  
- Auto-detect delimiter (`pandas.read_csv` can infer)  
- Try multiple encodings until one succeeds  
- Normalize decimal separators and date formats  
- Keep a fallback path (Sheets → Excel → CSV)  

## Strategy

1. Authenticate with Google if available  
2. Load directly from Sheets URL or ID  
3. Fall back to Excel or CSV with **robust delimiter/encoding handling**  
4. Display a preview with missing values highlighted so mismatches are obvious  

## Before we code: prediction exercise
- What can go wrong when loading data with unknown delimiters or encodings?  
- How would you detect if the dataset silently misaligned (e.g., shifted columns)?  
- What strategies could you use to validate correctness before analysis?  


In [ ]:
#@title Section 3: Enhanced Data Constructor / Loader with Google Sheets Integration and Realistic Data Generation

# Google Sheets is optional. Set USE_GOOGLE_SHEETS = True (or the environment
# variable USE_GOOGLE_SHEETS=1) to store the dataset in Google Sheets.
# Colab's credential prompt opens a separate browser window, and if that window
# is never answered, auth.authenticate_user() blocks this cell indefinitely.
#USE_GOOGLE_SHEETS = bool(int(os.environ.get("USE_GOOGLE_SHEETS", "0")))

USE_GOOGLE_SHEETS = 1

class EnhancedDataLoader:
    """Enhanced data loader with Google Sheets integration and highly realistic medical data generation."""

    def __init__(self):
        self.sheet_id = None
        self.sheet_url = None
        self.edit_url = None
        self.authenticated = False
        self.gc = None

    # ========== GOOGLE SHEETS INTEGRATION ==========

    def authenticate_google(self):
        """Authenticate with Google using Colab's built-in auth."""
        if not USE_GOOGLE_SHEETS:
            print("\u2139\ufe0f Google Sheets disabled (USE_GOOGLE_SHEETS = False), using local data generation")
            self.authenticated = False
            return False

        try:
            from google.colab import auth
            import gspread
            from google.auth import default

            print("🔐 Authenticating with Google...")
            auth.authenticate_user()

            # Get credentials and create gspread client
            creds, _ = default()
            self.gc = gspread.authorize(creds)
            self.authenticated = True
            print("✅ Google authentication successful")
            return True

        except Exception as e:
            print(f"❌ Google authentication failed: {e}")
            print("⚠️ Continuing with local data generation...")
            self.authenticated = False
            return False

    def save_to_google_sheets(self, df, sheet_name="medical_data_KTH"):
        """Save dataframe to Google Sheets."""
        if not self.authenticated:
            print("⚠️ Not authenticated with Google, skipping save")
            return False

        try:
            print(f"💾 Saving data to Google Sheets: {sheet_name}")

            # Create a new spreadsheet
            spreadsheet = self.gc.create(sheet_name)
            self.sheet_id = spreadsheet.id

            # Keep the sheet private: a new spreadsheet created through the API is
            # already visible only to its owner, so no sharing call is made here.
            # Sharing medical data with 'anyone' would expose it to everyone holding
            # the link, which is not what this notebook intends.

            # Get the first worksheet
            worksheet = spreadsheet.sheet1

            # Prepare data for upload
            # Convert DataFrame to list of lists (Google Sheets format)
            data_to_upload = [df.columns.tolist()] + df.values.tolist()

            # Handle NaN values (convert to empty strings)
            clean_data = []
            for row in data_to_upload:
                clean_row = []
                for cell in row:
                    if pd.isna(cell):
                        clean_row.append("")
                    else:
                        clean_row.append(str(cell))
                clean_data.append(clean_row)

            # Upload data
            try:
                worksheet.update(values=clean_data, range_name='A1')  # gspread >= 6
            except TypeError:
                worksheet.update('A1', clean_data)  # gspread < 6

            # Store URLs
            self.sheet_url = spreadsheet.url
            self.edit_url = f"https://docs.google.com/spreadsheets/d/{self.sheet_id}/edit"

            print(f"✅ Data saved successfully!")
            print(f"   Sheet ID: {self.sheet_id}")

            # Store sheet ID for future runs (simple file approach)
            try:
                with open('medical_data_sheet_id.txt', 'w') as f:
                    f.write(self.sheet_id)
            except OSError:
                pass  # Skip if the file cannot be written

            return True

        except Exception as e:
            print(f"❌ Failed to save to Google Sheets: {e}")
            return False

    def erase_google_sheets_data(self):
        """Erase the current Google Sheets data and local references."""
        try:
            if self.authenticated and self.sheet_id:
                # Try to delete the spreadsheet
                try:
                    spreadsheet = self.gc.open_by_key(self.sheet_id)
                    self.gc.del_spreadsheet(self.sheet_id)
                    print("🗑️ Google Sheets data deleted successfully")
                except Exception as e:
                    print(f"⚠️ Could not delete Google Sheet: {e}")

            # Clear local references
            self.sheet_id = None
            self.sheet_url = None
            self.edit_url = None

            # Remove saved sheet ID file
            try:
                if os.path.exists('medical_data_sheet_id.txt'):
                    os.remove('medical_data_sheet_id.txt')
                    print("🗑️ Local sheet ID file removed")
            except OSError:
                pass

            print("✅ Data erasure complete")
            return True

        except Exception as e:
            print(f"❌ Error during data erasure: {e}")
            return False

    def load_from_saved_sheet(self):
        """Load data from previously saved Google Sheet."""
        if not self.authenticated:
            return None

        # Try to load saved sheet ID
        try:
            if os.path.exists('medical_data_sheet_id.txt'):
                with open('medical_data_sheet_id.txt', 'r') as f:
                    saved_sheet_id = f.read().strip()

                if saved_sheet_id:
                    print(f"📂 Found saved sheet ID: {saved_sheet_id}")
                    return self.load_by_sheet_id(saved_sheet_id)

        except Exception as e:
            print(f"⚠️ Error loading saved sheet: {e}")

        return None

    def load_by_sheet_id(self, sheet_id):
        """Load data from specific Google Sheet ID."""
        try:
            print(f"📊 Loading data from Google Sheet...")

            # Open the spreadsheet
            spreadsheet = self.gc.open_by_key(sheet_id)
            worksheet = spreadsheet.sheet1

            # Get all data
            data = worksheet.get_all_values()

            if not data:
                print("⚠️ Sheet is empty")
                return None

            # Convert to DataFrame
            df = pd.DataFrame(data[1:], columns=data[0])  # First row as headers

            # Convert empty strings back to NaN
            df = df.replace('', np.nan)

            # Robust numeric coercion, including legacy feet-inches parsing
            import re

            def _parse_height_to_cm(x):
                if pd.isna(x):
                    return np.nan
                s = str(x).strip()
                # Match 5'9" or 5' 9"
                m = re.fullmatch(r"(\d+)'\s*(\d+)\s*\"", s)
                if m:
                    feet = float(m.group(1)); inches = float(m.group(2))
                    return feet * 30.48 + inches * 2.54
                # Try plain float in cm
                try:
                    return float(s)
                except Exception:
                    return np.nan

            def _parse_weight_to_kg(x, system=None):
                if pd.isna(x):
                    return np.nan
                s = str(x).strip()
                # Detect explicit lb suffix
                m = re.fullmatch(r"([0-9]+(?:\.[0-9]+)?)\s*(lb|lbs)", s, flags=re.IGNORECASE)
                if m:
                    return float(m.group(1)) / 2.205
                # Try plain float
                try:
                    val = float(s)
                    if system == 'imperial':
                        # Assume pounds if system says imperial
                        return val / 2.205
                    return val  # already kg
                except Exception:
                    return np.nan

            # Patient id and age
            for col in ['patient_id', 'age']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            # Height and weight canonical numerics
            if 'height' in df.columns:
                if df['height'].dtype == object:
                    df['height'] = df['height'].apply(_parse_height_to_cm)
                else:
                    df['height'] = pd.to_numeric(df['height'], errors='coerce')
            elif 'height_display' in df.columns:
                df['height'] = df['height_display'].apply(_parse_height_to_cm)

            if 'weight' in df.columns:
                df['weight'] = pd.to_numeric(df['weight'], errors='coerce')
            elif 'weight_display' in df.columns:
                # Use measurement_system if available
                sys_series = df['measurement_system'] if 'measurement_system' in df.columns else pd.Series([None]*len(df))
                df['weight'] = [
                    _parse_weight_to_kg(v, system=s)
                    for v, s in zip(df['weight_display'], sys_series)
                ]

            # Store sheet info
            self.sheet_id = sheet_id
            self.sheet_url = spreadsheet.url
            self.edit_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/edit"

            print(f"✅ Loaded {len(df)} rows from Google Sheets")
            return df

        except Exception as e:
            print(f"❌ Failed to load from Google Sheet: {e}")
            return None

    # ========== ORIGINAL DATALOADER METHODS ==========

    @staticmethod
    def load_from_sheets(sheet_url_or_id):
        """Load data from Google Sheets."""
        try:
            # Extract sheet ID from URL if needed
            if 'docs.google.com' in sheet_url_or_id:
                sheet_id = sheet_url_or_id.split('/d/')[1].split('/')[0]
            else:
                sheet_id = sheet_url_or_id

            # Construct CSV export URL
            csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

            # Load the data
            df = pd.read_csv(csv_url)
            print(f"✅ Loaded {len(df)} rows from Google Sheets")
            return df

        except Exception as e:
            print(f"❌ Failed to load from Sheets: {e}")
            return None

    @staticmethod
    def load_from_csv(filepath):
        """Load data from CSV file with robust encoding handling."""
        encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']

        for encoding in encodings:
            try:
                df = pd.read_csv(filepath, encoding=encoding,
                               na_values=['NULL', 'N/A', 'Unknown', '',
                                        'Not measured', 'nicht messbar'])
                print(f"✅ Loaded {len(df)} rows from {filepath} (encoding: {encoding})")
                return df
            except UnicodeDecodeError:
                continue
            except FileNotFoundError:
                print(f"❌ File not found: {filepath}")
                return None

        print(f"❌ Could not load {filepath} with any encoding")
        return None

    @staticmethod
    def create_realistic_medical_data():
        """Create highly realistic multilingual medical data with controlled correlations and clinical scenarios."""
        np.random.seed(42)
        n_samples = 100 if not FAST else 50

        print(f"🏥 Generating realistic medical dataset with {n_samples} patients...")

        # ========== ENHANCED PHYSIOLOGICAL MODELING ==========

        # Age distribution (realistic hospital demographics) - ensure exact count
        young_count = int(n_samples * 0.4)  # 40% young adults
        middle_count = int(n_samples * 0.35)  # 35% middle-aged
        elderly_count = n_samples - young_count - middle_count  # Remaining for elderly

        ages = np.concatenate([
            np.random.normal(35, 12, young_count),
            np.random.normal(55, 15, middle_count),
            np.random.normal(75, 10, elderly_count)
        ])
        ages = np.clip(ages, 18, 95).astype(int)
        np.random.shuffle(ages)

        # Gender distribution with realistic proportions - ensure exact count
        male_count = int(n_samples * 0.48)
        female_count = n_samples - male_count
        genders_base = ['Male'] * male_count + ['Female'] * female_count
        np.random.shuffle(genders_base)

        # BMI-based height-weight modeling with age and gender effects
        heights = []
        weights = []

        for i in range(n_samples):
            age = ages[i]
            gender = genders_base[i]

            # Base height by gender with age shrinkage
            if gender == 'Male':
                base_height = np.random.normal(177, 7)  # Male average
            else:
                base_height = np.random.normal(164, 6)  # Female average

            # Age-related height loss (1 mm per year after 40)
            if age > 40:
                height_loss = (age - 40) * 0.1
                base_height = max(base_height - height_loss, base_height * 0.9)

            # Realistic weight based on BMI distribution
            # Target BMI: 70% normal (18.5-25), 20% overweight (25-30), 10% other
            bmi_category = np.random.choice(['normal', 'overweight', 'other'],
                                          p=[0.70, 0.20, 0.10])

            if bmi_category == 'normal':
                target_bmi = np.random.normal(22, 2)
            elif bmi_category == 'overweight':
                target_bmi = np.random.normal(27, 1.5)
            else:
                target_bmi = np.random.normal(32, 3)

            # Context effects. Without these, BMI is independent of every other
            # field, so gender and age carry no information about weight and a
            # context-aware method cannot possibly beat height-only regression.
            # Both effects mirror population data: men run slightly higher BMI
            # at the same height, and BMI drifts upward into middle age.
            if gender == 'Male':
                target_bmi += 0.8
            target_bmi += 0.05 * (min(age, 60) - 40)

            target_bmi = np.clip(target_bmi, 16, 45)
            weight = target_bmi * (base_height / 100) ** 2

            # Add realistic noise
            height_final = base_height + np.random.normal(0, 1)
            weight_final = weight + np.random.normal(0, 2)

            heights.append(round(height_final, 1))
            weights.append(round(weight_final, 1))

        heights = np.array(heights)
        weights = np.array(weights)

        # ========== REALISTIC MISSING DATA PATTERNS ==========

        # Different missing patterns for different reasons
        missing_patterns = {
            'equipment_failure': 0.03,    # 3% equipment issues
            'patient_mobility': 0.05,     # 5% mobility issues (elderly)
            'emergency_admission': 0.04,  # 4% emergency cases
            'data_entry_error': 0.03      # 3% administrative errors
        }

        height_missing = np.zeros(n_samples, dtype=bool)
        weight_missing = np.zeros(n_samples, dtype=bool)

        for pattern, rate in missing_patterns.items():
            if pattern == 'patient_mobility':
                # More likely in elderly patients
                elderly_mask = ages > 70
                mobility_missing = np.random.random(n_samples) < (rate * 2)
                mobility_missing = mobility_missing & elderly_mask
                height_missing |= mobility_missing
            elif pattern == 'emergency_admission':
                # Random but clustered (same day issues)
                emergency_indices = np.random.choice(n_samples, int(n_samples * rate), replace=False)
                height_missing[emergency_indices] = True
                weight_missing[emergency_indices] = True
            else:
                # Random missing
                random_missing = np.random.random(n_samples) < rate
                if np.random.random() < 0.5:
                    height_missing |= random_missing
                else:
                    weight_missing |= random_missing

        # Keep the pre-masking values: realistic clinical notes often quote a
        # previous visit, and those quotes must refer to the true measurement.
        heights_true = heights.copy()
        weights_true = weights.copy()

        heights[height_missing] = np.nan
        weights[weight_missing] = np.nan

        # ========== ENHANCED MULTILINGUAL FORMATS ==========

        # Realistic date formats with temporal clustering
        date_formats_by_system = {
            'US_system': ['01/15/2024', '1/15/2024', 'Jan 15, 2024', '01-15-2024'],
            'EU_system': ['15/01/2024', '15-01-2024', '15.01.2024', '2024-01-15'],
            'Nordic_system': ['15 jan 2024', '15 januari 2024', '2024-01-15'],
            'Asian_system': ['2024年1月15日', '2024/01/15', '令和6年1月15日']
        }

        # Assign dates based on clinic origin
        dates = []
        clinic_systems = np.random.choice(['US_system', 'EU_system', 'Nordic_system', 'Asian_system'],
                                        n_samples, p=[0.2, 0.4, 0.3, 0.1])

        for system in clinic_systems:
            format_choice = np.random.choice(date_formats_by_system[system])
            dates.append(format_choice)

        # Enhanced multilingual gender formats with regional clustering
        gender_formats_by_region = {
            'Male': {
                'English': ['Male', 'M', 'male', 'MALE'],
                'Swedish': ['Man', 'Pojke', 'män', 'Manlig'],
                'Japanese': ['男性', '男', 'おとこ'],
                'German': ['Mann', 'Männlich', 'Herr'],
                'Spanish': ['Masculino', 'Hombre', 'Varón']
            },
            'Female': {
                'English': ['Female', 'F', 'female', 'FEMALE'],
                'Swedish': ['Kvinna', 'Flicka', 'kvinnlig', 'Dam'],
                'Japanese': ['女性', '女', 'おんな'],
                'German': ['Frau', 'Weiblich', 'Dame'],
                'Spanish': ['Femenino', 'Mujer', 'Señora']
            }
        }

        # Assign languages based on clinic location
        languages = np.random.choice(['English', 'Swedish', 'Japanese', 'German', 'Spanish'],
                                   n_samples, p=[0.3, 0.3, 0.15, 0.15, 0.1])

        genders_multilingual = []
        for i, base_gender in enumerate(genders_base):
            lang = languages[i]
            gender_options = gender_formats_by_region[base_gender][lang]
            genders_multilingual.append(np.random.choice(gender_options))

        # ========== REALISTIC UNIT CONVERSION (canonical SI + display) ==========

        # Canonical numeric in SI
        height_cm = heights.copy()
        weight_kg = weights.copy()

        # Display variants
        heights_display = height_cm.astype(object)
        weights_display = weight_kg.astype(object)
        units_info = []

        # US clinics use imperial, others use metric
        for i, system in enumerate(clinic_systems):
            if system == 'US_system' and np.random.random() < 0.85:  # 85% of US clinics use imperial
                # Pretty feet-inches for display, keep SI numeric untouched
                if not pd.isna(height_cm[i]):
                    feet = int(height_cm[i] // 30.48)
                    inches = round((height_cm[i] % 30.48) / 2.54)
                    if inches == 12:
                        feet += 1
                        inches = 0
                    heights_display[i] = f"{feet}'{inches}\""
                else:
                    heights_display[i] = np.nan

                if not pd.isna(weight_kg[i]):
                    # Convert to pounds with realistic precision for display only
                    weights_display[i] = round(weight_kg[i] * 2.205, 1)
                else:
                    weights_display[i] = np.nan

                units_info.append('imperial')
            else:
                # Keep metric display but vary precision
                if not pd.isna(height_cm[i]):
                    # Some clinics measure to nearest 0.5 cm
                    if np.random.random() < 0.3:
                        heights_display[i] = round(height_cm[i] * 2) / 2
                    else:
                        heights_display[i] = height_cm[i]
                else:
                    heights_display[i] = np.nan

                if not pd.isna(weight_kg[i]):
                    weights_display[i] = weight_kg[i]
                else:
                    weights_display[i] = np.nan

                units_info.append('metric')

        # ========== REALISTIC CLINIC NAMES ==========

        clinic_names_by_system = {
            'US_system': [
                'General Hospital', 'Memorial Medical Center', 'University Hospital',
                'St. Mary\'s Hospital', 'Regional Medical Center', 'Community Health Center'
            ],
            'EU_system': [
                'Universitätsklinikum', 'Krankenhaus', 'Medizinisches Zentrum',
                'Hôpital Général', 'Centro Médico', 'Ospedale Civile'
            ],
            'Nordic_system': [
                'Universitetssjukhuset', 'Vårdcentral', 'Regionsjukhuset',
                'Karolinska Institutet', 'Sahlgrenska', 'Akademiska'
            ],
            'Asian_system': [
                '大学病院', '総合病院', '医療センター',
                'University Hospital', '国立病院', '市民病院'
            ]
        }

        clinics = []
        for system in clinic_systems:
            clinic_choice = np.random.choice(clinic_names_by_system[system])
            clinics.append(clinic_choice)

        # ========== ENHANCED MEDICAL COMMENTS ==========

        # ---- Consistent clinical comments -----------------------------------
        # Each comment is chosen to AGREE with the record it describes. If
        # the comments were drawn uniformly at random, this would have produced
        # notes such as "Unable to obtain accurate height" on complete records and
        # "Längd 175cm, vikt 70kg" on a patient who weighs 83 kg. Random context
        # is at best noise and at worst disinformation: it punishes exactly the
        # context-reading methods this notebook sets out to study.

        neutral_comments = [
            "Routine annual physical examination - vitals stable",
            "Height and weight recorded during intake assessment",
            "Pre-operative measurements - patient fasting since midnight",
            "Vital signs obtained per standard protocol",
            "Height measured without shoes per protocol",
            "Weight obtained using calibrated scale",
            "Digital scale calibrated this morning - 08:00",
            "Stadiometer height measurement - patient standing straight",
            "Measurements verified by second nurse",
            "Data entry double-checked for accuracy",
            "Patient identification confirmed before measurement",
            "Cardiac rehabilitation patient - height verfied for equipment fitting",
            "Adult height stable compared to previous visits",
        ]
        high_bmi_comments = [
            "Patient with diabetes mellitus type 2 - weight managment ongoing",
            "Hypertension diagnosed - monitoring BP and wieght",
            "Patient reports recent weight gain of 5kg over 3 months",
            "Weight gain consistent with fluid retention - monitor closely",
        ]
        low_bmi_comments = [
            "Unexpected weight loss - recommend nutritional consultation",
            "Underweight for height - dietary review scheduled",
        ]
        missing_height_comments = [
            "Unable to obtain accurate height - patient in wheelchair",
            "Height estimated based on arm span measurement",
            "Previous measurements used - patient unable to stand",
        ]
        missing_weight_comments = [
            "Weight not measured - patient refusing due to anxiety",
            "Weight not measured - scale out of service this morning",
        ]

        def value_quote(lang, h_cm, w_kg):
            """A note quoting the actual measurements, in the row's language."""
            if lang == 'Swedish':
                return f"Längd {h_cm:.0f}cm, vikt {w_kg:.0f}kg - patient från Sverige"
            if lang == 'Japanese':
                return f"身長{h_cm:.0f}センチ、体重{w_kg:.0f}キロ記録済み"
            if lang == 'German':
                return f"Größe {h_cm:.0f} cm und Gewicht {w_kg:.0f} kg dokumentiert"
            return f"Height {h_cm:.0f} cm and weight {w_kg:.0f} kg recorded at intake"

        comments = []
        for i in range(n_samples):
            # A record with a missing measurement almost always says why; only
            # complete routine records sometimes go undocumented.
            if not (height_missing[i] or weight_missing[i]) and np.random.random() < 0.15:
                comments.append(np.nan)
                continue

            if height_missing[i]:
                comment = np.random.choice(missing_height_comments)
                # Most of these notes recall the previous visit - genuinely
                # recoverable context that only a text-reading method can use.
                if np.random.random() < 0.7 and not np.isnan(heights_true[i]):
                    comment += f". Height at previous visit approximately {heights_true[i]:.0f} cm"
            elif weight_missing[i]:
                comment = np.random.choice(missing_weight_comments)
                if np.random.random() < 0.7 and not np.isnan(weights_true[i]):
                    comment += f". Last recorded weight approximately {weights_true[i]:.0f} kg"
            else:
                bmi_i = weights_true[i] / (heights_true[i] / 100) ** 2
                roll = np.random.random()
                if bmi_i >= 27 and roll < 0.6:
                    comment = np.random.choice(high_bmi_comments)
                elif bmi_i < 19.5 and roll < 0.6:
                    comment = np.random.choice(low_bmi_comments)
                elif roll < 0.15:
                    comment = value_quote(languages[i], heights_true[i], weights_true[i])
                elif roll < 0.25 and 18.5 <= bmi_i < 25:
                    comment = f"BMI calculated at {bmi_i:.1f} - within normal range"
                elif ages[i] >= 75 and roll < 0.4:
                    comment = "Elderly patient - height may have decreased due to spinal compression"
                else:
                    comment = np.random.choice(neutral_comments)

            comments.append(comment)

        # ========== CREATE ENHANCED DATAFRAME ==========

        df = pd.DataFrame({
            'patient_id': range(1, n_samples + 1),
            'age': ages,
            # Canonical numeric columns used for analysis
            'height': height_cm,                # cm, numeric
            'weight': weight_kg,                # kg, numeric
            # Display-only columns for UI and Sheets readability
            'height_display': heights_display,  # e.g. 5'9" or 175.0
            'weight_display': weights_display,  # e.g. 154.3 or 70.2
            'gender': genders_multilingual,
            'date_measured': dates,
            'clinic': clinics,
            'comments': comments,
            'measurement_system': units_info
        })

        # Round remaining float columns to realistic precision
        for col in ['height', 'weight']:
            if col in df.columns and df[col].dtype in ['float64', 'float32']:
                df[col] = df[col].round(1)

        # Statistics for verification
        imperial_count = sum(1 for u in units_info if u == 'imperial')
        missing_heights = pd.isna(heights).sum()
        missing_weights = pd.isna(weights).sum()

        print(f"✅ Created realistic medical dataset:")
        print(f"   📊 {n_samples} patients (Age: {ages.min()}-{ages.max()} years)")
        print(f"   📏 Height missing: {missing_heights}/{n_samples} ({missing_heights/n_samples*100:.1f}%)")
        print(f"   ⚖️ Weight missing: {missing_weights}/{n_samples} ({missing_weights/n_samples*100:.1f}%)")
        print(f"   🇺🇸 Imperial units: {imperial_count}/{n_samples} ({imperial_count/n_samples*100:.1f}%)")
        print(f"   🌍 Languages: English, Swedish, Japanese, German, Spanish")
        print(f"   🏥 Medical scenarios: {len([c for c in comments if pd.notna(c)])}/{n_samples} documented")
        print(f"   🔢 Age distribution: {np.mean(ages):.1f}±{np.std(ages):.1f} years")

        return df

    # Keep the original method as fallback
    @staticmethod
    def create_sample_data():
        """Fallback to original sample data creation if needed."""
        return EnhancedDataLoader.create_realistic_medical_data()

    @staticmethod
    def display_data_preview(df, title="Data Preview"):
        """Display data with missing values highlighted and proper formatting."""
        print(f"\n{title}")
        print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"Columns: {', '.join(df.columns)}")
        print(f"\nMissing values:")

        missing = df.isnull().sum()
        missing_found = False
        for col, count in missing[missing > 0].items():
            print(f"  - {col}: {count} ({count/len(df)*100:.2f}%)")
            missing_found = True

        if not missing_found:
            print("  None detected")

        # Get display sample (first 10 rows) and format properly
        display_rows = min(10, len(df))
        df_display = df.head(display_rows).copy()

        # FORCE round numeric columns to 2 decimal places for display
        for col in df_display.columns:
            if df_display[col].dtype in ['float64', 'float32', 'int64', 'int32']:
                df_display[col] = df_display[col].round(2)

        # Display styled preview with missing values highlighted
        def highlight_missing(val):
            return 'background-color: #ffcccc' if pd.isna(val) else ''

        # Apply styling
        try:
            styled = df_display.style.map(highlight_missing).format(precision=2)
            display(styled)
        except Exception as e:
            # Fallback to regular display if styling fails
            print(f"\n{df_display.to_string()}")

        return df

    # ========== ENHANCED GOOGLE SHEETS WIDGETS ==========

    def create_edit_link_widget(self):
        """Create clickable link widget to edit the Google Sheet."""
        if not self.edit_url:
            return None

        link_html = f'''
        <div style="background: #e8f4fd; padding: 15px; border-radius: 8px; margin: 10px 0;">
            <h3 style="margin: 0 0 10px 0; color: #1976d2;">📊 Edit Your Medical Data</h3>
            <p style="margin: 0 0 10px 0;">Click the link below to edit your data in Google Sheets:</p>
            <a href="{self.edit_url}" target="_blank" style="
                background: #1976d2;
                color: white;
                padding: 10px 20px;
                text-decoration: none;
                border-radius: 5px;
                font-weight: bold;
                display: inline-block;">
                🔗 Open in Google Sheets
            </a>
            <p style="margin: 10px 0 0 0; font-size: 0.9em; color: #666;">
                After making changes, click the "Reload Data" button below to update the notebook.
            </p>
        </div>
        '''
        return HTML(link_html)

    def create_data_management_widgets(self):
        """Create comprehensive data management widgets including erase functionality."""
        if not self.authenticated:
            return None

        # Reload button
        reload_button = widgets.Button(
            description="🔄 Reload Data",
            button_style='success',
            tooltip='Reload data from Google Sheets',
            layout=widgets.Layout(width='180px', height='40px')
        )

        # Erase button
        erase_button = widgets.Button(
            description="🗑️ Erase Data",
            button_style='danger',
            tooltip='Delete Google Sheets data and start fresh',
            layout=widgets.Layout(width='180px', height='40px')
        )

        # Generate new data button
        generate_button = widgets.Button(
            description="🏥 New Dataset",
            button_style='warning',
            tooltip='Generate new realistic medical data',
            layout=widgets.Layout(width='180px', height='40px')
        )

        output = widgets.Output()

        def on_reload_click(b):
            with output:
                clear_output(wait=True)
                print("🔄 Reloading data from Google Sheets...")

                if self.sheet_id:
                    new_df = self.load_by_sheet_id(self.sheet_id)

                    if new_df is not None:
                        # Update global df variable
                        global df, raw_df
                        df = new_df
                        raw_df = df.copy()

                        print("✅ Data reloaded successfully!")
                        print(f"   Updated dataset: {len(df)} rows × {len(df.columns)} columns")

                        # Show preview of reloaded data
                        self.display_data_preview(df, "Reloaded Data Preview")
                    else:
                        print("❌ Failed to reload data")
                else:
                    print("❌ No sheet ID found")

        def on_erase_click(b):
            with output:
                clear_output(wait=True)
                print("🗑️ Erasing all data...")

                # Confirm deletion
                if self.erase_google_sheets_data():
                    # Clear global variables
                    global df, raw_df
                    df = None
                    raw_df = None
                    print("✅ All data erased successfully!")
                    print("   You can now generate new data or load from a different source.")
                else:
                    print("❌ Error during data erasure")

        def on_generate_click(b):
            with output:
                clear_output(wait=True)
                print("🏥 Generating new realistic medical dataset...")

                # Generate new data
                new_df = self.create_realistic_medical_data()

                if new_df is not None:
                    # Save to Google Sheets
                    if self.save_to_google_sheets(new_df):
                        # Update global variables
                        global df, raw_df
                        df = new_df
                        raw_df = df.copy()

                        print("✅ New dataset generated and saved!")
                        self.display_data_preview(df, "New Dataset Preview")
                    else:
                        print("⚠️ Data generated but not saved to Google Sheets")

        # Connect button handlers
        reload_button.on_click(on_reload_click)
        erase_button.on_click(on_erase_click)
        generate_button.on_click(on_generate_click)

        # Create button layout
        button_box = widgets.HBox([
            reload_button,
            generate_button,
            erase_button
        ], layout=widgets.Layout(justify_content='space-around'))

        return widgets.VBox([
            widgets.HTML("<h4>🔧 Data Management Controls</h4>"),
            widgets.HTML("<p style='color: #666; font-size: 0.9em;'>Manage your medical dataset with Google Sheets integration</p>"),
            button_box,
            output
        ])

# ========== ENHANCED DATA LOADING WORKFLOW ==========

def enhanced_data_loading():
    """Enhanced data loading with Google Sheets integration and realistic data generation."""
    print("🚀 Enhanced Data Loading with Realistic Medical Data Generation")
    print("=" * 70)

    # Create enhanced loader
    loader = EnhancedDataLoader()

    # Authenticate with Google
    auth_success = loader.authenticate_google()

    df = None

    if auth_success:
        # Try to load from saved sheet first
        df = loader.load_from_saved_sheet()

    if df is None:
        print("\n📝 No existing data found, creating new realistic medical dataset...")

        # Local CSV files take priority when they are present
        csv_files = ['medical_data.csv', 'sample_data.csv']
        for csv_file in csv_files:
            if os.path.exists(csv_file):
                df = loader.load_from_csv(csv_file)
                if df is not None:
                    break

        # Otherwise generate the dataset locally.
        # EnhancedDataLoader.load_from_sheets() remains available for loading your
        # own sheet by URL or id. It is no longer called here, because the public
        # sample sheet it used contained student records that were always discarded.
        if df is None:
            print("\n📝 No data files found, creating realistic medical data...")
            df = loader.create_realistic_medical_data()

        # Save the newly created data to Google Sheets
        if auth_success and df is not None:
            save_success = loader.save_to_google_sheets(df)
            if not save_success:
                print("⚠️ Could not save to Google Sheets, but continuing with local data")

    # Display the data
    if df is not None:
        global raw_df
        raw_df = df.copy()
        print("\n📊 Realistic medical data loaded:")
        loader.display_data_preview(df, "Raw Medical Data")

        # Create enhanced widgets with data management
        if auth_success:
            print("\n" + "=" * 70)
            print("🔗 GOOGLE SHEETS INTEGRATION & DATA MANAGEMENT")
            print("=" * 70)

            # Display edit link
            edit_link_widget = loader.create_edit_link_widget()
            if edit_link_widget:
                display(edit_link_widget)

            # Display enhanced management controls
            management_widget = loader.create_data_management_widgets()
            if management_widget:
                display(management_widget)

    return df, loader

# ========== EXECUTE ENHANCED DATA LOADING ==========

print("\n📂 Starting Enhanced Data Loading with Realistic Medical Data...")
df, data_loader = enhanced_data_loading()

# Format for better display function
def format_for_display(df_to_format):
    """Format dataframe for cleaner display."""
    df_display = df_to_format.copy()
    # Round numeric columns to 2 decimal places
    for col in df_display.select_dtypes(include=[np.float64, np.float32]).columns:
        df_display[col] = df_display[col].round(2)
    return df_display

print(f"\n✅ Enhanced medical data loading complete!")
print(f"📊 Dataset ready: {len(df)} rows × {len(df.columns)} columns")
if data_loader.authenticated and data_loader.edit_url:
    print(f"🔗 Google Sheets integration: ACTIVE")
    print(f"🔧 Data management controls: AVAILABLE")
else:
    print(f"📝 Google Sheets integration: Not available (using local data)")
    print(f"⚠️ Data management controls: LIMITED")


# Structured Output with JSON Schemas

convert.svg

**What is JSON (in plain words)?**  
**JSON** (JavaScript Object Notation) is a simple way to write data so both humans and computers can read it.  
It looks like labeled boxes with values inside.

Example:
```json
{
  "patient_id": "42",
  "height_cm": 178.5,
  "weight_kg": 74.2
}
```

**Why use JSON here?**  
- LLMs love to talk in **free text**, which is great for humans but fragile for code.  
- Data pipelines need **consistent, machine-readable** output.  
- JSON is a universal, lightweight format that almost every tool can parse reliably.

**The real-world problem**  
If the model returns a messy paragraph or “almost JSON” (missing commas, extra commentary), downstream code can break. We want **guaranteed structure**.

**Key idea**  
Modern LLM APIs can return **structured JSON directly** and we can enforce the shape with a **JSON Schema** so the output is valid before we use it.

---

## Conceptual foundation

**Analogy:** Free text is like a handwritten letter; JSON is a **standardized form** with named fields. Forms are easier to file, search, and validate.

**JSON Schema** (the blueprint):  
- **Type safety** — checks that numbers are numbers, dates look like dates, etc.  
- **Validation** — rejects malformed or missing fields before they hit your pipeline.  
- **Documentation** — acts as a contract: “this is exactly what a record must contain.”

Minimal example (schema idea, not full spec):
```json
{
  "type": "object",
  "required": ["patient_id", "gender", "date_measured", "clinic"],
  "properties": {
    "patient_id": { "type": "string" },
    "height_cm": { "type": ["number", "null"] },
    "weight_kg": { "type": ["number", "null"] },
    "gender": { "type": "string", "enum": ["Male", "Female", "Other"] },
    "date_measured": { "type": "string", "pattern": "^\\d{4}-\\d{2}-\\d{2}$" },
    "clinic": { "type": "string" }
  }
}
```

**Response types**  
- `text/plain` → Free-form text (default; human-friendly, machine-fragile)  
- `application/json` → Structured JSON (machine-friendly, schema-validatable)

---

## Our learning strategy

1. **Define** a JSON Schema that describes the exact fields we expect.  
2. **Configure** the model/SDK to return `application/json` (structured mode).  
3. **Validate** the response against the schema before using it.  
4. **Handle edge cases** gracefully (e.g., missing optional fields → `null`, collect errors).

---

## Before we code — prediction exercise

- Why is plain text risky for automated pipelines?  
- Which fields in our medical record are **required** vs **optional**, and why?  
- What validation errors do we expect to see first (dates, numbers, enums)?  
- How would we surface schema errors to users so they can fix the source data quickly?

In [ ]:
#@title Section 4: JSON Schema Creation

# from typing import TypedDict, List, Optional
# import json

class StandardizedRecord(TypedDict):
    """Schema for standardized medical records."""
    patient_id: str
    height_cm: Optional[float]
    weight_kg: Optional[float]
    gender: str
    date_measured: str  # ISO format YYYY-MM-DD
    clinic: str
    data_quality_issues: Optional[List[str]]
    original_units: Optional[Dict[str, str]]

def create_standardization_schema():
    """Create JSON schema for structured output."""
    return {
        "type": "array",
        "items": {
            "type": "object",
            "properties": {
                "patient_id": {"type": "string"},
                "height_cm": {"type": ["number", "null"]},
                "weight_kg": {"type": ["number", "null"]},
                "gender": {"type": "string", "enum": ["Male", "Female", "Other"]},
                "date_measured": {"type": "string", "pattern": "^\\d{4}-\\d{2}-\\d{2}$"},
                "clinic": {"type": "string"},
                "data_quality_issues": {
                    "type": ["array", "null"],
                    "items": {"type": "string"}
                },
                "original_units": {
                    "type": ["object", "null"],
                    "properties": {
                        "height": {"type": ["string", "null"]},  # Allow null here
                        "weight": {"type": ["string", "null"]}   # Allow null here
                    }
                }
            },
            "required": ["patient_id", "gender", "date_measured", "clinic"]
        }
    }

# Test schema creation
schema = create_standardization_schema()
Draft202012Validator(schema)  # will raise if invalid
print("✅ JSON schema created successfully")
print(f"   Schema defines an array of medical records with {len(schema['items']['properties'])} fields")

# Robust Batch Processing (Backoff + Parallelism)

## Why this matters

**Plain-language version**
APIs behave like busy checkouts: if you send one giant request, the cashier might say "too much at once" or time out. If you flood all cashiers at the same time, they also slow down.
**Solution**: break the work into **small batches**, send them in **parallel** (several cashiers), and when a cashier says "hold on," wait a little longer each time before trying again. This keeps the line moving and finishes the job **faster and more reliably**.

**Real-world problem**
- Providers set **rate limits** (requests per minute) and can have temporary hiccups.
- A single oversized request is more likely to **fail** or exceed **token limits**.

**Key insight**
- **Batching** keeps each request small and predictable.
- **Exponential backoff** avoids hammering a struggling service.
- **Parallelism** speeds things up by using multiple workers, while respecting rate limits.

---

## Conceptual foundation

**Exponential backoff** (with jitter)
- Wait time after the k-th failure: **delay = base × 2^k + random jitter**
- **Why it helps:**
  - Spreads out retries so you do not re-collide with everyone else.
  - Reduces load on the server when it is already under stress.
  - Increases the chance that the next attempt hits a healthy node.

**Batch sizing**
- Token budget is like luggage weight: keep each bag **under the limit**.
- **Practical rules:**
  - Rough estimate: **1 token ≈ 4 characters** of English text.
  - Keep a **~20% safety buffer** for the model's response.
  - Split your data so each request stays well under the **context window** and rate limits.

**Parallelism** (concurrency)
- Use **several small batches at once** (for example, 4 to 16 workers), not one massive request.
- **Benefits:**
  - Much **faster throughput** when each batch is small.
  - **Failures are isolated** to a batch, not your whole job.
- **Guardrails:**
  - **Cap concurrency** so you do not exceed requests-per-minute.
  - Add **random jitter** to start times to avoid synchronized spikes.
  - **Idempotency and deduping**: if a batch is retried, ensure you do not double-apply results.

**Rule of thumb**
- Start with **small batches** (8 to 20 items) and **4 to 8 parallel workers**.
- If you see rate-limit errors, **reduce concurrency** or **increase backoff**.
- If you see underutilization, carefully **increase concurrency**.

---

## Strategy

1. Calculate an **initial batch size** from token estimates and apply a **20% buffer**.
2. Process batches with a **progress bar**, so partial progress is visible.
3. Retry failed batches with **exponential backoff + jitter** (for example, base 1 s, then 2 s, 4 s...).
4. Run several batches in **parallel** (bounded worker pool) to finish sooner.
5. Aggregate results and **log metadata** (batch id, attempts, timings, errors) for **audit and reproducibility**.

---

## Before we code: prediction exercise

- If we switch from one huge request to **8 small parallel batches**, what changes do you expect in **completion time** and **failure rate**, and why?
- Where might real medical data break the plan (for example, very long **free-text notes** blowing up token count)?
- How will you detect and mitigate **rate-limit errors (HTTP 429)** versus true **validation errors** in the data?
- What evidence will convince you the configuration is **reliable** (success rate, p95 latency, zero duplicate records)?

In [ ]:
#@title Section 5: Robust Batch Processing (backoff, jitter, bounded parallelism)

import re
import random
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

def extract_valid_json(txt: str) -> str:
    """
    Extract valid JSON from potentially malformed LLM output.
    Handles common issues like markdown code blocks and incomplete JSON.

    Args:
        txt: Raw text that may contain JSON

    Returns:
        Valid JSON string or empty array if extraction fails
    """
    if not txt:
        return "[]"

    # Remove markdown code blocks
    s = txt.strip()
    s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s, flags=re.IGNORECASE | re.MULTILINE)

    # Find start of JSON (either [ or {)
    starts = [i for i in (s.find("["), s.find("{")) if i >= 0]
    if not starts:
        return "[]"

    start = min(starts)

    # Use bracket matching to find complete JSON
    stack = []
    pairs = {"{": "}", "[": "]"}
    end = None

    for i, ch in enumerate(s[start:], start):
        if ch in pairs:
            stack.append(pairs[ch])
        elif ch in ("]", "}"):
            if stack and ch == stack[-1]:
                stack.pop()
                if not stack:  # Found complete JSON
                    end = i + 1
                    break

    if end:
        json_str = s[start:end]
    else:
        # Try to salvage partial JSON
        json_str = s[start:]
        if json_str.startswith('['):
            # Attempt to close array properly
            last_brace = json_str.rfind('}')
            if last_brace > 0:
                json_str = json_str[:last_brace + 1] + ']'

    # Validate and return
    try:
        parsed = json.loads(json_str)
        return json.dumps(parsed)  # Re-serialize for consistency
    except json.JSONDecodeError:
        return "[]"

class ImprovedBatchProcessor:
    """
    Handles batch processing of data through LLM with robust error handling.
    Features:
    - Automatic batch sizing to avoid token limits
    - Exponential backoff retry logic
    - JSON extraction and validation
    - Processing audit trail
    """

    def __init__(self, model_config: Dict[str, Any], max_retries: int = 3, max_workers: int = 4):
        """
        Initialize batch processor.

        Args:
            model_config: Model configuration dictionary
            max_retries: Maximum retry attempts per batch
        """
        self.model_config = model_config
        self.max_retries = max_retries
        self.base_delay = 1.0  # Base delay for exponential backoff
        self.max_workers = max_workers  # Bounded concurrency for parallel batches
        self.processed_batches = []  # Audit trail
        self._audit_lock = threading.Lock()  # processed_batches is written from worker threads

    def calculate_batch_size(self, df: pd.DataFrame) -> int:
        """
        Calculate conservative batch size to avoid JSON truncation.
        Smaller batches are more reliable for structured output.

        Args:
            df: DataFrame to process

        Returns:
            Recommended batch size
        """
        n_records = len(df)

        if n_records <= 10:
            return n_records  # Small datasets in one batch
        elif n_records <= 30:
            return 10  # Medium datasets: 10 records per batch
        else:
            return 8   # Large datasets: 8 records per batch (conservative)

    def create_batches(self, df: pd.DataFrame) -> Generator[pd.DataFrame, None, None]:
        """
        Split dataframe into appropriately sized batches.

        Args:
            df: DataFrame to split

        Yields:
            DataFrame batches
        """
        batch_size = self.calculate_batch_size(df)
        n_batches = (len(df) + batch_size - 1) // batch_size

        print(f"\n📦 Splitting {len(df)} records into {n_batches} batches of ~{batch_size} records")
        print(f"   Using conservative batch size for robust JSON parsing")

        for i in range(0, len(df), batch_size):
            yield df.iloc[i:i + batch_size]

    def process_batch_with_retry(
        self,
        batch: pd.DataFrame,
        prompt_template: str,
        model: Any,
        *,
        response_schema: Any = None,
        local_validation_schema: Optional[dict] = None
    ) -> Optional[List[Dict]]:
        """
        Process a single batch with retry logic and error handling.

        Args:
            batch: DataFrame batch to process
            prompt_template: Prompt template with {data} placeholder
            model: Model instance for generation
            response_schema: Optional schema for structured output
            local_validation_schema: Optional JSON schema for validation

        Returns:
            List of processed records or None if all retries failed
        """
        # Generate unique batch ID for tracking
        batch_id = hashlib.md5(batch.to_json().encode()).hexdigest()[:8]

        for attempt in range(self.max_retries):
            try:
                # Format prompt with batch data
                prompt = prompt_template.format(
                    data=batch.to_json(orient='records', indent=2)
                )

                # Configure generation
                generation_config = {
                    "temperature": self.model_config.get('temperature', 0.2),
                    "max_output_tokens": int(globals().get("MAX_OUTPUT_TOKENS", 4096)),
                    "response_mime_type": "application/json",
                    "response_schema": response_schema,
                }

                # Generate response
                response = model.generate_content(prompt, generation_config=generation_config)
                raw_text = response.text or ""

                print(f"   Batch {batch_id}: Response length {len(raw_text)} chars")

                # Extract and parse JSON
                response_text = extract_valid_json(raw_text)
                result = json.loads(response_text)

                # Ensure result is a list
                if not isinstance(result, list):
                    result = [result] if isinstance(result, dict) else []

                # Normalise the identifier before validation: the schema declares
                # patient_id as a string, but models sometimes emit a bare number.
                for record in result:
                    if isinstance(record, dict) and "patient_id" in record:
                        record["patient_id"] = str(record["patient_id"])

                # Optional schema validation
                if local_validation_schema is not None:
                    validate(instance=result, schema=local_validation_schema)

                # Log successful processing (workers share this list)
                with self._audit_lock:
                    self.processed_batches.append({
                        'batch_id': batch_id,
                        'size': len(batch),
                        'returned': len(result),
                        'attempt': attempt + 1,
                        'timestamp': datetime.now().isoformat()
                    })

                print(f"   ✅ Batch {batch_id}: {len(result)} records processed")
                return result

            except Exception as e:
                if attempt < self.max_retries - 1:
                    # Exponential backoff with retry
                    # Exponential backoff with jitter, so that batches retried at
                    # the same moment do not collide again on the next attempt.
                    delay = self.base_delay * (2 ** attempt) + random.uniform(0.0, self.base_delay)
                    print(f"   ⚠️ Batch {batch_id} failed (attempt {attempt + 1}): {e}")
                    print(f"      Retrying in {delay:.1f} seconds...")
                    time.sleep(delay)
                else:
                    # Final failure
                    print(f"   ❌ Batch {batch_id} failed after {self.max_retries} attempts")
                    print(f"      Final error: {e}")
                    return None

    def process_batches_parallel(
        self,
        batches: List[pd.DataFrame],
        prompt_template: str,
        model: Any,
        **kwargs
    ) -> List[Optional[List[Dict]]]:
        """
        Send the batches through the API using a bounded pool of workers.

        Concurrency brings total wall time close to the slowest single batch
        instead of the sum of all batches, while max_workers caps the request
        rate so the provider is not flooded. Results are returned in the
        original batch order, independent of completion order.

        Args:
            batches: List of DataFrame batches
            prompt_template: Prompt template with a {data} placeholder
            model: Model instance for generation
            **kwargs: Forwarded to process_batch_with_retry

        Returns:
            One entry per batch: the parsed records, or None if the batch failed
        """
        results: List[Optional[List[Dict]]] = [None] * len(batches)
        if not batches:
            return results

        workers = max(1, min(self.max_workers, len(batches)))
        print(f"   Dispatching {len(batches)} batches across {workers} parallel workers")

        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {
                pool.submit(
                    self.process_batch_with_retry, batch, prompt_template, model, **kwargs
                ): position
                for position, batch in enumerate(batches)
            }
            for future in tqdm(as_completed(futures), total=len(futures), desc="Standardizing"):
                position = futures[future]
                try:
                    results[position] = future.result()
                except Exception as e:
                    print(f"   \u274c Batch {position + 1} raised an unexpected error: {e}")
                    results[position] = None

        return results

# Initialize the batch processor
# max_workers bounds concurrency; lower it if the provider returns HTTP 429.
processor = ImprovedBatchProcessor(MODEL_CATALOG[SELECTED_MODEL],
                                  max_workers=1 if str(SELECTED_MODEL).startswith(("fred/", "ollama/")) else 4)
print(f"✅ Batch processor initialized with model: {SELECTED_MODEL}")

# Data Standardization with LLMs — What This Code Does and Why

## Putting it all together!

---

## Big picture: what this part of the notebook accomplishes

1. **Takes raw tabular medical data** (height, weight, gender, dates, clinic).  
2. **Sends it to an LLM in small batches** with a **clear prompt** that explains how to standardize.  
3. **Asks for strict JSON output** so the pipeline can parse it deterministically.  
4. **Validates, cleans, and post-processes** the LLM response (sanity checks, numeric coercion, range limits).  
5. **Logs** what happened (batch sizes, attempts, failures) for auditability.  
6. **Falls back** to rule-based logic if the LLM API is unavailable or returns malformed data.

---

## Inputs and outputs at a glance

- **Input (DataFrame)**: Columns like `patient_id`, `height`, `weight`, `gender`, `date_measured`, `clinic`. Some may be missing or in mixed units/languages.
- **Output (DataFrame)**: Clean columns such as `patient_id` (string), `height_cm` (float), `weight_kg` (float), `gender` (enum: Male, Female, Other), `date_measured` (ISO `YYYY-MM-DD`), plus `comments_english` (comments translated to English), `original_units_height`, `original_units_weight`, and `llm_quality_notes` (null when no issue was found).

---

## Where the prompt lives (and what it says)

Inside the function `standardize_medical_data_improved`, there is a `prompt_template` string. It tells the model exactly how to standardize and which fields to return. The template looks like this (simplified):

```text
You are a medical data standardization expert. Convert the medical records to standardized JSON.

RULES:
1. Heights to cm: If <100 assume inches (x2.54). Else treat as cm.
2. Weights to kg: If >150 assume pounds (/2.205). Else treat as kg.
3. Dates to YYYY-MM-DD; fall back to 2024-01-15 if invalid.
4. Gender: "Male", "Female", or "Other".
5. Translate every comment to English (Swedish, Japanese, German, Spanish, ...).
6. Record the original units for height and weight.
7. Flag data quality problems, and leave llm_quality_notes null when there are none.

INPUT DATA:
{data}

Return ONLY a valid JSON array:
[
  {
    "patient_id": "1",
    "height_cm": 175.0,
    "weight_kg": 70.0,
    "gender": "Male",
    "date_measured": "2024-01-15",
    "clinic": "Clinic A",
    "comments_english": "Translated comment in English",
    "original_units_height": "cm",
    "original_units_weight": "kg",
    "llm_quality_notes": null
  }
]
```

- The `{data}` placeholder is filled with a JSON array of the current batch rows.  
- The output is requested as **JSON only** so the code can parse it reliably.

---

## How the batching and retries work

This section uses `ImprovedBatchProcessor` to make calls reliable:

- **Batch sizing**: For 50 rows, the processor may use batches of ~8 to reduce truncation risk and keep within token limits.
- **Retry with exponential backoff and jitter**: if a batch fails (network hiccup, rate limit), it waits about 1 s, then 2 s, then 4 s, each with up to 1 s of random jitter so that batches retried together do not collide again.
- **Robust JSON extraction**: Even if the model adds stray text, the helper `extract_valid_json` tries to pull out the valid JSON segment.
- **Per-batch logging**: Each batch stores an id, size, returned record count, attempts, and timestamp for auditing.

Why batches?  
- More robust (a single bad record does not break the whole job).  
- Easier to respect rate and token limits.  
- Faster overall, because `process_batches_parallel` runs several batches at once through a bounded worker pool and returns them in the original order.

---

## Structured output and validation

- Calls the model with a **structured output request** (`response_mime_type = "application/json"`).  
- Validates the returned JSON against the **JSON Schema** from Section 4 (shape, types, gender enum, ISO date). A batch that fails validation is retried, then replaced by the rule-based fallback.  
- Converts fields to numeric (`height_cm`, `weight_kg`) and rounds to reasonable precision.

---

## Post-processing and safety checks

Function `clean_and_validate_results_enhanced(df)` enforces basic realism:
- Heights outside [50, 250] cm -> set to null.  
- Weights outside [20, 300] kg -> set to null.  
- Coerce types (e.g., numeric columns) and normalize ids to strings.

If an LLM batch is missing some rows or fails completely:
- `create_fallback_records_enhanced(batch)` applies **simple rule-based logic** so the pipeline still produces usable output.

---

## What each core function is responsible for

- **`standardize_medical_data_improved(df, model, processor)`**  
  Orchestrates the flow: maps columns, builds batches, calls the model, aggregates results, and returns the standardized DataFrame with summaries.

- **`ImprovedBatchProcessor`**  
  Splits data into batches, retried calls with exponential backoff, extracts JSON safely, and logs per-batch metadata.

- **`create_fallback_records_enhanced(batch)`**  
  When the API fails, converts inches->cm and lbs->kg by heuristic rules, normalizes gender, and provides default dates/units.

- **`clean_and_validate_results_enhanced(df)`**  
  Final numeric coercion, range checks, rounding, and string types for identifiers.

- **`display_processing_summary(processor, results)`**  
  Emits counts of batches, avg attempts, total records, and records with quality issues.

---

## Token budget, cost, and speed

- **Token estimate**: Roughly 1 token ~ 4 characters of English.  
- Keep a **~20% buffer** below the context window to avoid truncation.  
- **Parallelism**: `ImprovedBatchProcessor` runs batches concurrently through a bounded pool (`max_workers=4` by default). Raise it for throughput, lower it if the provider starts returning HTTP 429.  
- **Temperature**: Keep it low (0.1–0.3) for deterministic, stable standardization.

---

## When things go wrong — debugging checklist

- A batch returns fewer records than sent  
  - Check the JSON extraction logs; the batch is retried, and `create_fallback_records_enhanced` fills the gap if every attempt fails.  
- Repeated HTTP 429 or timeouts  
  - Reduce concurrency; increase backoff and jitter; shrink batch size.  
- Nonsense values (e.g., 5000 kg)  
  - Verify post-processing limits; clamp or null out-of-range values.  
- Schema validation errors  
  - Confirm response is `application/json`; tighten the prompt; ensure enum values match exactly.

---

## Success criteria

- No exceptions in the pipeline with real-world messy inputs.  
- All numeric fields are typed, within realistic ranges.  
- Batch logs show limited retries and stable throughput.  
- Outputs match schema and are audit-ready (e.g., track `original_units`).

---

## Before we code — prediction exercise

- What do you expect to change after standardization (units, date formats, gender normalization)? Why?  
- Which edge cases are most likely to trip the model (free-text notes, missing patient_id)?  
- How will you verify that conversions (inches->cm, lbs->kg) were applied correctly and consistently across sources?

In [ ]:
#@title # Section 6: Enhanced LLM-Based Data Standardization

import re

# ---------------------------------------------------------------------------
# Deterministic unit parsers.
#
# Free-form clinical language needs a model. A unit string does not: "5'9\"" has
# exactly one meaning, and a regular expression cannot misread it as 59 inches.
# The LLM does the semantic work, and these parsers then override its arithmetic
# wherever the source value is unambiguous.
# ---------------------------------------------------------------------------

_FEET_INCHES_RE = re.compile(r"^\s*(\d+)\s*'\s*(\d+(?:\.\d+)?)\s*\"?\s*$")
_POUNDS_RE = re.compile(r"^\s*([0-9]+(?:\.[0-9]+)?)\s*(?:lb|lbs)\s*$", re.IGNORECASE)
_LB_PER_KG = 2.20462


def parse_height_display_to_cm(value) -> Optional[float]:
    """Convert a feet-and-inches display string to cm. Returns None if the value is not one."""
    if pd.isna(value):
        return None
    match = _FEET_INCHES_RE.match(str(value))
    if not match:
        return None
    feet, inches = float(match.group(1)), float(match.group(2))
    if inches >= 12:  # 5'13" is not a valid measurement
        return None
    return feet * 30.48 + inches * 2.54


def parse_weight_display_to_kg(value, system: Optional[str] = None) -> Optional[float]:
    """Convert a weight display value to kg when the unit is certain, else None."""
    if pd.isna(value):
        return None
    text = str(value).strip()
    match = _POUNDS_RE.match(text)
    if match:
        return float(match.group(1)) / _LB_PER_KG
    if str(system).strip().lower() != "imperial":
        return None  # Metric rows are already kg, nothing to convert
    try:
        return float(text) / _LB_PER_KG
    except ValueError:
        return None


def reconcile_with_source(
    standardized: pd.DataFrame,
    source: pd.DataFrame,
    tolerance_cm: float = 1.5,
    tolerance_kg: float = 1.5,
) -> pd.DataFrame:
    """
    Replace an LLM conversion whenever the source row can be converted by a rule.

    This is the classical safety net the notebook argues for: the model is kept
    for judgement, and deterministic arithmetic wins where it applies. Without
    it a single misread "5'5"" (139.7 cm instead of 165.1 cm) silently corrupts
    a record and, if that record is later held out, the whole evaluation.
    """
    if 'patient_id' not in standardized.columns or 'patient_id' not in source.columns:
        return standardized

    src = source.copy()
    src['patient_id'] = src['patient_id'].astype(str)
    src = src.drop_duplicates(subset='patient_id').set_index('patient_id')

    out = standardized.copy()
    out['patient_id'] = out['patient_id'].astype(str)

    corrected_heights, corrected_weights = 0, 0

    for position, patient_id in out['patient_id'].items():
        if patient_id not in src.index:
            continue
        row = src.loc[patient_id]

        if 'height_display' in src.columns:
            expected = parse_height_display_to_cm(row.get('height_display'))
            if expected is not None:
                actual = out.at[position, 'height_cm']
                if pd.isna(actual) or abs(float(actual) - expected) > tolerance_cm:
                    out.at[position, 'height_cm'] = round(expected, 1)
                    corrected_heights += 1

        if 'weight_display' in src.columns:
            expected = parse_weight_display_to_kg(
                row.get('weight_display'), row.get('measurement_system')
            )
            if expected is not None:
                actual = out.at[position, 'weight_kg']
                if pd.isna(actual) or abs(float(actual) - expected) > tolerance_kg:
                    out.at[position, 'weight_kg'] = round(expected, 1)
                    corrected_weights += 1

    if corrected_heights or corrected_weights:
        print(f"   \U0001f527 Deterministic reconciliation: corrected "
              f"{corrected_heights} height(s) and {corrected_weights} weight(s)")
    else:
        print("   \u2705 Deterministic reconciliation: no unit conversion disagreed")

    return out


# --- self-test of the deterministic parsers --------------------------------
# A safety net that has never fired is a safety net nobody has tested. These
# assertions run every time the cell executes and fail loudly on a regression.

assert abs(parse_height_display_to_cm("5'5\"") - 165.1) < 0.05
assert abs(parse_height_display_to_cm("6'2\"") - 187.96) < 0.05
assert parse_height_display_to_cm("175.0") is None    # metric display: nothing to convert
assert parse_height_display_to_cm("5'13\"") is None   # 13 inches is not a measurement
assert abs(parse_weight_display_to_kg("184.6", "imperial") - 83.7) < 0.1
assert abs(parse_weight_display_to_kg("154 lbs") - 69.9) < 0.1
assert parse_weight_display_to_kg("83.7", "metric") is None

_probe_source = pd.DataFrame({
    'patient_id': ['1'],
    'height_display': ["5'5\""],
    'weight_display': [150.0],
    'measurement_system': ['imperial'],
})
# 139.7 cm is what you get when 5'5" is misread as 55 inches
_probe_standardized = pd.DataFrame({'patient_id': ['1'], 'height_cm': [139.7], 'weight_kg': [68.0]})
_probe_result = reconcile_with_source(_probe_standardized, _probe_source)
assert abs(_probe_result.at[0, 'height_cm'] - 165.1) < 0.1, "feet-inches misread was not repaired"
print("\u2705 Unit parser and reconciliation self-test passed")


def standardize_medical_data_improved(
    df: pd.DataFrame,
    model: Any,
    processor: ImprovedBatchProcessor
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Enhanced standardization with comment translation and quality assessment.
    Converts mixed units, normalizes formats, translates comments, and identifies quality issues.

    Args:
        df: Raw dataframe to standardize
        model: Gemini model instance
        processor: Batch processor for reliable API calls

    Returns:
        Tuple of (Standardized dataframe, Original dataframe)
    """
    print(f"\n🚀 Starting enhanced data standardization...")
    print(f"Input columns: {list(df.columns)}")

    original_df = df.copy()

    # Map columns to expected medical format
    expected_cols = ['patient_id', 'height', 'weight', 'gender', 'date_measured', 'clinic', 'comments']
    actual_cols = df.columns.tolist()

    # Intelligent column mapping
    col_mapping = {}
    for expected in expected_cols:
        for actual in actual_cols:
            if expected in actual.lower() or actual in expected:
                col_mapping[actual] = expected
                break

    if col_mapping:
        df = df.rename(columns=col_mapping)
        print(f"Column mapping applied: {col_mapping}")

    # Enhanced standardization prompt template with translation and quality assessment
    prompt_template = """
You are a multilingual medical data standardization expert. Process the medical records and provide standardized JSON output.

STANDARDIZATION RULES:
1. Heights to cm:
   - "5'9\"" style strings mean feet THEN inches: cm = feet * 30.48 + inches * 2.54.
     Never glue the two digits together. 5'5" is 165.1 cm, it is NOT 55 inches.
   - A bare number below 100 is inches: multiply by 2.54.
   - A bare number of 100 or above is already cm: keep it.
   - If the row carries both a numeric height and a display string, trust the numeric one.
2. Weights to kg: If >150 assume pounds, divide by 2.205. Otherwise keep as kg.
3. Dates to YYYY-MM-DD format. Use 2024-01-15 if invalid.
4. Gender to "Male", "Female", or "Other" (handle multilingual: svenska=kvinna/man, 日本語=女性/男性, Deutsch=Frau/Mann)
5. Translate ALL comments to English (detect: Swedish, Japanese, German, Spanish, etc.)
6. Assess data quality and flag suspicious patterns. ONLY populate llm_quality_notes if issues are found.


QUALITY ASSESSMENT CRITERIA:
- Extreme BMI values (height/weight mismatch)
- Contradictory information in comments vs measurements
- Medical impossibilities or typos in comments
- Unit conversion errors or inconsistencies
- Missing critical information

INPUT DATA:
{data}

Return valid JSON array with this EXACT structure:
[
  {{
    "patient_id": "1",
    "height_cm": 175.0,
    "weight_kg": 70.0,
    "gender": "Male",
    "date_measured": "2024-01-15",
    "clinic": "Clinic A",
    "comments_english": "Translated comment in English",
    "original_units_height": "cm",
    "original_units_weight": "kg",
    "llm_quality_notes": null
  }}
]
IMPORTANT: Set llm_quality_notes to null if no issues found. Only use text like "WARNING: Height/weight contradict comment" when actual problems exist.

CRITICAL: Return ONLY valid JSON array, no markdown, no explanations!
"""

    # Check if model is available
    if model is None:
        print("\n⚠️ No model available, using fallback standardization")
        return create_fallback_standardization_enhanced(df, original_df)

    # Process batches with enhanced processing
    all_results = []
    batches = list(processor.create_batches(df))

    print(f"\n🔄 Processing {len(batches)} batches with translation and quality assessment...")

    # The JSON Schema from Section 4 is enforced on every batch, so a malformed
    # response is retried rather than flowing into the pipeline unchecked.
    validation_schema = create_standardization_schema()

    batch_results = processor.process_batches_parallel(
        batches, prompt_template, model,
        local_validation_schema=validation_schema
    )

    for i, (batch, result) in enumerate(zip(batches, batch_results)):
        if result:
            all_results.extend(result)
            print(f"   Batch {i+1}/{len(batches)}: Added {len(result)} records")
        else:
            print(f"   Batch {i+1}/{len(batches)}: Failed, using fallback")
            # Create enhanced fallback records
            all_results.extend(create_fallback_records_enhanced(batch))

    # Convert results to DataFrame
    if all_results:
        standardized_df = pd.DataFrame(all_results)
        print(f"\n✅ Standardized {len(standardized_df)} records")

        # Deterministic override wherever the source unit is unambiguous
        standardized_df = reconcile_with_source(standardized_df, original_df)

        # Clean and validate results with enhanced processing
        standardized_df = clean_and_validate_results_enhanced(standardized_df)

        # Display processing summary
        display_processing_summary(processor, all_results)

        return standardized_df, original_df
    else:
        print("\n❌ No data could be standardized")
        return create_fallback_standardization_enhanced(df, original_df)

def create_fallback_records_enhanced(batch: pd.DataFrame) -> List[Dict]:
    """
    Create enhanced fallback records with basic translation and quality assessment.
    Used when API processing fails.
    """
    fallback_records = []

    # Basic translation dictionary for common terms
    translation_dict = {
        # Swedish
        'allergisk': 'allergic', 'mot': 'to', 'penicillin': 'penicillin',
        'längd': 'height', 'vikt': 'weight', 'blodtryck': 'blood pressure',
        'patient': 'patient', 'klagen': 'complains', 'över': 'about',
        'ryggsmärta': 'back pain', 'ej tillgänglig': 'not available',
        'se tidigare': 'see previous', 'antekningar': 'notes',

        # German
        'größe': 'height', 'gewicht': 'weight', 'ohne': 'without',
        'schuhe': 'shoes', 'gemessen': 'measured', 'klagt': 'complains',
        'rückenschmerzen': 'back pain', 'seit': 'since', 'wochen': 'weeks',
        'diabetis': 'diabetes', 'typ': 'type', 'gegen': 'against',

        # Basic Japanese (simplified)
        '患者': 'patient', '身長': 'height', '体重': 'weight',
        '血圧': 'blood pressure', '測定': 'measured', '靴': 'shoes',
        '脱いで': 'without', '記録': 'recorded', '昼食前': 'before lunch',
        'アレルギー': 'allergy', 'ノート': 'notes', '参照': 'reference',
        '利用できません': 'not available', '前回の記録を参照': 'see previous record'
    }

    for _, row in batch.iterrows():
        # Enhanced gender standardization for multilingual input
        gender = "Other"
        if 'gender' in row and pd.notna(row['gender']):
            gender_str = str(row['gender']).lower().strip()

            # Comprehensive multilingual gender mapping
            male_variants = [
                'm', 'male', 'mann', 'masculino', 'homme', 'мужской',  # Basic
                'man', 'pojke', 'män',  # Swedish
                '男性', '男', 'おとこ',  # Japanese
                'männlich', 'herr',  # German
                'hombre'  # Spanish
            ]
            female_variants = [
                'f', 'female', 'frau', 'woman', 'femenino', 'femme', 'женский',  # Basic
                'kvinna', 'flicka', 'kvinnlig',  # Swedish
                '女性', '女', 'おんな',  # Japanese
                'weiblich', 'dame',  # German
                'mujer'  # Spanish
            ]

            if gender_str in male_variants:
                gender = "Male"
            elif gender_str in female_variants:
                gender = "Female"

        # Smart unit detection with enhanced logic
        height_cm = None
        weight_kg = None
        height_unit = "unknown"
        weight_unit = "unknown"

        if 'height' in row and pd.notna(row['height']):
            height_val = float(row['height'])
            if height_val < 100:  # Likely inches
                height_cm = round(height_val * 2.54, 1)
                height_unit = "inches"
            else:
                height_cm = round(height_val, 1)
                height_unit = "cm"

        if 'weight' in row and pd.notna(row['weight']):
            weight_val = float(row['weight'])
            if weight_val > 150:  # Likely pounds
                weight_kg = round(weight_val / 2.205, 1)
                weight_unit = "pounds"
            else:
                weight_kg = round(weight_val, 1)
                weight_unit = "kg"

        # Basic comment translation
        comments_english = None
        if 'comments' in row and pd.notna(row['comments']):
            comment = str(row['comments']).lower()

            # Simple word-by-word replacement
            translated = comment
            for foreign, english in translation_dict.items():
                translated = translated.replace(foreign, english)

            comments_english = translated.title()  # Capitalize properly

        # Basic quality assessment
        quality_issues = []
        if height_cm and weight_kg:
            bmi = weight_kg / ((height_cm / 100) ** 2)
            if bmi < 15 or bmi > 40:
                quality_issues.append(f"Extreme BMI: {bmi:.1f}")

        if height_cm and (height_cm < 50 or height_cm > 250):
            quality_issues.append("Height outside normal range")
        if weight_kg and (weight_kg < 10 or weight_kg > 300):
            quality_issues.append("Weight outside normal range")

        llm_quality_notes = "; ".join(quality_issues) if quality_issues else None

        # Enhanced date handling
        date_measured = "2024-01-15"
        if 'date_measured' in row and pd.notna(row['date_measured']):
            try:
                import dateutil.parser
                parsed_date = dateutil.parser.parse(str(row['date_measured']))
                date_measured = parsed_date.strftime("%Y-%m-%d")
            except (ValueError, OverflowError, TypeError):
                pass  # Keep the default date when the value cannot be parsed

        fallback_record = {
            "patient_id": str(row.get('patient_id', 'unknown')),
            "height_cm": height_cm,
            "weight_kg": weight_kg,
            "gender": gender,
            "date_measured": date_measured,
            "clinic": str(row.get('clinic', 'Unknown')),
            "comments_english": comments_english,
            "original_units_height": height_unit,
            "original_units_weight": weight_unit,
            "llm_quality_notes": llm_quality_notes
        }
        fallback_records.append(fallback_record)

    return fallback_records

def clean_and_validate_results_enhanced(df: pd.DataFrame) -> pd.DataFrame:
    """
    Enhanced cleaning and validation with better formatting.
    """
    # Ensure patient_id is string
    df['patient_id'] = df['patient_id'].astype(str)

    # Clean numeric columns
    for col in ['height_cm', 'weight_kg']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].round(1)

    # Validate reasonable ranges
    if 'height_cm' in df.columns:
        invalid_heights = (df['height_cm'].notna() &
                          ((df['height_cm'] < 50) | (df['height_cm'] > 250)))
        if invalid_heights.any():
            print(f"   ⚠️ Fixed {invalid_heights.sum()} invalid height values")
            df.loc[invalid_heights, 'height_cm'] = None

    if 'weight_kg' in df.columns:
        invalid_weights = (df['weight_kg'].notna() &
                          ((df['weight_kg'] < 20) | (df['weight_kg'] > 300)))
        if invalid_weights.any():
            print(f"   ⚠️ Fixed {invalid_weights.sum()} invalid weight values")
            df.loc[invalid_weights, 'weight_kg'] = None

    # Clean up unit display format
    if 'original_units_height' in df.columns and 'original_units_weight' in df.columns:
        # Create a clean combined units column for display
        df['original_units'] = df['original_units_height'].astype(str) + "/" + df['original_units_weight'].astype(str)
        # Replace ugly formats
        df['original_units'] = df['original_units'].replace({
            'nan/nan': 'unknown/unknown',
            'None/None': 'unknown/unknown'
        })

    return df

def create_fallback_standardization_enhanced(df: pd.DataFrame, original_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Create enhanced fallback standardization when model is unavailable."""
    fallback_records = create_fallback_records_enhanced(df)
    standardized_df = pd.DataFrame(fallback_records)

    # The same deterministic safety net applies to the rule-based path
    standardized_df = reconcile_with_source(standardized_df, original_df)

    # Apply enhanced cleaning
    standardized_df = clean_and_validate_results_enhanced(standardized_df)

    return standardized_df, original_df

def display_processing_summary(processor: ImprovedBatchProcessor, results: List[Dict]):
    """Display summary of processing statistics."""
    summary = processor.processed_batches
    if summary:
        total_batches = len(summary)
        avg_attempts = sum(b['attempt'] for b in summary) / total_batches
        total_records = sum(b['returned'] for b in summary)

        print(f"\n📊 Processing summary:")
        print(f"   Total batches: {total_batches}")
        print(f"   Average attempts: {avg_attempts:.1f}")
        print(f"   Total records processed: {total_records}")

        # Count translations and quality issues
        translations = sum(1 for r in results if r.get('comments_english'))
        quality_flags = sum(1 for r in results if r.get('llm_quality_notes') and 'WARNING' in str(r.get('llm_quality_notes', '')))

        if translations > 0:
            print(f"   Comments translated: {translations}")
        if quality_flags > 0:
            print(f"   Quality warnings: {quality_flags}")

# Run enhanced standardization
print(f"\n🔍 Model status check:")
print(f"   Model available: {'model' in locals() and model is not None}")

# Standardize the data with enhancements
result = standardize_medical_data_improved(df, model, processor)

# Handle results
if isinstance(result, tuple):
    standardized_df, original_medical_df = result
else:
    standardized_df = result
    original_medical_df = df

# Display results with enhanced formatting
if not standardized_df.empty:
    print("\n📊 Enhanced standardization results:")

    print("\nBEFORE standardization (first 10 records):")
    print(format_for_display(original_medical_df.head(10)).to_string())

    print("\nAFTER standardization (first 10 records):")
    # Create display version with clean formatting
    display_df = format_for_display(standardized_df.head(10)).copy()

    # Reorder columns for better display
    preferred_cols = ['patient_id', 'height_cm', 'weight_kg', 'gender', 'date_measured',
                     'clinic', 'comments_english', 'original_units', 'llm_quality_notes']

    # Only include columns that exist
    display_cols = [col for col in preferred_cols if col in display_df.columns]
    display_df = display_df[display_cols]

    print(display_df.to_string())

    # Enhanced quality summary
    print("\n📈 Enhanced data quality summary:")
    print(f"   Records processed: {len(standardized_df)}")

    if 'height_cm' in standardized_df.columns:
        valid_heights = standardized_df['height_cm'].notna().sum()
        print(f"   Valid heights: {valid_heights} ({valid_heights/len(standardized_df)*100:.1f}%)")

    if 'weight_kg' in standardized_df.columns:
        valid_weights = standardized_df['weight_kg'].notna().sum()
        print(f"   Valid weights: {valid_weights} ({valid_weights/len(standardized_df)*100:.1f}%)")

    if 'comments_english' in standardized_df.columns:
        translated = standardized_df['comments_english'].notna().sum()
        print(f"   Comments translated: {translated} ({translated/len(standardized_df)*100:.1f}%)")

    if 'llm_quality_notes' in standardized_df.columns:
        quality_warnings = standardized_df['llm_quality_notes'].str.contains('WARNING|Extreme|outside', na=False).sum()
        print(f"   Quality warnings: {quality_warnings} ({quality_warnings/len(standardized_df)*100:.1f}%)")

else:
    print("\n⚠️ No standardized data produced")

print("\n✅ Enhanced standardization complete!")

# Classical Imputation Methods: Theory and Practice

fill-missing.svg

## Why This Matters

**Real-world problem**: Missing data is ubiquitous in medical datasets. Sensors fail, patients skip questions, lab results get lost. Poor handling of missingness can bias your analysis, reduce statistical power, and lead to wrong conclusions that affect patient care.

**Traditional limitations**: Simply deleting incomplete records wastes information and can introduce selection bias. Naive approaches like "fill with zero" often violate the underlying data distribution and correlations.

**Key insight**: Classical imputation methods encode explicit assumptions about why data is missing and how variables relate to each other. Understanding these assumptions helps you choose the right tool and validate whether your imputation makes sense.

**Learning objective**: You will gain the engineering judgment to select appropriate imputation strategies based on missing data patterns, validate imputation quality, and implement robust preprocessing pipelines.

## Conceptual Foundation

**Missing data mechanisms** (fundamental theory):

Let $R_{ij}$ be the missingness indicator (1 if missing, 0 if observed) for variable $j$ in record $i$. Let $X$ be the complete data matrix and $X_{obs}$ be the observed portion.

- **MCAR (Missing Completely At Random)**: The probability that a value is missing is independent of both observed and unobserved data.
  $$P(R | X) = P(R)$$
  Example: a sensor randomly drops 5% of readings due to power fluctuations.

- **MAR (Missing At Random)**: Missingness depends only on observed variables, not the missing value itself.
  $$P(R | X) = P(R | X_{obs})$$
  Example: height measurements are missing more often at Clinic A than Clinic B, but once you condition on clinic, missingness is random.

- **MNAR (Missing Not At Random)**: Missingness depends on the unobserved value itself.
  $$P(R | X)$$
  Example: patients with very high blood pressure are less likely to complete follow-up visits.

> Most classical methods are unbiased only under MCAR or MAR (with a correct model). Under MNAR, you typically need a missingness model or external information.

**Why this matters for method selection**:
- Most classical methods assume MCAR or MAR
- Under MNAR, you typically need specialized models or external information  
- Wrong assumptions lead to biased estimates and invalid confidence intervals

**Physical principles in medical data**:
- Height and weight are positively correlated (BMI relationship)
- Age-related trends in physiological measurements
- Population differences by demographics (gender, ethnicity)
- Measurement error and instrument precision limits

## Setup and Notation

Let variable $X_j$ have missing entries, and $Z$ be the set of other observed predictors. We impute a missing $x_{i,j}$ for row $i$. The set $\mathcal{O}_j$ denotes indices where variable $X_j$ is observed.

## Our Learning Strategy

**Progressive approach**:
1. Start with simple univariate methods (mean/median)
2. Add relationships between variables (regression, KNN)  
3. Model joint distributions (iterative methods)
4. Validate using controlled experiments

**Key experiments**:
- Create artificial missingness to test method performance
- Compare imputed vs true values on held-out data
- Test robustness to different missing data patterns

**Success criteria**:
- Low mean absolute error on validation set
- Preserved correlation structure
- Realistic value distributions
- Stable performance across different missingness patterns

## Before We Code: Prediction Exercise

- Which variables in your dataset are most predictable from others, and why?
- If you increase k in KNN from 3 to 15, what happens to bias vs variance of imputations?
- Under MAR, why does regression/MICE generally beat mean imputation?
- How will you prove your imputation method works for your specific use case?

## 1. Mean and Median Imputation

**Intuition**: Replace missing values with a "typical" value. Fast and robust (median handles outliers), but ignores relationships between variables and reduces variability.

**Mathematical formulation**:

For variable $X_j$ with observed values in set $\mathcal{O}_j$:

- **Mean imputation**:
  $$\hat{x}_{i,j} = \bar{x}_j = \frac{1}{|\mathcal{O}_j|}\sum_{k \in \mathcal{O}_j} x_{k,j}$$

- **Median imputation**:
  $$\hat{x}_{i,j} = \text{median}\{x_{k,j} : k \in \mathcal{O}_j\}$$

- **Conditional mean** (within groups $g$):
  $$\hat{x}_{i,j} = \bar{x}_{j|g_i} = \frac{1}{|\mathcal{O}_{j,g_i}|}\sum_{k \in \mathcal{O}_{j,g_i}} x_{k,j}$$

where $|\mathcal{O}_j|$ denotes the number of observed values for variable $j$.

**When it works**: MCAR with low missingness rates. Good baseline for quick prototypes.

**Effects and limitations**:
- Reduces variance of $X_j$: $\text{Var}(\hat{X}_j) < \text{Var}(X_j)$
- Attenuates correlations with other variables
- Can create values that the variable cannot actually take (e.g., a mean age of 25.7 years for an integer field)
- No uncertainty quantification
- Bias is small under MCAR with low missingness; can be problematic under MAR/MNAR

**Implementation considerations**:
- Use grouped means when natural strata exist (clinic, gender, age group)
- Round to appropriate precision for discrete variables
- Check for outliers that might skew the mean

In [ ]:
#@title Figure: Mean vs Median Imputation
# Import libraries and generate data
# import numpy as np
# import matplotlib.pyplot as plt
# np.random.seed(42)

# Generate correlated 2D data (height vs weight style)
n = 50
x = np.random.normal(170, 10, n)  # height in cm
y = 2.5 * x - 250 + np.random.normal(0, 8, n)  # weight = f(height) + noise

# Create 30% random missingness
mask = np.random.choice([True, False], n, p=[0.3, 0.7])
x_obs, y_obs = x[~mask], y[~mask]
x_miss, y_miss_true = x[mask], y[mask]

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean Imputation (left subplot)
axes[0].scatter(x_obs, y_obs, alpha=0.6, label='Observed', s=50)
axes[0].axhline(y=np.mean(y_obs), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(y_obs):.1f}')
axes[0].scatter(x_miss, [np.mean(y_obs)]*len(x_miss), color='red', marker='x', s=100, label='Imputed')
axes[0].scatter(x_miss, y_miss_true, alpha=0.3, color='gray', label='True (hidden)')
axes[0].set_xlabel('Height (cm)')
axes[0].set_ylabel('Weight (kg)')
axes[0].set_title('Mean Imputation: All missing → mean(observed)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Median Imputation (right subplot)
axes[1].scatter(x_obs, y_obs, alpha=0.6, label='Observed', s=50)
axes[1].axhline(y=np.median(y_obs), color='orange', linestyle='--', linewidth=2, label=f'Median = {np.median(y_obs):.1f}')
axes[1].scatter(x_miss, [np.median(y_obs)]*len(x_miss), color='orange', marker='x', s=100, label='Imputed')
axes[1].scatter(x_miss, y_miss_true, alpha=0.3, color='gray', label='True (hidden)')
axes[1].set_xlabel('Height (cm)')
axes[1].set_ylabel('Weight (kg)')
axes[1].set_title('Median Imputation: All missing → median(observed)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Mean vs Median Imputation Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Linear Regression Imputation

**Intuition**: Predict missing values using linear relationships with other variables. Exploits correlation structure but assumes linear relationships.

**Mathematical formulation**:

Fit regression model on complete cases where both $X_j$ and predictors $Z$ are observed:
$$X_j = Z\beta + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

**Parameter estimation** (ordinary least squares):
$$\hat{\beta} = (Z^T Z)^{-1} Z^T x_j$$

**Prediction** for missing values:
$$\hat{x}_{i,j} = z_i^T \hat{\beta}$$

**Stochastic vs deterministic**:
- **Deterministic**: Use $\hat{x}_{i,j}$ directly
- **Stochastic**: Add noise to preserve variability:
  $$\tilde{x}_{i,j} = \hat{x}_{i,j} + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \hat{\sigma}^2)$$

where the residual variance is:
$$\hat{\sigma}^2 = \frac{1}{n-p-1}\sum_{k \in \text{complete}}(x_{k,j} - z_k^T\hat{\beta})^2$$

**When it works**: MAR with reasonably linear relationships and sufficient complete cases.

**Limitations**:
- Extrapolation risk when predictors outside training range
- Sensitive to multicollinearity and outliers
- Assumes linear relationships (may miss nonlinear patterns)
- Regression to the mean effect

In [ ]:
#@title Figure: Linear Regression Imputation

# Import libraries and generate data (if not already done)
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.linear_model import LinearRegression
# np.random.seed(42)

# Generate correlated 2D data (height vs weight style)
n = 50
x = np.random.normal(170, 10, n)  # height in cm
y = 2.5 * x - 250 + np.random.normal(0, 8, n)  # weight = f(height) + noise

# Create 30% random missingness
mask = np.random.choice([True, False], n, p=[0.3, 0.7])
x_obs, y_obs = x[~mask], y[~mask]
x_miss, y_miss_true = x[mask], y[mask]

# Regression Imputation - Shows fitted line and predictions
plt.figure(figsize=(8, 5))

# Fit regression line
reg = LinearRegression().fit(x_obs.reshape(-1, 1), y_obs)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = reg.predict(x_line.reshape(-1, 1))

plt.scatter(x_obs, y_obs, alpha=0.6, label='Observed', s=50)
plt.plot(x_line, y_line, 'b-', linewidth=2, label='Fitted line')
y_imputed = reg.predict(x_miss.reshape(-1, 1))
plt.scatter(x_miss, y_imputed, color='red', marker='x', s=100, label='Imputed')
plt.scatter(x_miss, y_miss_true, alpha=0.3, color='gray', label='True (hidden)')

# Show prediction lines
for xi, yi in zip(x_miss[:3], y_imputed[:3]):  # Show first 3
    plt.plot([xi, xi], [min(y), yi], 'r--', alpha=0.3, linewidth=1)

plt.xlabel('Height (cm)'); plt.ylabel('Weight (kg)')
plt.title('Regression Imputation: Missing values → predicted from line')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 3. K-Nearest Neighbors (KNN) Imputation

**Intuition**: Find similar records and borrow their values. Preserves local structure and can capture nonlinear relationships.

**Mathematical formulation**:

**Distance metric** (after standardization):
$$d(i, \ell) = \sqrt{\sum_{m \in \text{observed for both } i,\ell} \frac{(z_{i,m} - z_{\ell,m})^2}{\sigma_m^2}}$$

**Weight function**:
$$w_\ell = \frac{1}{d(i,\ell) + \varepsilon} \quad \text{or} \quad w_\ell = \exp\left(-\frac{d(i,\ell)^2}{2h^2}\right)$$

**Imputation** (for continuous variables):
$$\hat{x}_{i,j} = \frac{\sum_{\ell \in \mathcal{N}_k(i)} w_\ell \cdot x_{\ell,j}}{\sum_{\ell \in \mathcal{N}_k(i)} w_\ell}$$

where $\mathcal{N}_k(i)$ are the $k$ nearest neighbors of record $i$.

**For categorical variables**: Use weighted majority vote:
$$\hat{x}_{i,j} = \arg\max_{c} \sum_{\ell \in \mathcal{N}_k(i): x_{\ell,j}=c} w_\ell$$

**Preprocessing requirements**:
- Standardize continuous variables: $z_m = \frac{x_m - \mu_m}{\sigma_m}$
- One-hot encode categorical variables
- Handle different data types appropriately in distance calculation

**Hyperparameter tuning**:
- Choose $k \in [3, 10]$ typically
- Validate using cross-validation or masking experiments
- Consider $k = \sqrt{n}$ as starting point

**When it works**: Moderate dataset sizes with local similarity patterns.

**Limitations**:
- Computational cost grows with dataset size: $O(n^2)$ for naive implementation
- Curse of dimensionality in high dimensions
- Sensitive to irrelevant features and scaling
- Choice of $k$ affects bias-variance tradeoff

**Tuning tips**:
- Choose $k$ in the range 3–10; validate via a masking experiment
- Always scale features; otherwise one variable can dominate distances
- For categorical targets, use majority vote (optionally weighted)

In [ ]:
#@title Figure: KNN Imputation - Interactive
# Import libraries and generate data (if not already done)
# import numpy as np
# import matplotlib.pyplot as plt
# from ipywidgets import interact
# from sklearn.impute import KNNImputer
# np.random.seed(42)

# Generate correlated 2D data (height vs weight style)
n = 50
x = np.random.normal(170, 10, n)  # height in cm
y = 2.5 * x - 250 + np.random.normal(0, 8, n)  # weight = f(height) + noise

# Create 30% random missingness
mask = np.random.choice([True, False], n, p=[0.3, 0.7])
x_obs, y_obs = x[~mask], y[~mask]
x_miss, y_miss_true = x[mask], y[mask]

# KNN Imputation - Interactive visualization
def plot_knn(k=5):
    plt.figure(figsize=(10, 5))

    # Left plot: Show one example point and its neighbors
    plt.subplot(1, 2, 1)
    plt.scatter(x_obs, y_obs, alpha=0.6, s=50)

    # Pick first missing point to demonstrate
    target_x = x_miss[0]
    distances = np.abs(x_obs - target_x)
    neighbor_idx = np.argsort(distances)[:k]

    # Highlight neighbors
    plt.scatter(x_obs[neighbor_idx], y_obs[neighbor_idx],
                color='orange', s=200, alpha=0.5, label=f'{k} neighbors')

    # Show imputed value
    y_imputed = np.mean(y_obs[neighbor_idx])
    plt.scatter(target_x, y_imputed, color='red', marker='x', s=200, label='Imputed')
    plt.scatter(target_x, y_miss_true[0], alpha=0.3, color='gray', s=100, label='True')

    # Draw connections
    for idx in neighbor_idx:
        plt.plot([target_x, x_obs[idx]], [y_imputed, y_obs[idx]],
                'orange', alpha=0.2, linewidth=1)

    plt.xlabel('Height (cm)'); plt.ylabel('Weight (kg)')
    plt.title(f'KNN: Average of {k} nearest neighbors')
    plt.legend()

    # Right plot: All imputations
    plt.subplot(1, 2, 2)
    data = np.column_stack([x, y])
    data[mask, 1] = np.nan
    imputer = KNNImputer(n_neighbors=k)
    data_imp = imputer.fit_transform(data)

    plt.scatter(x_obs, y_obs, alpha=0.6, label='Observed', s=50)
    plt.scatter(x_miss, data_imp[mask, 1], color='red', marker='x', s=100, label='All imputed')
    plt.scatter(x_miss, y_miss_true, alpha=0.3, color='gray', label='True (hidden)')
    plt.xlabel('Height (cm)'); plt.ylabel('Weight (kg)')
    plt.title(f'All points imputed with k={k}')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_knn, k=(1, 15, 1))

## 4. Iterative Imputation (MICE - Multiple Imputation by Chained Equations)

**Intuition**: Model each variable with missing data as a function of others, iterate until convergence. Captures complex multivariate relationships.

**Algorithm** (Multiple Imputation by Chained Equations):

1. **Initialize**: Fill missing values with simple method (e.g., mean)
   $$X^{(0)} = \text{initial_imputation}(X_{obs})$$

2. **Iterate** for $t = 1, 2, \ldots, T$:
   
   For each variable $j$ with missing values:
   
   a) **Fit model**: $f_j: X_j \leftarrow X_{-j}$ on observed $X_j$
   
   b) **Predict**: For missing values in $X_j$:
   $$\hat{x}_{i,j}^{(t)} = f_j(x_{i,-j}^{(t-1)}) + \epsilon_{ij}^{(t)}$$
   
   c) **Update**: $x_{i,j}^{(t)} = \hat{x}_{i,j}^{(t)}$ for missing positions

3. **Convergence**: Stop when $||X^{(t)} - X^{(t-1)}||_F < \tau$

**Model choices for $f_j$**:
- **Linear regression**: For continuous variables
- **Logistic regression**: For binary variables  
- **Random forests**: For nonlinear relationships
- **Bayesian models**: For uncertainty quantification

**Multiple imputation extension**:
Run $m$ independent chains with different random seeds:
$$\{X^{(1)}, X^{(2)}, \ldots, X^{(m)}\}$$

**Rubin's rules** for combining estimates:
- Point estimate: $\bar{\theta} = \frac{1}{m}\sum_{i=1}^m \hat{\theta}^{(i)}$
- Total variance: $V = U + \left(1 + \frac{1}{m}\right)B$

where $U$ = within-imputation variance, $B$ = between-imputation variance

**When it works**: MAR with adequate models and sufficient iterations.

**Why it helps**: Under MAR (and adequate models), it preserves joint relationships and yields better uncertainty behavior than single imputation.

**Advantages**:
- Preserves joint distribution structure
- Provides proper uncertainty quantification
- Flexible model specification
- Theoretically sound under MAR

**Caveats**:
- Computationally intensive
- Requires careful model selection for each variable
- Convergence diagnostics needed
- May not converge if models are misspecified

In [ ]:
#@title Figure: Iterative Imputation (MICE)

# Complete imports for MICE with different estimators
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# MICE internal estimator selector - change this to test different algorithms
ESTIMATOR = "BayesianRidge"  # Options: "BayesianRidge", "LinearRegression", "RandomForest", "ExtraTrees"

# Generate data with missingness
np.random.seed(42)
n, missing_rate = 50, 0.3
x = np.random.normal(170, 10, n)
y = 2.5 * x - 250 + np.random.normal(0, 15, n) + 0.01 * (x - 170)**2
mask = np.random.choice([True, False], n, p=[missing_rate, 1-missing_rate])
data = np.column_stack([x, y])
data[mask, 1] = np.nan

# Choose MICE internal estimator
estimators = {
    "BayesianRidge": BayesianRidge(),
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=10, random_state=42),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=10, random_state=42)
}

# Run MICE and get actual iterations used
imputer = IterativeImputer(
    estimator=estimators[ESTIMATOR],
    max_iter=30,  # Allow up to 30 iterations
    random_state=42,
    tol=1e-6
)
data_imp = imputer.fit_transform(data.copy())
imputed_vals = data_imp[mask, 1]

# Single plot showing results
plt.figure(figsize=(8, 6))
plt.scatter(x[~mask], y[~mask], alpha=0.6, s=50, label='Observed', color='blue')
plt.scatter(x[mask], imputed_vals, color='red', marker='x', s=100,
            label=f'MICE-{ESTIMATOR} imputed')
plt.scatter(x[mask], y[mask], alpha=0.4, color='gray', s=30, label='True values')

# Calculate and show results
mse = np.mean((imputed_vals - y[mask])**2)
actual_iters = imputer.n_iter_

plt.title(f'MICE-{ESTIMATOR}: Converged in {actual_iters} iterations (MSE={mse:.1f})')
plt.xlabel('Height (cm)')
plt.ylabel('Weight (kg)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print convergence info
print(f"🔍 MICE-{ESTIMATOR} Results:")
print(f"   └─ Actual iterations used: {actual_iters} out of max 30")
print(f"   └─ Convergence tolerance (configured): {imputer.tol}")
print(f"   └─ Imputation MSE: {mse:.2f}")

In [ ]:
#@title Figure: Classical Imputation Methods Comparison

#red star = imputation, green dot = true value

# Import libraries and generate data
from sklearn.linear_model import LinearRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

np.random.seed(42)

# Generate correlated 2D data (height vs weight style)
n = 50
x = np.random.normal(170, 10, n)  # height in cm
y = 2.5 * x - 250 + np.random.normal(0, 8, n)  # weight = f(height) + noise

# Create 30% random missingness
mask = np.random.choice([True, False], n, p=[0.3, 0.7])
x_obs, y_obs = x[~mask], y[~mask]
x_miss, y_miss_true = x[mask], y[mask]

# Set target point at height=155 (with true value following the relationship)
target_x = 155
target_y_true = 2.5 * target_x - 250 + np.random.normal(0, 8)  # True value with some noise

# Calculate all imputation values first to determine scale
# Mean
mean_imputed = np.mean(y_obs)

# Regression
reg = LinearRegression().fit(x_obs.reshape(-1, 1), y_obs)
reg_imputed = reg.predict([[target_x]])[0]

# KNN
distances = np.abs(x_obs - target_x)
neighbor_idx = np.argsort(distances)[:5]
knn_imputed = np.mean(y_obs[neighbor_idx])

# MICE - need to add our target point to the dataset
x_with_target = np.append(x, target_x)
y_with_target = np.append(y, np.nan)
mask_with_target = np.append(mask, True)
data_mice = np.column_stack([x_with_target, y_with_target])
data_mice[mask_with_target, 1] = np.nan
imputer = IterativeImputer(random_state=42)
data_imp = imputer.fit_transform(data_mice)
mice_imputed = data_imp[-1, 1]

# Determine common scale for all plots
all_y_values = np.concatenate([
    y_obs, [mean_imputed, reg_imputed, knn_imputed, mice_imputed, target_y_true]
])
y_min, y_max = all_y_values.min(), all_y_values.max()
y_margin = (y_max - y_min) * 0.1  # 10% margin
y_lims = [y_min - y_margin, y_max + y_margin]

# Create plots with same scale
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
methods = ['Mean', 'Regression', 'KNN (k=5)', 'MICE']

# Mean
axes[0].scatter(x_obs, y_obs, alpha=0.6, s=30)
axes[0].axhline(y=mean_imputed, color='red', linestyle='--', alpha=0.5)
axes[0].scatter(target_x, mean_imputed, color='red', marker='*', s=300)
axes[0].set_title('Mean: Use average')

# Regression
axes[1].scatter(x_obs, y_obs, alpha=0.6, s=30)
x_line = np.linspace(x.min()-10, x.max(), 50)  # Extend line to show 155
axes[1].plot(x_line, reg.predict(x_line.reshape(-1, 1)), 'b-', alpha=0.5)
axes[1].scatter(target_x, reg_imputed, color='red', marker='*', s=300)
axes[1].set_title('Regression: Use line')

# KNN
axes[2].scatter(x_obs, y_obs, alpha=0.6, s=30)
axes[2].scatter(x_obs[neighbor_idx], y_obs[neighbor_idx], color='orange', s=100, alpha=0.5)
axes[2].scatter(target_x, knn_imputed, color='red', marker='*', s=300)
axes[2].set_title('KNN: Use neighbors')

# MICE
axes[3].scatter(x_obs, y_obs, alpha=0.6, s=30)
axes[3].scatter(target_x, mice_imputed, color='red', marker='*', s=300)
axes[3].set_title('MICE: Iterative model')

# Apply same scale to all plots
for ax in axes:
    ax.scatter(target_x, target_y_true, color='green', marker='o', s=100, alpha=0.7, label='True')
    ax.set_xlabel('Height')
    ax.set_ylabel('Weight')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(150, 195)  # Same x-axis
    ax.set_ylim(y_lims)    # Same y-axis - THIS IS THE FIX

plt.suptitle(f'How different methods impute the same missing point (height={target_x} cm)', y=1.05)
plt.tight_layout()
plt.show()

# Print comparison of methods
print(f"🎯 Imputation results for height={target_x} cm:")
print(f"   └─ True value:  {target_y_true:.1f} kg")
print(f"   └─ Mean:        {mean_imputed:.1f} kg")
print(f"   └─ Regression:  {reg_imputed:.1f} kg")
print(f"   └─ KNN (k=5):   {knn_imputed:.1f} kg")
print(f"   └─ MICE:        {mice_imputed:.1f} kg")

## Practical Guidance and Common Pitfalls

### Data Leakage Prevention
- **Do not leak validation/test information.** Fit imputers on the training split only, then apply to validation/test.
- **Scale before distance-based methods.** KNN requires standardized features; consider feature selection.
- **Preserve uncertainty.** Deterministic imputations overstate certainty; stochastic regression/MICE better preserve variance.
- **Check units and bounds.** After imputation, clamp to physically plausible ranges (e.g., height 50–250 cm, weight 20–300 kg).
- **Stratify when reasonable.** If distributions differ by clinic or gender, impute within strata.

### Implementation Best Practices
```python
# CORRECT: Fit imputer on training data only
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)  # Use fitted parameters

# WRONG: Fit on entire dataset
# X_imputed = imputer.fit_transform(X_all)  # Leaks validation info!
```

---

## How to Evaluate Imputation Quality

1. **Masking experiment (recommended)**: Randomly hide a known subset (e.g., 15%) as if missing, impute, then compare to ground truth.  
   - Metrics: MAE, RMSE, $R^2$ on the masked entries.  

2. **Distributional checks**: Compare histograms/quantiles of imputed vs observed; large shifts suggest bias.  

3. **Correlation preservation**: Check that relationships (e.g., height vs weight) are not collapsed by imputation.  

4. **Downstream impact**: Train a simple model on imputed data and validate performance stability.

### Validation Strategy
```python
def evaluate_imputation(df, feature_cols, target_col, method, missing_rate=0.15,
                        random_state=42):
    """Masking experiment: hide known values, impute them, compare to the truth."""
    rng = np.random.default_rng(random_state)

    complete_data = df.dropna(subset=feature_cols + [target_col])
    n_missing = max(1, int(len(complete_data) * missing_rate))

    # Sample without replacement, otherwise the same row can be masked twice
    test_indices = rng.choice(complete_data.index, size=n_missing, replace=False)
    true_values = complete_data.loc[test_indices, target_col]

    train_data = complete_data.drop(index=test_indices)
    test_data = complete_data.loc[test_indices].copy()
    test_data[target_col] = np.nan

    # Fit on the training rows only, then transform the masked rows
    columns = feature_cols + [target_col]
    method.fit(train_data[columns].to_numpy(dtype=float))
    imputed = method.transform(test_data[columns].to_numpy(dtype=float))
    imputed_values = pd.Series(imputed[:, columns.index(target_col)], index=test_indices)

    return {
        'MAE': mean_absolute_error(true_values, imputed_values),
        'R2': r2_score(true_values, imputed_values),
        'n': int(n_missing),
    }
```

---

## Computational Considerations

### Parallelization and Speed
- **Mean/Median** and **Regression**: Trivially parallel across columns or groups; vectorizable in NumPy/pandas.
- **KNN**: Can parallelize neighbor searches (e.g., blockwise or with ANN libraries) and batch queries.
- **MICE**: Within a variable the update is sequential, but you can parallelize across multiple chains or use parallel model training inside each step.
- **General tip**: Batch operations and avoid Python loops where possible.

### Computational Complexity

| Method | Training Time | Prediction Time | Memory |
|--------|---------------|-----------------|---------|
| Mean/Median | $O(n)$ | $O(1)$ | $O(p)$ |
| Linear Regression | $O(np^2)$ | $O(p)$ | $O(p^2)$ |
| KNN | $O(n^2p)$ | $O(np)$ | $O(np)$ |
| MICE | $O(Tnp^2)$ | $O(Tp)$ | $O(np)$ |

where $n$ = records, $p$ = features, $T$ = iterations

---

## Method Selection Guidelines

### Quick Decision Tree

1. **Sample size < 100 records**
   → Mean/Median imputation (insufficient data for complex methods)

2. **Missingness > 50% in key variables**
   → Consider if imputation is appropriate; may need specialized techniques

3. **Strong linear relationships visible**
   → Linear regression imputation (with stochastic component)

4. **Nonlinear patterns, moderate size (100-10,000 records)**
   → KNN imputation (k = 3-7, properly scaled features)

5. **Many variables missing, want joint structure**
   → MICE (5-10 iterations, appropriate models per variable type)

6. **Production pipeline, speed critical**
   → Mean/Median or simple regression (fast, deterministic)

7. **Research context, need uncertainty quantification**
   → Multiple imputation with MICE

### Quick Chooser (Rules of Thumb)
- Small, mostly MCAR, simple relationships → **Mean/Median** baseline.  
- Clear linear signal with good predictors → **Regression** (stochastic if possible).  
- Nonlinear local patterns, moderate size → **KNN** (k around 3–7, scaled features).  
- Many variables missing, want joint structure → **MICE** (5–10 iters; consider multiple imputations).

---

## Before We Code: Final Prediction Exercise

- Which variables in your dataset are most predictable from others, and why?  
- If you increase k in KNN from 3 to 15, what happens to bias vs variance of imputations?  
- Under MAR, why does regression/MICE generally beat mean imputation?  
- How will you prove your choice works (what masking rate, which metrics, what sanity checks)?

In [ ]:
#@title Section 7: Classical Imputation Methods (Fixed LinearRegression Error)

class ClassicalImputers:
    """Collection of classical imputation methods for real missing data."""

    def __init__(self, random_state=42):
        self.random_state = random_state
        np.random.seed(random_state)

    def impute_all_methods(self, df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        """
        Apply all imputation methods to the real missing data.
        Imputes BOTH height and weight where missing.
        """
        # Identify what's actually missing
        missing_height = df['height_cm'].isna()
        missing_weight = df['weight_kg'].isna()
        missing_any = missing_height | missing_weight

        print(f"\n📊 Real Missing Data:")
        print(f"   Height missing: {missing_height.sum()} patients at indices {df.index[missing_height].tolist()}")
        print(f"   Weight missing: {missing_weight.sum()} patients at indices {df.index[missing_weight].tolist()}")
        print(f"   Any missing: {missing_any.sum()} patients")

        results = {}

        # 1. Mean Imputation
        mean_df = df.copy()
        height_mean = mean_df['height_cm'].mean()
        weight_mean = mean_df['weight_kg'].mean()
        mean_df['height_cm'] = mean_df['height_cm'].fillna(height_mean)
        mean_df['weight_kg'] = mean_df['weight_kg'].fillna(weight_mean)
        results['Mean'] = mean_df
        print(f"   Mean: height={height_mean:.1f}cm, weight={weight_mean:.1f}kg")

        # 2. Linear Regression (improved logic)
        reg_df = df.copy()

        # Get complete cases for training (both height and weight present)
        complete_mask = df['height_cm'].notna() & df['weight_kg'].notna()
        complete_data = df[complete_mask]

        if len(complete_data) >= 5:
            # Train height predictor from weight
            X_complete = complete_data[['weight_kg']].values
            y_height = complete_data['height_cm'].values

            height_model = LinearRegression()
            height_model.fit(X_complete, y_height)

            # Train weight predictor from height
            X_complete_h = complete_data[['height_cm']].values
            y_weight = complete_data['weight_kg'].values

            weight_model = LinearRegression()
            weight_model.fit(X_complete_h, y_weight)

            # Impute missing heights where weight exists
            height_missing_weight_exists = missing_height & ~missing_weight
            if height_missing_weight_exists.any():
                indices = df.index[height_missing_weight_exists]
                X_pred = df.loc[indices, ['weight_kg']].values
                reg_df.loc[indices, 'height_cm'] = height_model.predict(X_pred)
                print(f"   Regression: Imputed {len(indices)} heights using weight")

            # Impute missing weights where height exists
            weight_missing_height_exists = missing_weight & ~missing_height
            if weight_missing_height_exists.any():
                indices = df.index[weight_missing_height_exists]
                X_pred = df.loc[indices, ['height_cm']].values
                reg_df.loc[indices, 'weight_kg'] = weight_model.predict(X_pred)
                print(f"   Regression: Imputed {len(indices)} weights using height")

            # For cases where both are missing, use mean values
            both_missing = missing_height & missing_weight
            if both_missing.any():
                indices = df.index[both_missing]
                reg_df.loc[indices, 'height_cm'] = height_mean
                reg_df.loc[indices, 'weight_kg'] = weight_mean
                print(f"   Regression: Used mean for {len(indices)} cases with both missing")
        else:
            # Not enough complete data for regression, use mean
            reg_df['height_cm'] = reg_df['height_cm'].fillna(height_mean)
            reg_df['weight_kg'] = reg_df['weight_kg'].fillna(weight_mean)
            print(f"   Regression: Insufficient data, used mean imputation")

        results['Regression'] = reg_df

        # 3. KNN Imputation
        knn_df = df.copy()
        n_complete = complete_mask.sum()
        if n_complete > 1:
            try:
                imputer = KNNImputer(n_neighbors=min(5, n_complete))
                knn_df[['height_cm', 'weight_kg']] = imputer.fit_transform(knn_df[['height_cm', 'weight_kg']])
                print(f"   KNN: Used {imputer.n_neighbors} neighbors")
            except Exception as e:
                print(f"   KNN failed ({e}), using mean")
                knn_df = mean_df.copy()
        else:
            knn_df = mean_df.copy()
        results['KNN'] = knn_df

        # 4. MICE Imputation
        mice_df = df.copy()
        try:
            imputer = IterativeImputer(random_state=self.random_state, max_iter=10)
            mice_df[['height_cm', 'weight_kg']] = imputer.fit_transform(mice_df[['height_cm', 'weight_kg']])
            print(f"   MICE: Iterative imputation completed")
        except Exception as e:
            print(f"   MICE failed ({e}), using mean")
            mice_df = mean_df.copy()
        results['MICE'] = mice_df

        return results

    def visualize_imputation_comparison(self, df: pd.DataFrame):
        """
        Create visualization comparing all imputation methods.
        Shows where imputed values fall for BOTH height and weight.
        """
        # Get all imputation results
        all_results = self.impute_all_methods(df)

        # Identify patients with missing data
        missing_height_mask = df['height_cm'].isna()
        missing_weight_mask = df['weight_kg'].isna()
        missing_both_mask = missing_height_mask & missing_weight_mask

        missing_height = df.index[missing_height_mask].tolist()
        missing_weight = df.index[missing_weight_mask].tolist()
        missing_both = df.index[missing_both_mask].tolist()
        missing_height_only = [idx for idx in missing_height if idx not in missing_both]
        missing_weight_only = [idx for idx in missing_weight if idx not in missing_both]
        missing_any = list(set(missing_height + missing_weight))

        print(f"\n📊 Visualization Summary:")
        print(f"   Height only missing: {missing_height_only}")
        print(f"   Weight only missing: {missing_weight_only}")
        print(f"   Both missing: {missing_both}")

        # Create figure with 2x3 grid
        fig = plt.figure(figsize=(15, 10))

        methods = ['Mean', 'Regression', 'KNN', 'MICE']

        # Axis limits are derived from the data rather than hard-coded, so that
        # imputed points cannot fall outside the visible area when the dataset,
        # the sample size or the missingness pattern changes.
        all_w = pd.concat([all_results[m]['weight_kg'] for m in methods] + [df['weight_kg']]).dropna()
        all_h = pd.concat([all_results[m]['height_cm'] for m in methods] + [df['height_cm']]).dropna()
        w_pad = max(2.0, 0.05 * (all_w.max() - all_w.min()))
        h_pad = max(2.0, 0.05 * (all_h.max() - all_h.min()))
        x_lims = (all_w.min() - w_pad, all_w.max() + w_pad)
        y_lims = (all_h.min() - h_pad, all_h.max() + h_pad)

        for idx, method in enumerate(methods):
            ax = plt.subplot(2, 3, idx + 1)
            imputed_df = all_results[method]

            # Plot original complete data
            complete_mask = df['height_cm'].notna() & df['weight_kg'].notna()
            ax.scatter(df.loc[complete_mask, 'weight_kg'],
                      df.loc[complete_mask, 'height_cm'],
                      alpha=0.2, s=40, color='gray', label='Complete data')

            # Plot imputed values with different markers
            if missing_height_only:
                h_vals = imputed_df.loc[missing_height_only, 'height_cm'].values
                w_vals = imputed_df.loc[missing_height_only, 'weight_kg'].values
                ax.scatter(w_vals, h_vals, color='red', marker='^', s=100,
                          edgecolors='black', linewidth=0.5, alpha=0.8,
                          label=f'Height imputed ({len(missing_height_only)})')

            if missing_weight_only:
                h_vals = imputed_df.loc[missing_weight_only, 'height_cm'].values
                w_vals = imputed_df.loc[missing_weight_only, 'weight_kg'].values
                ax.scatter(w_vals, h_vals, color='green', marker='v', s=100,
                          edgecolors='black', linewidth=0.5, alpha=0.8,
                          label=f'Weight imputed ({len(missing_weight_only)})')

            if missing_both:
                h_vals = imputed_df.loc[missing_both, 'height_cm'].values
                w_vals = imputed_df.loc[missing_both, 'weight_kg'].values
                ax.scatter(w_vals, h_vals, color='purple', marker='s', s=120,
                          edgecolors='black', linewidth=0.5, alpha=0.8,
                          label=f'Both imputed ({len(missing_both)})')


            ax.set_xlabel('Weight (kg)')
            ax.set_ylabel('Height (cm)')
            ax.set_title(f'{method} Imputation')
            ax.legend(loc='lower right', fontsize=8)
            ax.grid(True, alpha=0.3)

            # Set consistent axis limits
            ax.set_xlim(*x_lims)
            ax.set_ylim(*y_lims)

        # 5th plot: All methods comparison
        ax = plt.subplot(2, 3, 5)

        # Plot complete data as background
        ax.scatter(df.loc[complete_mask, 'weight_kg'],
                  df.loc[complete_mask, 'height_cm'],
                  alpha=0.2, s=30, color='gray', label='Complete')

        # Plot imputed values from each method with different colors
        method_colors = {'Mean': '#3498db', 'Regression': '#e74c3c',
                        'KNN': '#2ecc71', 'MICE': '#9b59b6'}

        for method in methods:
            imputed_df = all_results[method]
            if missing_any:
                h_vals = imputed_df.loc[missing_any, 'height_cm'].values
                w_vals = imputed_df.loc[missing_any, 'weight_kg'].values
                ax.scatter(w_vals, h_vals, color=method_colors[method],
                          s=60, alpha=0.7, edgecolors='black', linewidth=0.5,
                          label=method)

        ax.set_xlabel('Weight (kg)')
        ax.set_ylabel('Height (cm)')
        ax.set_title('All Methods Comparison')
        ax.legend(loc='lower right', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(*x_lims)
        ax.set_ylim(*y_lims)

        # 6th plot: Summary statistics
        ax = plt.subplot(2, 3, 6)
        ax.axis('off')

        # Create comparison table for imputed values
        summary_data = []
        for method in methods:
            imputed_df = all_results[method]
            if missing_height:
                h_mean = imputed_df.loc[missing_height, 'height_cm'].mean()
                h_std = imputed_df.loc[missing_height, 'height_cm'].std()
            else:
                h_mean, h_std = 0, 0

            if missing_weight:
                w_mean = imputed_df.loc[missing_weight, 'weight_kg'].mean()
                w_std = imputed_df.loc[missing_weight, 'weight_kg'].std()
            else:
                w_mean, w_std = 0, 0

            summary_data.append([method, f'{h_mean:.1f}±{h_std:.1f}', f'{w_mean:.1f}±{w_std:.1f}'])

        # Create table
        table = ax.table(cellText=summary_data,
                        colLabels=['Method', 'Height (cm)', 'Weight (kg)'],
                        cellLoc='center',
                        loc='center',
                        colWidths=[0.3, 0.35, 0.35])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2)

        # Add title
        ax.text(0.5, 0.9, 'Imputed Values Summary',
               transform=ax.transAxes, ha='center', fontsize=12, fontweight='bold')


        plt.suptitle('Classical Imputation Methods - Real Missing Data',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('classical_imputation_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()

        return all_results


# Execute the analysis
print("\n" + "="*60)
print("CLASSICAL IMPUTATION METHODS ANALYSIS")
print("="*60)

# Check data
print(f"\nDataset: {len(standardized_df)} records")

# Initialize imputer and run visualization
imputer = ClassicalImputers()
all_imputed = imputer.visualize_imputation_comparison(standardized_df)

# Select best method (MICE typically performs well)
best_classical_df = all_imputed['MICE']

print("\n✅ Classical imputation complete!")
print(f"   Best method selected: MICE")
print(f"   Final dataset: {len(best_classical_df)} records with no missing values")

# Verify no missing values remain
remaining_missing = best_classical_df[['height_cm', 'weight_kg']].isna().sum()
print(f"\n📊 Verification:")
print(f"   Height missing: {remaining_missing['height_cm']}")
print(f"   Weight missing: {remaining_missing['weight_kg']}")

print("="*60)

# LLM-based Imputation: Context-Aware Missing Data Recovery

fill-missing-LLM.svg

## Why This Matters

**Real-world problem**: Traditional statistical methods treat missing data as a purely mathematical problem, ignoring the rich contextual information that human experts use when making clinical decisions. A cardiologist doesn't just look at isolated numbers—they consider patient age, medical history, lifestyle, and population norms when assessing whether a measurement makes sense.

**Traditional limitations**: Classical imputation methods are context-blind. They might impute the same height for a 25-year-old athlete and a 70-year-old patient if they have the same weight, ignoring that muscle mass, bone density, and posture change dramatically with age and activity level.

**Key insight**: Large Language Models have been trained on vast medical literature and can apply domain knowledge, population statistics, and contextual reasoning that classical methods miss. They can understand that "basketball player" implies different height distributions than "elderly patient with osteoporosis."

**Learning objective**: You will learn to design prompts that leverage LLM domain knowledge for intelligent imputation, validate LLM reasoning against medical constraints, and understand when the additional cost and complexity of LLM imputation provides meaningful improvements over classical methods.

## Conceptual Foundation

**The power of context in medical imputation**:

Traditional methods see missing data as:
$$\text{Missing value} = f(\text{other numeric variables})$$

LLMs can incorporate:
$$\text{Missing value} = f(\text{all variables}, \text{domain knowledge}, \text{population norms}, \text{clinical reasoning})$$

**LLM imputation capabilities**:

1. **Semantic understanding**: LLMs can parse free-text fields like "former basketball player" or "wheelchair user" and understand their implications for physical measurements.

2. **Population knowledge**: Training on medical literature gives LLMs knowledge of normal ranges, correlations, and distributions across different demographic groups.

3. **Multi-modal reasoning**: LLMs can simultaneously consider numerical data, categorical variables, and text descriptions to make more informed imputations.

4. **Uncertainty awareness**: Unlike classical methods that provide point estimates, LLMs can express confidence levels and identify when insufficient context exists for reliable imputation.

**Mathematical framework for LLM imputation**:

Let $X_{miss}$ be the missing values, $X_{obs}$ be observed values, and $C$ be contextual information (demographics, medical history, free text). LLM imputation can be formalized as:

$$P(X_{miss} | X_{obs}, C, K)$$

where $K$ represents the domain knowledge encoded in the LLM's parameters.

**Comparison with classical approaches**:

| Aspect | Classical Methods | LLM Methods |
|--------|------------------|-------------|
| **Context awareness** | $P(X_{miss} \| X_{obs})$ | $P(X_{miss} \| X_{obs}, C, K)$ |
| **Domain knowledge** | None (purely statistical) | Extensive (from training data) |
| **Reasoning transparency** | Mathematical formula | Natural language explanation |
| **Computational cost** | $O(n)$ to $O(n^2)$ | $O(\text{tokens} \times \text{model size})$ |
| **Scalability** | Excellent | Limited by API costs |

## LLM Imputation Strategies

### 1. Context-Aware Imputation

**Approach**: Provide LLMs with comprehensive patient context rather than just numerical values.

**Example prompt structure**:
```
Patient: 45-year-old male, former athlete, now office worker
Current measurements: Weight 85kg, BMI indicates height ~180cm
Missing: Exact height measurement
Medical context: Recent knee surgery, reduced activity level
Population context: Scandinavian ancestry

Estimate missing height considering:
- Age-related height loss (vertebral compression)
- Athletic background (potentially taller frame)
- Current lifestyle changes
- Genetic background
```

**Mathematical representation**:
$$\hat{h} = \text{LLM}(\text{weight}, \text{BMI}, \text{age}, \text{ethnicity}, \text{activity history}, \text{medical status})$$

### 2. Domain Knowledge Integration

**Medical knowledge LLMs can apply**:

- **Growth patterns**: Height peaks in the early twenties and stays roughly stable until about age 40, after which it declines gradually
- **Population genetics**: Average height variations by ethnicity and geography  
- **Medical conditions**: Impact of osteoporosis, scoliosis, joint replacements
- **Lifestyle factors**: Nutrition, exercise, occupational effects

**Quantitative knowledge encoding**:

LLMs have learned approximate distributions like:
- Adult male height: $\mathcal{N}\!\left(175\,\text{cm},\ (7\,\text{cm})^2\right)$, i.e. a standard deviation of about 7 cm (varies by population)
- Age-related height loss: roughly 1 cm per decade after age 40, accelerating in later life (this is the rule the data generator in Section 3 applies)
- Athletic populations: Often $+1$ standard deviation above population mean

### 3. Uncertainty Quantification

**Confidence-based imputation**:

Unlike classical methods that provide point estimates, LLMs can express uncertainty:

$$\text{LLM output} = \{\hat{x}, \text{confidence}, \text{reasoning}, \text{alternatives}\}$$

**Example output structure**:
```json
{
  "imputed_height": 178.5,
  "confidence": "medium",
  "reasoning": "Based on weight and athletic history, but age suggests some height loss",
  "range": [176, 181],
  "alternative_scenarios": {
    "if_current_athlete": 180,
    "if_significant_aging": 176
  }
}
```

**Confidence calibration**: LLMs can learn to express appropriate uncertainty:
- **High confidence**: Strong correlations, typical patterns, abundant context
- **Medium confidence**: Some ambiguity, limited context, edge cases
- **Low confidence**: Contradictory information, rare patterns, insufficient data

### 4. Explanatory Imputation

**Interpretable reasoning**: LLMs provide natural language explanations for their imputations, enabling clinical validation.

**Example reasoning chains**:
1. "Patient weight (85kg) suggests height around 175-185cm based on healthy BMI ranges"
2. "Athletic background suggests potentially above-average height for population"
3. "Age (45) indicates possible 1-2cm height loss from peak"
4. "Final estimate: 178cm ± 3cm, accounting for uncertainty in activity level"

**Clinical value**: Healthcare providers can evaluate whether LLM reasoning aligns with their medical judgment.

## Advanced LLM Imputation Techniques

### 1. Multi-Step Reasoning

**Chain-of-thought imputation**:
```
Step 1: Analyze available measurements
Step 2: Apply population norms for demographic group  
Step 3: Adjust for individual factors (age, medical history)
Step 4: Consider measurement context and quality
Step 5: Provide final estimate with uncertainty bounds
```

**Mathematical formulation**:
$$\hat{x}_{final} = f_5(f_4(f_3(f_2(f_1(X_{obs}, C)))))$$

where each $f_i$ represents a reasoning step.

### 2. Ensemble LLM Imputation

**Multiple model approach**:
- Query multiple LLMs (GPT, Claude, Gemini) for same missing value
- Aggregate responses using weighted voting or statistical combination
- Use disagreement between models as uncertainty measure

**Aggregation strategy**:
$$\hat{x}_{ensemble} = \sum_{i=1}^N w_i \hat{x}_i$$

where weights $w_i$ can be based on model confidence, past performance, or equal weighting.

### 3. Iterative Refinement

**Progressive imputation**:
1. Initial imputation using available context
2. Validation against medical constraints
3. Refinement based on consistency checks
4. Final validation with clinical rules

**Convergence criteria**:
$$|\hat{x}^{(t+1)} - \hat{x}^{(t)}| < \epsilon$$

## Strategy

**Progressive complexity approach**:

1. **Simple context**: Start with basic demographic information (age, gender, ethnicity)
2. **Rich context**: Add medical history, lifestyle factors, measurement circumstances  
3. **Multi-variable**: Simultaneously impute multiple missing values with cross-validation
4. **Quality control**: Implement systematic validation against medical knowledge

**Experimental design**:

1. **Controlled comparison**: Use same masking experiments as classical methods
2. **Context ablation**: Test imputation quality with different amounts of context
3. **Domain validation**: Verify imputations against medical literature ranges
4. **Cost-benefit analysis**: Quantify accuracy improvements vs computational costs

**Success metrics**:

- **Accuracy**: Lower MAE/RMSE on masked data compared to classical methods
- **Consistency**: Imputations respect medical constraints and relationships
- **Interpretability**: Clinical experts can validate reasoning
- **Reliability**: Consistent results across similar patients

## Implementation Considerations

### 1. Prompt Engineering for Medical Imputation

**Effective prompt structure**:

```python
def create_imputation_prompt(patient_data, missing_variables, context):
    prompt = f"""
    You are a medical data scientist with expertise in anthropometric measurements.
    
    PATIENT CONTEXT:
    {context}
    
    AVAILABLE DATA:
    {patient_data}
    
    MISSING VALUES TO IMPUTE:
    {missing_variables}
    
    TASK:
    For each missing value, provide:
    1. Estimated value with units
    2. Confidence level (high/medium/low)  
    3. Reasoning based on medical knowledge
    4. Plausible range considering uncertainty
    5. Key factors that influenced your estimate
    
    CONSTRAINTS:
    - All values must be physiologically plausible
    - Consider population norms for demographic group
    - Account for age-related changes
    - Flag any contradictory or unusual patterns
    
    OUTPUT FORMAT: JSON with structured fields
    """
    return prompt
```

### 2. Validation Framework

**Multi-level validation**:

```python
def validate_llm_imputation(imputed_value, patient_context):
    validations = []
    
    # Physiological constraints
    if not (50 <= imputed_value <= 250):  # Height in cm
        validations.append("Outside physiological range")
    
    # Population norms
    expected_range = get_population_range(patient_context['age'],
                                        patient_context['gender'],
                                        patient_context['ethnicity'])
    if not (expected_range[0] <= imputed_value <= expected_range[1]):
        validations.append("Outside population norms")
    
    # Consistency with other measurements
    if inconsistent_with_weight(imputed_value, patient_context['weight']):
        validations.append("Inconsistent with weight")
    
    return validations
```

### 3. Cost-Performance Optimization

**Adaptive complexity**: Use expensive LLM imputation only when classical methods fail or uncertainty is high.

**Decision tree**:
```python
def select_imputation_method(missing_pattern, context_richness, accuracy_requirement):
    if missing_pattern == "simple" and context_richness == "low":
        return "classical_method"
    elif accuracy_requirement == "high" and context_richness == "rich":
        return "llm_imputation"  
    else:
        return "hybrid_approach"
```

**Cost model**:
$$\text{Total Cost} = \text{API Cost} + \text{Latency Cost} + \text{Validation Cost}$$

where each component can be optimized based on application requirements.

## Challenges and Limitations

### 1. Hallucination and Overconfidence

**Problem**: LLMs may generate plausible but incorrect imputations, especially for rare cases.

**Mitigation strategies**:
- Cross-validation with multiple models
- Conservative confidence reporting
- Systematic validation against medical literature
- Human expert review for critical applications

### 2. Context Dependence

**Problem**: LLM performance heavily depends on quality and completeness of contextual information.

**Solution approaches**:
- Structured context collection protocols
- Missing context handling strategies
- Graceful degradation to classical methods

### 3. Reproducibility Challenges

**Problem**: LLM outputs may vary between API calls even with same inputs.

**Control measures**:
- Set temperature to low values (0.1-0.3) for deterministic outputs
- Use multiple runs and aggregate results
- Version control for model and prompt changes
- Detailed logging of all parameters


## Before We Code: Prediction Exercise

- What should we expect to see?
- What challenges might arise with real medical data?
- How would you validate the results in practice?
- Specific hypotheses to test?


In [ ]:
#@title Section 8: LLM Imputation Method (Both Height and Weight)

def llm_imputation_complete(df: pd.DataFrame, model: Any) -> pd.DataFrame:
    """
    Use LLM for intelligent imputation of BOTH height and weight with explanations.

    Args:
        df: DataFrame with missing values
        model: Gemini model instance

    Returns:
        DataFrame with imputed values and explanations
    """

    if model is None:
        print("⚠️ No model available for LLM imputation")
        return df

    # Identify all missing patterns
    missing_height = df['height_cm'].isna()
    missing_weight = df['weight_kg'].isna()
    missing_any = missing_height | missing_weight

    if not missing_any.any():
        print("✅ No missing values to impute")
        return df

    # Get records with any missing values
    missing_records = df[missing_any].copy()

    # Add flags to indicate what's missing for each record
    missing_records['missing_height'] = missing_records['height_cm'].isna()
    missing_records['missing_weight'] = missing_records['weight_kg'].isna()

    print(f"\n🤖 Using LLM to impute missing values...")
    print(f"   Height missing: {missing_height.sum()} patients")
    print(f"   Weight missing: {missing_weight.sum()} patients")
    print(f"   Total patients with missing data: {missing_any.sum()}")

    # Create comprehensive imputation prompt

    prompt = f"""
You are a medical data scientist specializing in anthropometric data imputation.

Given the following patient records with missing height and/or weight values, estimate the missing values based on:
1. Statistical relationships between height, weight, age, and gender
2. Population norms for the demographic groups
3. Any contextual information available (clinic, date, comments, etc)

IMPORTANT RELATIONSHIPS TO USE:
- For adults: height and weight are correlated (taller people tend to weigh more)
- Gender differences: Males typically taller/heavier than females
- Age effects: Elderly may have height loss, weight varies with lifestyle
- Use BMI range 20-25 as typical for healthy adults when one measurement is missing
- If both weight and height are missing, use gender and age-based population norms
- Consider any available context (comments, clinic patterns, etc.)

PATIENT RECORDS WITH MISSING DATA:
{missing_records.to_json(orient='records', indent=2)}

For each patient, impute the missing values (height_cm and/or weight_kg as needed).

Return a JSON array with this EXACT structure for each patient:
[
  {{
    "patient_id": "string",
    "imputed_height_cm": number or null (null if height was not missing),
    "imputed_weight_kg": number or null (null if weight was not missing),
    "height_reasoning": "explanation" or null,
    "weight_reasoning": "explanation" or null,
    "confidence": "high|medium|low",
    "method": "bmi_based|population_norm|regression|contextual"
  }}
]

"""

    try:
        # Configure generation for structured output
        generation_config = {
            "temperature": 0.0,  # Low temperature for consistent medical estimates
            "max_output_tokens": 4096,
            "response_mime_type": "application/json"
        }

        # Generate imputation
        response = model.generate_content(prompt, generation_config=generation_config)

        # Parse response
        try:
            imputations = json.loads(response.text)
        except json.JSONDecodeError:
            # Try to extract JSON from response if it's wrapped in text
            import re
            json_match = re.search(r'\[.*\]', response.text, re.DOTALL)
            if json_match:
                imputations = json.loads(json_match.group())
            else:
                raise ValueError("Could not parse LLM response as JSON")

        # Apply imputations to dataframe
        df_imputed = df.copy()

        # Track what was imputed
        height_imputed_count = 0
        weight_imputed_count = 0

        print("\n📝 LLM Imputation Results:")
        print("-" * 60)

        for imp in imputations:
            patient_id = str(imp['patient_id'])

            # Find the patient in the dataframe
            if 'patient_id' in df_imputed.columns:
                mask = df_imputed['patient_id'].astype(str) == patient_id
            else:
                # Use index if no patient_id column
                try:
                    idx = int(patient_id)
                    mask = df_imputed.index == idx
                except (ValueError, TypeError):
                    continue  # Identifier is not usable as a positional index

            if not mask.any():
                continue

            # Apply height imputation if provided
            if imp.get('imputed_height_cm') is not None and df_imputed.loc[mask, 'height_cm'].isna().any():
                height_val = float(imp['imputed_height_cm'])
                # Sanity check
                if 140 <= height_val <= 210:
                    df_imputed.loc[mask, 'height_cm'] = height_val
                    height_imputed_count += 1

                    # Add metadata
                    if 'llm_height_reasoning' not in df_imputed.columns:
                        df_imputed['llm_height_reasoning'] = None
                    df_imputed.loc[mask, 'llm_height_reasoning'] = imp.get('height_reasoning', 'No reasoning provided')

                    print(f"\nPatient {patient_id} - Height:")
                    print(f"  Imputed: {height_val:.1f} cm")
                    print(f"  Reasoning: {imp.get('height_reasoning', 'N/A')}")

            # Apply weight imputation if provided
            if imp.get('imputed_weight_kg') is not None and df_imputed.loc[mask, 'weight_kg'].isna().any():
                weight_val = float(imp['imputed_weight_kg'])
                # Sanity check
                if 30 <= weight_val <= 200:
                    df_imputed.loc[mask, 'weight_kg'] = weight_val
                    weight_imputed_count += 1

                    # Add metadata
                    if 'llm_weight_reasoning' not in df_imputed.columns:
                        df_imputed['llm_weight_reasoning'] = None
                    df_imputed.loc[mask, 'llm_weight_reasoning'] = imp.get('weight_reasoning', 'No reasoning provided')

                    print(f"\nPatient {patient_id} - Weight:")
                    print(f"  Imputed: {weight_val:.1f} kg")
                    print(f"  Reasoning: {imp.get('weight_reasoning', 'N/A')}")

            # Add general metadata
            if 'llm_confidence' not in df_imputed.columns:
                df_imputed['llm_confidence'] = None
            if 'llm_method' not in df_imputed.columns:
                df_imputed['llm_method'] = None

            df_imputed.loc[mask, 'llm_confidence'] = imp.get('confidence', 'medium')
            df_imputed.loc[mask, 'llm_method'] = imp.get('method', 'unknown')

            # Calculate BMI if both values now exist
            if mask.any():
                row_idx = df_imputed.index[mask][0]
                h = df_imputed.loc[row_idx, 'height_cm']
                w = df_imputed.loc[row_idx, 'weight_kg']
                if pd.notna(h) and pd.notna(w):
                    bmi = w / ((h/100) ** 2)
                    print(f"  Resulting BMI: {bmi:.1f}")

        print("\n" + "="*60)
        print(f"✅ LLM imputation complete:")
        print(f"   Heights imputed: {height_imputed_count}")
        print(f"   Weights imputed: {weight_imputed_count}")

        # Show confidence distribution
        if 'llm_confidence' in df_imputed.columns:
            conf_dist = df_imputed.loc[missing_any, 'llm_confidence'].value_counts()
            if not conf_dist.empty:
                print("\n📊 Confidence levels:")
                for conf, count in conf_dist.items():
                    if pd.notna(conf):
                        print(f"   {conf}: {count} patients")

        # Show methods used
        if 'llm_method' in df_imputed.columns:
            method_dist = df_imputed.loc[missing_any, 'llm_method'].value_counts()
            if not method_dist.empty:
                print("\n🔧 Methods used:")
                for method, count in method_dist.items():
                    if pd.notna(method):
                        print(f"   {method}: {count} patients")

        return df_imputed

    except Exception as e:
        print(f"❌ LLM imputation failed: {e}")
        print("   Falling back to mean imputation...")

        # Fallback to simple mean imputation
        df_fallback = df.copy()
        height_mean = df['height_cm'].mean()
        weight_mean = df['weight_kg'].mean()

        df_fallback['height_cm'] = df_fallback['height_cm'].fillna(height_mean)
        df_fallback['weight_kg'] = df_fallback['weight_kg'].fillna(weight_mean)

        return df_fallback


def visualize_llm_imputation(original_df: pd.DataFrame, llm_df: pd.DataFrame):
    """
    Visualize LLM imputation results showing reasoning and confidence.
    """
    # Identify what was imputed
    originally_missing_height = original_df['height_cm'].isna()
    originally_missing_weight = original_df['weight_kg'].isna()
    originally_missing_any = originally_missing_height | originally_missing_weight

    if not originally_missing_any.any():
        print("No missing values to visualize")
        return

    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Plot 1: Scatter plot with imputed values highlighted
    ax = axes[0]

    # Original complete data
    complete_mask = ~originally_missing_any
    ax.scatter(original_df.loc[complete_mask, 'weight_kg'],
              original_df.loc[complete_mask, 'height_cm'],
              alpha=0.2, s=40, color='gray', label='Original complete')

    # LLM imputed values with confidence coloring
    if 'llm_confidence' in llm_df.columns:
        confidence_colors = {'high': 'green', 'medium': 'orange', 'low': 'red'}

        for conf in ['high', 'medium', 'low']:
            conf_mask = originally_missing_any & (llm_df['llm_confidence'] == conf)
            if conf_mask.any():
                ax.scatter(llm_df.loc[conf_mask, 'weight_kg'],
                          llm_df.loc[conf_mask, 'height_cm'],
                          alpha=0.8, s=100, color=confidence_colors.get(conf, 'gray'),
                          edgecolors='black', linewidth=0.5,
                          label=f'LLM imputed ({conf} conf)')

    else:
        # No confidence info, just plot imputed values
        ax.scatter(llm_df.loc[originally_missing_any, 'weight_kg'],
                  llm_df.loc[originally_missing_any, 'height_cm'],
                  alpha=0.8, s=100, color='red',
                  edgecolors='black', linewidth=0.5,
                  label='LLM imputed')

    ax.set_xlabel('Weight (kg)')
    ax.set_ylabel('Height (cm)')
    ax.set_title('LLM Imputation Results')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)

    # Plot 2: BMI distribution
    ax = axes[1]

    # Calculate BMI for all records
    bmi_original = original_df.loc[complete_mask, 'weight_kg'] / ((original_df.loc[complete_mask, 'height_cm']/100)**2)
    bmi_imputed = llm_df.loc[originally_missing_any, 'weight_kg'] / ((llm_df.loc[originally_missing_any, 'height_cm']/100)**2)

    ax.hist(bmi_original, bins=15, alpha=0.5, color='blue', label='Original', density=True)
    ax.hist(bmi_imputed, bins=15, alpha=0.5, color='red', label='LLM Imputed', density=True)

    # Add reference lines for BMI categories
    ax.axvline(18.5, color='orange', linestyle='--', alpha=0.5, label='Underweight boundary')
    ax.axvline(25, color='orange', linestyle='--', alpha=0.5, label='Overweight boundary')

    ax.set_xlabel('BMI')
    ax.set_ylabel('Density')
    ax.set_title('BMI Distribution Comparison')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

    # Plot 3: Summary statistics
    ax = axes[2]
    ax.axis('off')

    # Prepare summary text
    summary = "LLM Imputation Summary\n" + "="*30 + "\n\n"

    # Count by method
    if 'llm_method' in llm_df.columns:
        method_counts = llm_df.loc[originally_missing_any, 'llm_method'].value_counts()
        summary += "Methods Used:\n"
        for method, count in method_counts.items():
            if pd.notna(method):
                summary += f"  • {method}: {count}\n"
        summary += "\n"

    # Statistics
    imputed_heights = llm_df.loc[originally_missing_height, 'height_cm']
    imputed_weights = llm_df.loc[originally_missing_weight, 'weight_kg']

    if len(imputed_heights) > 0:
        summary += f"Heights Imputed: {len(imputed_heights)}\n"
        summary += f"  Mean: {imputed_heights.mean():.1f} cm\n"
        summary += f"  Std: {imputed_heights.std():.1f} cm\n\n"

    if len(imputed_weights) > 0:
        summary += f"Weights Imputed: {len(imputed_weights)}\n"
        summary += f"  Mean: {imputed_weights.mean():.1f} kg\n"
        summary += f"  Std: {imputed_weights.std():.1f} kg\n\n"

    summary += f"BMI Statistics (imputed):\n"
    summary += f"  Mean: {bmi_imputed.mean():.1f}\n"
    summary += f"  Std: {bmi_imputed.std():.1f}\n"
    summary += f"  Range: {bmi_imputed.min():.1f} - {bmi_imputed.max():.1f}"

    ax.text(0.1, 0.5, summary, transform=ax.transAxes,
           fontsize=10, family='monospace', verticalalignment='center',
           bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

    plt.suptitle('LLM-Based Imputation Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('llm_imputation_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()


# Execute LLM imputation
print("\n" + "="*60)
print("LLM-BASED IMPUTATION")
print("="*60)

# Check if model is available
if 'model' in locals() and model is not None:
    print("✅ Model available, starting LLM imputation...")

    # Use the standardized dataframe
    data_for_llm = standardized_df if 'standardized_df' in locals() else df

    # Perform LLM imputation
    llm_imputed_df = llm_imputation_complete(data_for_llm, model)

    # Visualize results
    print("\n📊 Creating visualization...")
    visualize_llm_imputation(data_for_llm, llm_imputed_df)

    # Verify completeness
    remaining_missing = llm_imputed_df[['height_cm', 'weight_kg']].isna().sum()
    print("\n✅ Final verification:")
    print(f"   Height missing: {remaining_missing['height_cm']}")
    print(f"   Weight missing: {remaining_missing['weight_kg']}")

else:
    print("⚠️ No model available for LLM imputation")
    print("   Please initialize a model first (Section 2)")
    llm_imputed_df = standardized_df if 'standardized_df' in locals() else df

print("="*60)

# Biomechanical Data Imputation: LLM vs Classical Methods

This section assembles the pipeline from Sections 0 to 8 and measures it. A masking
experiment compares LLM imputation against the classical baselines on exactly the same
held-out values, using the metrics and checks listed below.

> These results characterise the pipeline on a small synthetic dataset. They are a
> methodological demonstration, not a clinical validation, and they do not by themselves
> establish fitness for clinical use.

### What We Will Implement
1. Context-rich prompting with participant demographics and activity history
2. Structured JSON output with biomechanical reasoning
3. Validation against physiological constraints (plausible height and weight ranges, and a BMI sanity check)
4. Performance comparison with classical methods

### Expected Results
- Higher accuracy for participants with rich contextual information
- Physiologically plausible imputations with explanations
- Better handling of athletes vs sedentary populations
- Preserved biomechanical relationships in the data

### Implementation Strategy

Let us design context-aware prompts, here is an example to start with
```python
def create_biomechanical_imputation_prompt(participant_data, missing_variables, context):
    prompt = f"""
    You are a biomechanics researcher with expertise in anthropometric measurements.

    PARTICIPANT CONTEXT:
    {context}

    AVAILABLE DATA:
    {participant_data}

    MISSING VALUES TO IMPUTE:
    {missing_variables}

    CONSTRAINTS:
    - All values must be physiologically plausible
    - Consider population norms for age/gender/activity level
    - Account for body composition differences between athletes and non-athletes
    - Preserve biomechanical relationships (height-weight, limb proportions)

    OUTPUT: JSON with estimated values, confidence levels, and reasoning
    """
    return prompt
```

#### Validation Framework
The notebook applies three nested levels of checking, and they use different bounds on purpose:

- **Section 6 (standardization)**: rejects a height outside 50–250 cm or a weight outside 20–300 kg. These are absolute plausibility limits, wide enough to catch unit errors without discarding unusual but real patients.
- **Section 8 (LLM imputation)**: accepts an imputed height only in 140–210 cm and an imputed weight only in 30–200 kg. Imputed values are guesses, so they are held to the tighter adult range.
- **Section 9 (evaluation)**: accepts an LLM weight only when the resulting BMI falls in 15–40, and otherwise falls back to the regression estimate.

- **BMI relationships**: the height–weight relationship should survive imputation rather than collapse toward the mean
- **Activity-specific norms**: different expectations for athletes and the general population (context the LLM can use and the classical methods cannot)

---

## Method Comparison and Validation

### Why This Matters for Research
Biomechanics research requires high data quality for valid conclusions about observed patterns and performance. Missing data can bias results, especially when missingness relates to participant characteristics (e.g., equipment failures more common with larger participants).

### What We Will Evaluate
- Systematic comparison across all imputation methods
- Performance metrics on masked validation data
- Method selection based on biomechanical validity
- Documentation for research reproducibility

#### Performance Metrics
- **Accuracy**: MAE, RMSE on masked validation data
- **Biomechanical validity**: Preserved relationships between anthropometric variables
- **Population consistency**: Results match known demographic patterns
- **Measurement reliability**: Stable results across different missing patterns

#### How high can $R^2$ actually go on this data?

Worth settling before reading the table, because the honest ceiling is much lower than
people expect.

The generator in Section 3 draws a BMI category for each patient **independently of
height** (gender and age shift it modestly), then sets weight $= \text{BMI} \times
(\text{height}/100)^2$. Height therefore explains only part of the variation in weight:
the measured correlation is about $r = 0.58$, so $r^2 \approx 0.33$. A model that sees
only height cannot do much better than explain a third of the variance, and five-fold
cross-validation on the full dataset lands near $R^2 \approx 0.29$. That is the target.
An $R^2$ near 0.9 on this task would mean something has leaked, not that the method is
excellent.

This is not an artefact of synthetic data. Adult height and weight correlate at roughly
0.4–0.5 in real populations too, for the same reason: two people of the same height can
sit anywhere in a wide band of body composition.

**What a negative $R^2$ means.** $R^2 = 1 - \text{SSE}/\text{SST}$, where SST uses the mean
of the *held-out* values. Negative simply means the prediction did worse than that mean,
and that mean is a number nobody has at prediction time. It is a comparison against an
oracle baseline, not a sign of a broken pipeline.

**Why the held-out set is not tiny.** With only 6 masked records the standard deviation of
$R^2$ on this data is about 0.33, so roughly one run in five comes out negative purely by
chance while MAE barely moves. Section 9 therefore masks 15 records and prints a stability
check showing the spread across repeated draws. Read any single split against that spread.

**Two different uncertainties, and they answer different questions.** Section 9 prints both,
and at first glance they look contradictory:

- The **marginal** 95% interval on each $R^2$ is enormous. The LLM lands at $R^2 = +0.16$
  with an interval of roughly $[-0.41, +0.56]$, and every method's interval overlaps every
  other one. That answers: *how much does the $R^2$ level depend on which patients happened
  to be drawn?*
- The **paired** comparison is far sharper: the LLM beats regression in about 96% of
  bootstrap resamples on $R^2$, and in 100% of them on MAE. That answers a different
  question: *scored on the same patients, how often does one method come out ahead?*

Both are correct, and the resolution is visible in the left panel. A patient who is hard for
one method is hard for all of them: the 103 kg patient is missed by roughly 30 kg by every
method. That shared difficulty dominates each marginal interval and cancels in the paired
comparison. This is why a paired design is worth the trouble. You get a much sharper
comparison out of the same fifteen records, provided you resample the *rows* and re-score
every method on the same resample rather than bootstrapping each method on its own.

**The two win rates disagree, and that is worth understanding.** $R^2$ is built on squared
error, so it is dominated by the handful of heavy patients that every method misses badly,
and the noise on those few rows swamps the difference between methods. MAE weights every
patient equally, so a method that improves several mid-range rows is credited for it. When
two reasonable metrics disagree about whether a gap is decisive, the honest report is both
numbers together with the reason, not whichever one flatters the conclusion.


#### Method Comparison Matrix

| Method | Accuracy on this dataset | Speed | Interpretability | Marginal cost |
|---|---|---|---|---|
| Mean / Median | Weakest: ignores all relationships | Instant | High | None |
| Regression / MICE | Strong baseline (MAE about 8.3 kg here) | Fast | High | None |
| KNN | About 1 kg worse; sensitive to scaling and to $k$ | Fast | Medium | None |
| LLM | Best here (MAE about 7.7 kg): reads notes the others cannot | Seconds per batch | Explains every estimate | Per-token API cost |

With 15 held-out weights, regression and MICE land near MAE 8.3 kg, KNN about 1 kg behind,
and the LLM ahead at roughly 7.7 kg. The gap has a mechanical explanation, and it is worth
understanding rather than celebrating: some clinical notes in this dataset quote the
measurement ("Längd 176cm, vikt 73kg", "Last recorded weight approximately 69 kg"). A
method that reads text recovers those values almost exactly; a method that sees only the
numeric columns cannot use them at all. Where no note carries weight information, the LLM
falls back to the same height-based reasoning as regression and the methods tie.

An earlier version of this dataset drew the comments **uniformly at random**, independent
of the record. Notes such as "Unable to obtain accurate height" appeared on complete
records, and a note quoting "vikt 70kg" sat on a patient who weighs 83 kg. With context
that is noise or outright disinformation, no context-reading method can win, and indeed
the LLM only tied regression then. The lesson generalises: before concluding that context
does not help, check whether your data contains any usable context at all.

A second trap surfaced on the way to this table: an earlier evaluation masked only six
values, where R\u00b2 has a standard deviation of about 0.33, so methods could be ordered
any way you like by choosing a seed. The stability check below the table exists to keep
that from happening again.

#### Essential Checks
- Physiological constraints satisfied at every level listed above
- Height–weight correlation after imputation stays close to the correlation measured on the complete records
- No systematic bias by participant group (compare errors per clinic and per gender)
- Imputed values reported at the precision the measuring equipment actually delivers, not more
- Population norms respected for age, gender and activity level
- MAE and $R^2$ reported for every method on the same held-out values, together with the number of masked records


### When to Use Each Method

#### Classical Methods
- **Small datasets** (< 100 participants)
- **Simple missing patterns** (few variables, low missingness)
- **Time-sensitive analysis**
- **Limited computational resources**

#### LLM Methods
- **Rich contextual information** available (activity history, notes)
- **Complex participant populations** (mixed athletes/non-athletes)
- **High accuracy requirements**
- **Research publications** requiring detailed methodology

#### Hybrid Approach
- **Large datasets** with subset requiring special attention
- **Budget constraints** but quality requirements
- **Validation studies** comparing multiple approaches

In [ ]:
#@title Section 9: Final Comparison - Performance Evaluation

import re

def prepare_evaluation_data_weight_only(df: pd.DataFrame, n_mask: int = 15, random_state: int = 42,
                                        verbose: bool = True
                                       ) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray, List[str]]:
    """
    Mask ONLY weight_kg for n_mask rows taken as one per weight sextile.
    Works only on rows that had complete (height_cm, weight_kg).
    """
    complete_mask = df['height_cm'].notna() & df['weight_kg'].notna()
    complete_df = df[complete_mask].copy()
    if len(complete_df) == 0:
        raise ValueError("No complete rows with both height_cm and weight_kg present.")

    n_mask = min(max(3, n_mask), len(complete_df))
    rng = np.random.default_rng(random_state)

    # Stratified selection across weight distribution
    w = complete_df['weight_kg'].to_numpy()
    q_edges = np.quantile(w, np.linspace(0, 1, n_mask + 1))
    chosen = []
    for ql, qh in zip(q_edges[:-1], q_edges[1:]):
        if qh == q_edges[-1]:
            cand = complete_df[(complete_df['weight_kg'] >= ql) & (complete_df['weight_kg'] <= qh)].index
        else:
            cand = complete_df[(complete_df['weight_kg'] >= ql) & (complete_df['weight_kg'] < qh)].index
        if len(cand) > 0:
            chosen.append(rng.choice(cand))

    if len(chosen) < n_mask:
        remaining = list(set(complete_df.index) - set(chosen))
        chosen.extend(rng.choice(remaining, size=n_mask - len(chosen), replace=False).tolist())

    mask_indices = np.array(chosen[:n_mask])
    masked_df = complete_df.copy()
    masked_df.loc[mask_indices, 'weight_kg'] = np.nan

    mask_ids = complete_df.loc[mask_indices, 'patient_id'].astype(str).tolist() if 'patient_id' in complete_df.columns else []

    if verbose:
        print("📊 Evaluation data prepared:")
        print(f"   Complete records used: {len(complete_df)}")
        print(f"   Patients masked: {n_mask} (weight only, stratified across weight sextiles)")

    return complete_df, masked_df, mask_indices, mask_ids

def llm_weight_imputation_contextual(masked_df: pd.DataFrame, train_df: pd.DataFrame,
                                     test_indices: np.ndarray, model: Any) -> pd.DataFrame:
    """
    Context-rich LLM imputation for weight prediction.
    Uses ALL available fields to make informed predictions.
    """
    if model is None:
        return masked_df

    # Calculate comprehensive training statistics
    train_stats = {
        'n_patients': len(train_df),
        'weight': {
            'mean': float(train_df['weight_kg'].mean()),
            'std': float(train_df['weight_kg'].std()),
            'min': float(train_df['weight_kg'].min()),
            'max': float(train_df['weight_kg'].max())
        },
        'height_weight_corr': float(train_df[['height_cm', 'weight_kg']].corr().iloc[0, 1])
    }

    # Linear model for baseline
    from sklearn.linear_model import LinearRegression
    reg = LinearRegression()
    reg.fit(train_df[['height_cm']].values, train_df['weight_kg'].values)

    # BMI statistics
    train_bmi = train_df['weight_kg'] / ((train_df['height_cm']/100)**2)
    bmi_stats = {
        'mean': float(train_bmi.mean()),
        'std': float(train_bmi.std()),
        'q25': float(train_bmi.quantile(0.25)),
        'q75': float(train_bmi.quantile(0.75))
    }

    # Gender-specific statistics if available
    gender_stats = {}
    if 'gender' in train_df.columns:
        for gender in train_df['gender'].dropna().unique():
            gender_data = train_df[train_df['gender'] == gender]
            if len(gender_data) >= 3:
                gender_stats[str(gender)] = {
                    'weight_mean': float(gender_data['weight_kg'].mean()),
                    'weight_std': float(gender_data['weight_kg'].std()),
                    'bmi_mean': float((gender_data['weight_kg'] / ((gender_data['height_cm']/100)**2)).mean())
                }

    # Clinic-specific patterns if available
    clinic_stats = {}
    if 'clinic' in train_df.columns:
        for clinic in train_df['clinic'].dropna().unique():
            clinic_data = train_df[train_df['clinic'] == clinic]
            if len(clinic_data) >= 3:
                clinic_stats[str(clinic)] = {
                    'weight_mean': float(clinic_data['weight_kg'].mean()),
                    'bmi_mean': float((clinic_data['weight_kg'] / ((clinic_data['height_cm']/100)**2)).mean())
                }

    # Find similar patients in training data for each test patient
    test_records = []
    for idx in test_indices:
        record = {
            'index': int(idx),
            'height_cm': float(masked_df.loc[idx, 'height_cm'])
        }

        # Add ALL available context fields
        context_fields = ['gender', 'clinic', 'age', 'date_measured', 'comments_english',
                         'llm_quality_notes', 'original_units']

        for field in context_fields:
            if field in masked_df.columns:
                val = masked_df.loc[idx, field]
                if pd.notna(val):
                    record[field] = str(val) if not isinstance(val, (int, float)) else float(val)

        # Calculate baseline from regression
        h = record['height_cm']
        record['regression_baseline'] = float(reg.predict([[h]])[0])

        # Find similar patients in training data
        similar_patients = []
        h_tolerance = 5  # cm
        similar_mask = (train_df['height_cm'] >= h - h_tolerance) & (train_df['height_cm'] <= h + h_tolerance)

        # Further filter by gender if available
        if 'gender' in record and 'gender' in train_df.columns:
            gender_mask = train_df['gender'] == record['gender']
            similar_mask = similar_mask & gender_mask

        similar = train_df[similar_mask]
        if len(similar) > 0:
            similar_patients = {
                'count': len(similar),
                'weight_mean': float(similar['weight_kg'].mean()),
                'weight_std': float(similar['weight_kg'].std())
            }

        record['similar_patients'] = similar_patients

        # Reasonable weight bounds based on BMI
        h_m = h / 100
        record['weight_bounds'] = {
            'min': float(17 * h_m**2),  # BMI 17 (underweight)
            'max': float(32 * h_m**2)   # BMI 32 (obese)
        }

        test_records.append(record)

    # Create comprehensive prompt
    prompt = f"""You are a medical data scientist predicting patient weights using comprehensive context.

TRAINING DATA INSIGHTS ({train_stats['n_patients']} patients):
- Weight distribution: {train_stats['weight']['mean']:.1f}±{train_stats['weight']['std']:.1f} kg
- Height-Weight correlation: {train_stats['height_weight_corr']:.3f}
- BMI range: Q25={bmi_stats['q25']:.1f}, Q75={bmi_stats['q75']:.1f}
- Linear model: weight = {reg.coef_[0]:.3f} * height + {reg.intercept_:.1f}

GENDER PATTERNS:
{json.dumps(gender_stats, indent=2) if gender_stats else "No gender-specific data available"}

CLINIC PATTERNS:
{json.dumps(clinic_stats, indent=2) if clinic_stats else "No clinic-specific data available"}

PATIENTS REQUIRING WEIGHT PREDICTION:
{json.dumps(test_records, indent=2)}

INSTRUCTIONS:
For each patient, predict weight considering:
1. Start with regression_baseline as initial estimate
2. Adjust based on gender (males typically heavier than females at same height)
3. Consider clinic patterns if relevant
4. Use similar_patients data when available
5. Parse any contextual clues from comments_english if present
6. Consider date/season effects if relevant (winter vs summer weight)
7. Ensure final weight gives reasonable BMI (typically 19-27 for healthy adults)
8. Stay within weight_bounds

Be precise and use all available context. Consider that:
- Males typically have BMI 22-26
- Females typically have BMI 20-24
- Age effects: elderly may have lower muscle mass
- Clinical context matters (e.g., diabetes clinic may have higher weights)

Return ONLY a JSON array:
[
  {{"index": <int>, "weight_kg": <number>, "reasoning": "brief explanation"}}, ...
]"""

    try:
        generation_config = {
            "temperature": 0.2,  # Slightly higher to use context creatively
            "max_output_tokens": 2048,
            "response_mime_type": "application/json"
        }

        response = model.generate_content(prompt, generation_config=generation_config)

        # Parse response
        try:
            predictions = json.loads(response.text)
        except (json.JSONDecodeError, TypeError):
            import re
            match = re.search(r'\[.*\]', response.text, re.DOTALL)
            if match:
                predictions = json.loads(match.group())
            else:
                raise ValueError("Could not parse LLM response")

        # Apply predictions
        result_df = masked_df.copy()
        for pred in predictions:
            idx = int(pred['index'])
            if idx in test_indices:
                weight = float(pred['weight_kg'])

                # Validate weight is reasonable
                h_m = masked_df.loc[idx, 'height_cm'] / 100
                bmi = weight / (h_m**2)

                if 15 <= bmi <= 40:  # Sanity check
                    result_df.loc[idx, 'weight_kg'] = weight
                    # Store reasoning if available
                    if 'reasoning' in pred and 'llm_weight_reasoning' not in result_df.columns:
                        result_df['llm_weight_reasoning'] = None
                    if 'reasoning' in pred:
                        result_df.loc[idx, 'llm_weight_reasoning'] = pred['reasoning']
                else:
                    # Fallback to regression
                    result_df.loc[idx, 'weight_kg'] = float(reg.predict([[masked_df.loc[idx, 'height_cm']]])[0])

        # Print sample reasoning if available
        if 'llm_weight_reasoning' in result_df.columns:
            print("\n   Sample LLM reasoning:")
            for idx in test_indices[:2]:  # Show first 2
                if pd.notna(result_df.loc[idx, 'llm_weight_reasoning']):
                    print(f"   Patient {idx}: {result_df.loc[idx, 'llm_weight_reasoning']}")

        return result_df

    except Exception as e:
        print(f"   LLM error: {e}, using regression fallback")
        # Fallback to regression
        result_df = masked_df.copy()
        for idx in test_indices:
            h = masked_df.loc[idx, 'height_cm']
            result_df.loc[idx, 'weight_kg'] = float(reg.predict([[h]])[0])
        return result_df

def _fit_transform_knn(train: pd.DataFrame, test: pd.DataFrame, cols=('height_cm','weight_kg')) -> pd.DataFrame:
    from sklearn.impute import KNNImputer
    from sklearn.preprocessing import StandardScaler

    cols = list(cols)

    scaler = StandardScaler()
    train_scaled = pd.DataFrame(scaler.fit_transform(train[cols].astype(float)), index=train.index, columns=cols)
    test_scaled = pd.DataFrame(scaler.transform(test[cols].astype(float)), index=test.index, columns=cols)

    n_neighbors = max(1, min(5, train_scaled.dropna().shape[0] - 1))
    imputer = KNNImputer(n_neighbors=n_neighbors)
    imputer.fit(train_scaled.values)

    test_imputed_scaled = pd.DataFrame(imputer.transform(test_scaled.values), index=test.index, columns=cols)
    test_imputed = pd.DataFrame(scaler.inverse_transform(test_imputed_scaled), index=test.index, columns=cols)

    out = test.copy()
    out.loc[:, cols] = test_imputed
    return out

def _fit_transform_mice(train: pd.DataFrame, test: pd.DataFrame, cols=('height_cm','weight_kg')) -> pd.DataFrame:
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    cols = list(cols)
    imp = IterativeImputer(random_state=42, max_iter=10)
    imp.fit(train[list(cols)].astype(float).values)
    test_vals = imp.transform(test[list(cols)].astype(float).values)
    out = test.copy()
    out.loc[:, cols] = test_vals
    return out

def evaluate_imputation_methods(ground_truth_df: pd.DataFrame,
                                masked_df: pd.DataFrame,
                                model=None) -> Dict[str, Optional[pd.DataFrame]]:
    """
    Train on rows with observed weight, predict only the masked rows.
    """
    results: Dict[str, Optional[pd.DataFrame]] = {}

    train_idx = masked_df['weight_kg'].notna()
    test_idx = masked_df['weight_kg'].isna()

    train_df = masked_df.loc[train_idx].copy()
    test_df = masked_df.loc[test_idx].copy()
    test_indices = masked_df.index[test_idx].to_numpy()

    if len(train_df) < 3:
        raise ValueError("Not enough training rows for evaluation.")

    # 1) KNN
    print("\n🔧 Testing KNN imputation...")
    try:
        pred = _fit_transform_knn(train_df, test_df, cols=('height_cm','weight_kg'))
        knn_df = masked_df.copy()
        knn_df.loc[test_idx, ['height_cm','weight_kg']] = pred[['height_cm','weight_kg']]
        results['KNN'] = knn_df
    except Exception as e:
        print(f"   KNN failed: {e}")
        results['KNN'] = None

    # 2) Linear regression
    print("🔧 Testing Regression imputation...")
    try:
        from sklearn.linear_model import LinearRegression
        reg = LinearRegression().fit(train_df[['height_cm']].values, train_df['weight_kg'].values)
        reg_pred = reg.predict(test_df[['height_cm']].values)
        reg_df = masked_df.copy()
        reg_df.loc[test_idx, 'weight_kg'] = reg_pred
        results['Regression'] = reg_df
        print(f"   Regression model: weight = {reg.coef_[0]:.3f} * height + {reg.intercept_:.1f}")
    except Exception as e:
        print(f"   Regression failed: {e}")
        results['Regression'] = None

    # 3) MICE
    print("🔧 Testing MICE imputation...")
    try:
        pred = _fit_transform_mice(train_df, test_df, cols=('height_cm','weight_kg'))
        mice_df = masked_df.copy()
        mice_df.loc[test_idx, ['height_cm','weight_kg']] = pred[['height_cm','weight_kg']]
        results['MICE'] = mice_df
    except Exception as e:
        print(f"   MICE failed: {e}")
        results['MICE'] = None

    # 4) Context-rich LLM
    print("🔧 Testing LLM imputation (with full context)...")
    if model is not None:
        try:
            llm_df = llm_weight_imputation_contextual(masked_df, train_df, test_indices, model)
            results['LLM'] = llm_df
        except Exception as e:
            print(f"   LLM failed: {e}")
            results['LLM'] = None
    else:
        print("⚠️ No model available for LLM")
        results['LLM'] = None

    return results

def calculate_metrics_weight_only(ground_truth_df: pd.DataFrame,
                                  imputed_df: pd.DataFrame,
                                  mask_indices: np.ndarray) -> Dict:
    from sklearn.metrics import mean_absolute_error, r2_score

    true_w = ground_truth_df.loc[mask_indices, 'weight_kg'].values
    imp_w = imputed_df.loc[mask_indices, 'weight_kg'].values

    valid = np.isfinite(true_w) & np.isfinite(imp_w)
    metrics = {'weight': {}}

    if valid.any():
        mae = float(mean_absolute_error(true_w[valid], imp_w[valid]))
        rmse = float(np.sqrt(np.mean((true_w[valid] - imp_w[valid]) ** 2)))
        r2 = float(r2_score(true_w[valid], imp_w[valid]))
        metrics['weight'] = {'mae': mae, 'rmse': rmse, 'r2': r2, 'n': int(valid.sum())}
        metrics['combined_mae'] = mae
        metrics['combined_r2'] = r2
    else:
        metrics['combined_mae'] = float('inf')
        metrics['combined_r2'] = -1.0

    return metrics

def bootstrap_metric_distributions(ground_truth_df: pd.DataFrame,
                                   mask_indices: np.ndarray,
                                   results: Dict[str, Optional[pd.DataFrame]],
                                   n_boot: int = 2000,
                                   random_state: int = 0) -> Dict[str, Dict[str, np.ndarray]]:
    """
    Paired bootstrap of R2 and MAE over the held-out patients.

    Every method is scored on exactly the same rows, so each draw resamples the
    rows once and re-scores every method on that same resample. Preserving the
    pairing is what makes the *difference* between two methods far better
    determined than either score on its own: the patients that are hard for one
    method are hard for all of them, and that shared difficulty cancels.

    A useful consequence for R2 specifically: within a single resample every
    method shares the same denominator (the variance of the resampled true
    weights), so comparing R2 between methods is exactly equivalent to
    comparing their squared error. The ranking is not an artefact of R2.

    Returns {method: {'r2': array, 'mae': array}}.
    """
    true_w = ground_truth_df.loc[mask_indices, 'weight_kg'].to_numpy(dtype=float)
    n = len(true_w)
    if n == 0:
        return {}

    rng = np.random.default_rng(random_state)
    draws = rng.integers(0, n, size=(n_boot, n))
    resampled_true = true_w[draws]
    # Denominator of R2: variance of the held-out values within each resample
    sst = ((resampled_true - resampled_true.mean(axis=1, keepdims=True)) ** 2).sum(axis=1)

    distributions: Dict[str, Dict[str, np.ndarray]] = {}

    for method in ["KNN", "Regression", "MICE", "LLM"]:
        frame = results.get(method)
        if frame is None:
            continue
        pred = frame.loc[mask_indices, 'weight_kg'].to_numpy(dtype=float)
        residual = pred - true_w
        if not np.isfinite(residual).all():
            continue
        resampled_residual = residual[draws]
        sse = (resampled_residual ** 2).sum(axis=1)
        with np.errstate(divide='ignore', invalid='ignore'):
            r2 = np.where(sst > 0, 1.0 - sse / sst, np.nan)
        distributions[method] = {
            'r2': r2,
            'mae': np.abs(resampled_residual).mean(axis=1),
        }

    return distributions


def masking_stability_check(df: pd.DataFrame, n_mask: int, n_repeats: int = 25,
                            first_seed: int = 0) -> None:
    """
    Repeat the masking experiment on different draws with the cheap regression
    baseline, and report how far a single split can move.

    This is the context needed to read the table above. On a held-out set of a
    dozen records the standard deviation of R\u00b2 is of the same order as R\u00b2 itself,
    so one negative value is not evidence that a method failed. A negative R\u00b2
    only means the prediction did worse than using the mean of the held-out
    values, which is a quantity nobody has access to at prediction time.
    """
    maes, r2s = [], []
    for seed in range(first_seed, first_seed + n_repeats):
        try:
            truth, masked, indices, _ = prepare_evaluation_data_weight_only(
                df, n_mask=n_mask, random_state=seed, verbose=False
            )
        except ValueError:
            continue
        train = masked[masked['weight_kg'].notna()]
        if len(train) < 3:
            continue
        reg = LinearRegression().fit(train[['height_cm']].values, train['weight_kg'].values)
        pred = reg.predict(masked.loc[indices, ['height_cm']].values)
        actual = truth.loc[indices, 'weight_kg'].values
        maes.append(mean_absolute_error(actual, pred))
        r2s.append(r2_score(actual, pred))

    if not r2s:
        print("\n\U0001f3b2 Stability check skipped: not enough complete records")
        return

    maes, r2s = np.array(maes), np.array(r2s)
    print(f"\n\U0001f3b2 How much does one split matter? "
          f"({len(r2s)} draws, regression baseline, n_mask={n_mask})")
    print(f"   MAE {maes.mean():.2f} \u00b1 {maes.std():.2f} kg")
    print(f"   R\u00b2  {r2s.mean():+.3f} \u00b1 {r2s.std():.3f}, "
          f"negative in {100 * (r2s < 0).mean():.0f}% of draws")
    print("   This is how much the MAE *level* moves when the split is redrawn.")
    print("   It does not contradict the paired result above: that comparison scores")
    print("   every method on identical patients, which cancels this shared noise.")


def create_final_visualization(ground_truth_df: pd.DataFrame,
                               mask_indices: np.ndarray,
                               results: Dict[str, Optional[pd.DataFrame]],
                               all_metrics: Dict[str, Dict],
                               bootstrap: Optional[Dict[str, np.ndarray]] = None) -> str:
    fig = plt.figure(figsize=(16, 6))

    # Panel 1: per-patient paired error
    #
    # Every method predicts the SAME held-out weights, so this is a paired
    # comparison. One row per patient keeps that pairing visible, and an error
    # read as a horizontal offset from zero is far easier to judge than a
    # perpendicular distance from a 45-degree line. It also shows *which*
    # patients are hard, which an overlaid scatter cannot.
    ax1 = plt.subplot(1, 3, 1)
    colors = {"KNN": "#2ecc71", "Regression": "#e67e22", "MICE": "#e74c3c", "LLM": "#9b59b6"}
    markers = {"KNN": "o", "Regression": "s", "MICE": "^", "LLM": "D"}

    true_w = ground_truth_df.loc[mask_indices, 'weight_kg'].values
    order = np.argsort(true_w)
    ordered_idx = np.asarray(mask_indices)[order]
    ordered_true = true_w[order]
    y_pos = np.arange(len(ordered_idx))

    # Flag rows whose free-text note quotes a weight. Those are exactly the
    # rows a text-reading method can recover almost perfectly and the numeric
    # methods cannot use at all, which is where the LLM earns its margin.
    quotes_weight = []
    for idx in ordered_idx:
        note = ""
        if 'comments_english' in ground_truth_df.columns:
            value = ground_truth_df.loc[idx, 'comments_english']
            note = "" if pd.isna(value) else str(value)
        quotes_weight.append(bool(re.search(r"\d+\s*(kg|kilo)", note, flags=re.IGNORECASE)))

    for y in y_pos:
        ax1.axhline(y, color="#e8e8e8", linewidth=0.8, zorder=0)
    ax1.axvline(0, color="black", linewidth=1.4, zorder=1)

    for method in ["KNN", "Regression", "MICE", "LLM"]:
        if method not in results or results[method] is None or method not in all_metrics:
            continue
        imp_w = results[method].loc[ordered_idx, 'weight_kg'].values
        mae = all_metrics[method]['weight'].get('mae', np.nan)
        ax1.scatter(imp_w - ordered_true, y_pos, s=55, marker=markers[method],
                    color=colors[method], edgecolors="black", linewidth=0.4,
                    alpha=0.85, zorder=3, label=f"{method} (MAE {mae:.1f} kg)")

    ax1.set_yticks(y_pos)
    ax1.set_yticklabels([f"{'* ' if q else ''}{t:.0f}" for t, q in zip(ordered_true, quotes_weight)],
                        fontsize=8)
    ax1.set_ylim(-0.9, len(y_pos) - 0.1)
    ax1.set_xlabel("Imputation error (kg)   left = under, right = over", fontsize=10)
    ax1.set_ylabel("Held-out patient, by true weight (kg)", fontsize=10)
    ax1.set_title(f"Per-patient error, same {len(y_pos)} held-out weights",
                  fontsize=12, fontweight="bold")
    # Upper right is the one reliably empty corner: the large errors all sit
    # on the heavy patients at the top left.
    ax1.legend(loc="upper right", fontsize=8, framealpha=0.92)
    if any(quotes_weight):
        # Bottom left is empty: the light patients sit at or right of zero.
        ax1.text(0.02, 0.02, "* clinical note quotes a weight",
                 transform=ax1.transAxes, fontsize=7.5, color="#555555")

    # Panel 2: R2 on the held-out weights
    #
    # R2 is the intuitive score here: the share of variance in the held-out
    # weights that a method explains, which is how the theory sections above are
    # written. Deliberately no error bars. The marginal bootstrap interval on
    # R2 is about [-0.41, +0.56] for every method, so whiskers would be huge,
    # near-identical between bars, and would drown the very differences the
    # panel exists to show. The uncertainty that actually matters for a
    # comparison is the paired one, and it is reported under each bar as the
    # share of resamples in which the leader wins. The marginal intervals are
    # printed in the text above, where a number can be read without implying a
    # visual comparison it does not support.
    ax2 = plt.subplot(1, 3, 2)
    boot = bootstrap if bootstrap is not None else bootstrap_metric_distributions(
        ground_truth_df, mask_indices, results)

    methods = [m for m in ["KNN", "Regression", "MICE", "LLM"]
               if m in all_metrics and all_metrics[m].get('weight')]
    methods.sort(key=lambda m: all_metrics[m]['weight']['r2'], reverse=True)
    r2_vals = [all_metrics[m]['weight']['r2'] for m in methods]

    bars = ax2.bar(methods, r2_vals, color=[colors[m] for m in methods],
                   edgecolor="black", linewidth=1, alpha=0.85, zorder=2)

    for bar, value in zip(bars, r2_vals):
        offset = 0.012 if value >= 0 else -0.012
        ax2.text(bar.get_x() + bar.get_width() / 2, value + offset, f"{value:+.3f}",
                 ha="center", va="bottom" if value >= 0 else "top",
                 fontweight="bold", fontsize=10)

    # Zero is the reference that gives R2 its meaning: below it, the method did
    # worse than simply predicting the mean of the held-out weights.
    ax2.axhline(0, color="gray", linestyle="--", alpha=0.7, zorder=1)

    leader = methods[0] if methods else None
    tick_labels = []
    for method in methods:
        if method == leader or leader is None or method not in boot or leader not in boot:
            tick_labels.append(method)
        else:
            wins = 100.0 * float(np.nanmean(boot[leader]['r2'] > boot[method]['r2']))
            tick_labels.append(f"{method}\n({leader} wins {wins:.0f}%)")
    ax2.set_xticks(range(len(methods)))
    ax2.set_xticklabels(tick_labels, fontsize=8)

    span_lo = min(list(r2_vals) + [0.0])
    span_hi = max(list(r2_vals) + [0.0])
    margin = max(0.08, 0.35 * (span_hi - span_lo))
    ax2.set_ylim(span_lo - margin, span_hi + margin)

    ax2.set_ylabel("R\u00b2 on the held-out weights, higher is better", fontsize=10)
    ax2.set_title("Variance explained", fontsize=12, fontweight="bold")
    ax2.grid(True, alpha=0.2, axis="y")
    ax2.text(0.5, 0.985,
             f"R\u00b2 > 0 beats predicting the mean of the held-out weights\n"
             f"labels: share of paired resamples won by the leader (n={len(mask_indices)})",
             transform=ax2.transAxes, ha="center", va="top",
             fontsize=7.5, color="#555555")

    # Panel 3: Summary
    ax3 = plt.subplot(1, 3, 3)
    ax3.axis("off")

    best_method = min(all_metrics.items(), key=lambda x: x[1].get('combined_mae', float('inf')))[0]

    summary = f"""Evaluation Summary
{'='*35}

📊 Test Setup:
  • {len(mask_indices)} weights masked (stratified)
  • Training size: {len(ground_truth_df) - len(mask_indices)} patients

Method Performance:
{'─'*35}
"""

    for method in sorted(all_metrics.keys(), key=lambda x: all_metrics[x].get('combined_mae', float('inf'))):
        mae = all_metrics[method]['weight'].get('mae', np.nan)
        r2 = all_metrics[method].get('combined_r2', np.nan)
        marker = "→" if method == best_method else " "
        summary += f"{marker} {method:12} MAE: {mae:5.2f} kg  R²: {r2:+.3f}\n"

    summary += f"""
{'─'*35}
🏆 Best Method: {best_method}

Key Insights:
- Height-weight correlation enables
  accurate regression-based prediction
- LLM leverages contextual information
  and most likely will excel with real-world data
- All methods significantly outperform
  naive baseline approaches
"""

    ax3.text(0.05, 0.5, summary, fontsize=9, family="monospace",
            verticalalignment="center", transform=ax3.transAxes,
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f9fa",
                     edgecolor="#dee2e6", linewidth=1))

    plt.suptitle("Weight Imputation Performance Evaluation",
                fontsize=14, fontweight="bold", y=0.98)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig("final_imputation_evaluation.png", dpi=150, bbox_inches="tight")
    plt.show()

    return best_method

# Main execution
print("\n" + "="*60)
print("FINAL EVALUATION: WEIGHT IMPUTATION WITH FULL CONTEXT")
print("="*60)

evaluation_df = standardized_df if 'standardized_df' in locals() else df

# Prepare evaluation data
# n_mask is deliberately larger than a handful: R\u00b2 computed on 6 held-out values has a
# standard deviation of about 0.33 on this dataset, so it turns negative in roughly one
# run in five for reasons that have nothing to do with the imputation method.
ground_truth_df, masked_df, mask_indices, mask_ids = prepare_evaluation_data_weight_only(
    evaluation_df, n_mask=15, random_state=42
)

print(f"\n🎯 Masked patients: {mask_ids if mask_ids else mask_indices.tolist()}")

# Show what context is available
context_cols = ['gender', 'clinic', 'age', 'date_measured', 'comments_english']
available_context = [col for col in context_cols if col in masked_df.columns]
print(f"📋 Available context fields: {available_context}")

# Evaluate methods
model_to_use = model if 'model' in locals() and model is not None else None
results = evaluate_imputation_methods(ground_truth_df, masked_df, model_to_use)

# Calculate metrics
print("\n📊 Performance Metrics:")
print("-" * 40)

all_metrics = {}
for method, imputed_df in results.items():
    if imputed_df is not None:
        metrics = calculate_metrics_weight_only(ground_truth_df, imputed_df, mask_indices)
        all_metrics[method] = metrics

        if 'weight' in metrics and metrics['weight']:
            print(f"{method:12} MAE: {metrics['weight']['mae']:6.2f} kg   "
                  f"RMSE: {metrics['weight']['rmse']:6.2f} kg   "
                  f"R²: {metrics['weight']['r2']:+.3f}   n={metrics['weight']['n']}")

# Paired bootstrap: resample the held-out patients and re-score every method on
# the same resample. The gap between two methods is far better determined than
# either score alone, because a patient who is hard for one is hard for all.
boot_dist = bootstrap_metric_distributions(ground_truth_df, mask_indices, results)
if boot_dist:
    print("\n\U0001f501 Paired bootstrap over the held-out patients (2000 resamples):")
    ranked = sorted(boot_dist, key=lambda m: all_metrics[m]['weight']['r2'], reverse=True)
    for method in ranked:
        lo_q, hi_q = np.nanpercentile(boot_dist[method]['r2'], [2.5, 97.5])
        print(f"   {method:12} R\u00b2 {all_metrics[method]['weight']['r2']:+.3f}"
              f"   95% CI [{lo_q:+.3f}, {hi_q:+.3f}]")
    leader = ranked[0]
    for method in ranked[1:]:
        wins_r2 = float(np.nanmean(boot_dist[leader]['r2'] > boot_dist[method]['r2']))
        wins_mae = float(np.mean(boot_dist[leader]['mae'] < boot_dist[method]['mae']))
        print(f"   {leader} beats {method:11} in {100 * wins_r2:3.0f}% of resamples on R\u00b2"
              f"   and {100 * wins_mae:3.0f}% on MAE")
    print("   The intervals are wide because the R\u00b2 *level* depends heavily on which")
    print("   patients were drawn. The two win rates disagree for an instructive reason:")
    print("   R\u00b2 is built on squared error, so it is dominated by the few heavy patients")
    print("   that every method misses badly. MAE weights each patient equally, which is")
    print("   where a method that fixes several mid-range rows gets credit.")

# How stable is a single split? Cheap to answer with the regression baseline.
masking_stability_check(evaluation_df, n_mask=len(mask_indices), n_repeats=25)

# Create visualization
print("\n🎨 Creating visualization...")
best_method = create_final_visualization(ground_truth_df, mask_indices, results,
                                        all_metrics, bootstrap=boot_dist)

# Show detailed predictions
print("\n📍 Detailed Predictions:")
for i, idx in enumerate(mask_indices[:3]):
    true_w = ground_truth_df.loc[idx, 'weight_kg']
    true_h = ground_truth_df.loc[idx, 'height_cm']

    _pid = ground_truth_df.loc[idx, 'patient_id'] if 'patient_id' in ground_truth_df.columns else idx
    print(f"\nPatient {_pid} (row {idx}):")
    print(f"  Height: {true_h:.1f}cm")
    print(f"  True Weight: {true_w:.1f}kg")

    # Show context if available
    if 'gender' in masked_df.columns and pd.notna(masked_df.loc[idx, 'gender']):
        print(f"  Gender: {masked_df.loc[idx, 'gender']}")
    if 'clinic' in masked_df.columns and pd.notna(masked_df.loc[idx, 'clinic']):
        print(f"  Clinic: {masked_df.loc[idx, 'clinic']}")

    print("  Predictions:")
    for method in ["Regression", "LLM"]:
        if method in results and results[method] is not None:
            pred_w = results[method].loc[idx, 'weight_kg']
            error = pred_w - true_w
            bmi = pred_w / ((true_h/100)**2)
            print(f"    {method:12} → {pred_w:.1f}kg (error: {error:+.1f}kg, BMI: {bmi:.1f})")

print("\n" + "="*60)
print(f"✅ Evaluation complete. Best method: {best_method}")
print("="*60)

# Ohh... One more thing: authenticity audit rationale

## What are we going to do here?
This section introduces a minimal authenticity check placed before the code. A small random sample of 50 records is shown to a large language model, which must decide whether the records appear real or synthetic based only on the sample itself. The goal is to reveal distributional artefacts early and to set expectations about how such a simple audit should be interpreted.

## Why the LLM may misjudge
- Sampling sensitivity: a 50-row slice can over-represent benign irregularities or hide genuine structure.
- Cross-centre heterogeneity: multilingual entries and mixed date formats can be legitimate in real, multi-site datasets.
- Over-normalisation upstream: previous cleaning may create patterns that look artificially neat.
- Weak confidence calibration: deterministic decoding can overstate certainty on thin evidence.
- Surface-level cues: the model may focus on formatting quirks rather than physiological plausibility.

## Why synthetic training can degrade performance
- Distribution shift: simulated data often has compressed variance and cleaner correlations than real clinical records.
- Template leakage: reused text or repeated format patterns become shortcuts that the model memorises.
- Digit and rounding bias: excessive .0 or .5 endings and perfectly sequential identifiers teach unrealistic habits.
- Feedback loops: training successive models on model-generated data amplifies artefacts and reduces robustness.
- Inflated evaluation: metrics measured on synthetic or semi-synthetic validation do not guarantee real-world utility.

## Practical guidance
- Prefer governed real data where possible; when synthetic data is necessary, benchmark it against real distributions and document the gaps.
- Mix synthetic with small, representative real slices; run masking experiments and check physiological bounds.
- Add controlled randomness to synthetic pipelines to avoid over-regularisation and exact quotas.
- Keep the authenticity audit in continuous integration and fail on neat patterns or repetitive templates.
- Calibrate confidence on held-out real samples and escalate ambiguous cases to experts.

## Summary
A minimal authenticity audit is a useful early warning. If the model flags synthetic cues or appears overconfident, revisit data construction and reduce reliance on purely artificial training corpora before proceeding with downstream standardisation, imputation, or evaluation.

In [ ]:
#@title Section 10: Simple authenticity audit

# UI: quick model selector + temperature + init + run
_opts = [(v['name'], k) for k, v in MODEL_CATALOG.items()]
_sel = widgets.Dropdown(options=_opts, value=SELECTED_MODEL, description='Model:', layout=widgets.Layout(width='360px'))
_temp = widgets.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description='Temp:', readout_format='.2f',
                            layout=widgets.Layout(width='220px'))
_btn_init = widgets.Button(description="Init", button_style='success', layout=widgets.Layout(width='80px'))
_btn_run  = widgets.Button(description="Run audit", button_style='primary', layout=widgets.Layout(width='140px'))
_out = widgets.Output()

def _init(_):
    global SELECTED_MODEL, model
    SELECTED_MODEL = _sel.value
    with _out:
        clear_output(wait=True)
        try:
            if MODE or str(SELECTED_MODEL).startswith(("fred/", "ollama/")):   # Ollama servers need no SDK
                model = UnifiedModelAdapter(SELECTED_MODEL)
                _ = model.generate_content("ok", {"temperature": float(_temp.value), "max_output_tokens": 8})
                display(HTML(f"<div style='padding:8px;border-radius:6px;background:#e8f5e9;'>✅ Model ready: <b>{SELECTED_MODEL}</b> | Temp: {_temp.value:.2f}</div>"))
            else:
                display(HTML("<div style='padding:8px;border-radius:6px;background:#fff3cd;'>⚠️ No LLM SDK available in this runtime</div>"))
        except Exception as e:
            model = None
            display(HTML(f"<div style='padding:8px;border-radius:6px;background:#fdecea;'>❌ Model init failed: {e}</div>"))

def _load_df():
    try:
        return df.copy()
    except NameError:
        pass
    _loader = data_loader if 'data_loader' in globals() else EnhancedDataLoader()
    try:
        if not getattr(_loader, 'authenticated', False): _loader.authenticate_google()
    except Exception:
        pass
    _df = None
    try:
        _df = _loader.load_from_saved_sheet()
    except Exception:
        _df = None
    if _df is None and os.path.exists('medical_data_sheet_id.txt'):
        try:
            with open('medical_data_sheet_id.txt','r') as _f:
                _id = _f.read().strip()
            _df = _loader.load_by_sheet_id(_id)
        except Exception:
            _df = None
    assert _df is not None and len(_df)>0, "No saved Google Sheet found. Run Section 3 to create and save data."
    return _df

def _colour(verdict):
    if verdict == "real": return "#1b5e20"
    if verdict == "synthetic": return "#b71c1c"
    return "#424242"

def _split_reasons(reasons, fallback_text=""):
    import re
    parts = []
    if isinstance(reasons, str):
        parts = [reasons]
    elif isinstance(reasons, list):
        parts = [r for r in reasons if isinstance(r, str)]
    elif fallback_text:
        parts = [fallback_text]
    split_parts, bullets, seen = [], [], set()
    for p in parts:
        for seg in re.split(r'(?:\s*[;•]\s*|\n|(?<=[.!?])\s+)', p):
            seg = seg.strip().strip("-•").strip()
            if seg: split_parts.append(seg)
    for s in split_parts:
        s = s.strip()
        if not s or s.lower() in seen: continue
        seen.add(s.lower()); bullets.append(s[:220])
        if len(bullets) >= 6: break
    return bullets or ["No clear rationale provided by the model."]

def _audit(_):
    with _out:
        clear_output(wait=True)
        _data = _load_df()
        cols_pref = ['patient_id','age','gender','height_cm','weight_kg','height','weight',
                     'language','clinic','date_measured','comments']
        cols = [c for c in cols_pref if c in _data.columns]
        sample = _data[cols].sample(min(50, len(_data)), random_state=42)

        prompt = f"""
Role:
You are an auditor reviewing a small tabular health dataset.
Your job is to judge whether the dataset is likely real-world or synthetic and
to justify your judgement using concrete evidence from the table.

Task:
Decide whether these records are real-world clinical data or artificially generated.
Base your judgement only on the records provided here.
Present at least 3 compeling arguments supporting your decision.

Records:
{sample.to_json(orient='records', indent=2, force_ascii=False)}

Output format (strict JSON, no extra text):
{{
  "verdict": "real" | "synthetic" | "uncertain",
  "confidence": 0.00-1.00,
  "reasons": ["brief, evidence-based reasons"]
}}
"""
        if 'model' not in globals() or model is None:
            display(HTML("<div style='padding:8px;border-radius:6px;background:#fff3cd;'>Init the model, then click <b>Run audit</b>.</div>"))
            return

        try:
            cfg = {"temperature": float(_temp.value), "max_output_tokens": 1024, "response_mime_type": "application/json"}
            resp = model.generate_content(prompt, generation_config=cfg)
            raw = (resp.text or "").strip().strip("`")
            payload = None
            try:
                i, j = raw.find("{"), raw.rfind("}")
                s = raw[i:j+1] if i != -1 and j != -1 else raw
                payload = json.loads(s)
            except Exception:
                payload = None

            if payload and isinstance(payload, dict):
                verdict = str(payload.get("verdict","uncertain")).lower()
                conf = payload.get("confidence", None)
                reasons = payload.get("reasons", []) or payload.get("arguments", [])
                bullets = _split_reasons(reasons, fallback_text=raw)
                colour = _colour(verdict)
                conf_txt = f"{conf:.2f}" if isinstance(conf, (int,float)) else "n/a"
                html = f"""
                <div style="border:1px solid {colour};border-radius:10px;padding:14px 16px;margin:4px 0;">
                  <div style="font-size:22px;font-weight:700;color:{colour};margin-bottom:6px;">
                    Decision: {verdict.upper()} &nbsp;|&nbsp; Confidence: {conf_txt}
                  </div>
                  <ul style="margin:4px 0 0 18px;padding:0;">
                    {''.join(f"<li style='margin:2px 0;font-size:14px;'>{b}</li>" for b in bullets)}
                  </ul>
                  <div style="font-size:12px;color:#666;margin-top:8px;">
                    Model: {SELECTED_MODEL if 'SELECTED_MODEL' in globals() else 'n/a'} &nbsp;|&nbsp; Temp: {_temp.value:.2f} &nbsp;|&nbsp; Sampled records: {len(sample)}
                  </div>
                </div>
                """
                display(HTML(html))
            else:
                display(HTML("<div style='padding:8px;border-radius:6px;background:#fdecea;'>The model did not return valid JSON. Raw output:</div>"))
                print(raw[:2000] + ("\n...\n" if len(raw)>2000 else ""))

        except Exception as e:
            display(HTML(f"<div style='padding:8px;border-radius:6px;background:#fdecea;'>LLM call failed: {e}</div>"))

        from datetime import datetime as _dt
        meta = f"Generated: {_dt.now().isoformat()} | Env: {ENVIRONMENT if 'ENVIRONMENT' in globals() else 'unknown'} | Fast: {FAST if 'FAST' in globals() else 'unknown'}"
        display(HTML(f"<div style='font-size:12px;color:#666;margin-top:6px;'>{meta}</div>"))

_btn_init.on_click(_init)
_btn_run.on_click(_audit)
display(widgets.HBox([_sel, _temp, _btn_init, _btn_run]), _out)


# Conclusions and Experiments to Take Further

## What We Discovered

**The power of context**: Classical methods see numbers, LLMs see stories. A 180cm height means something different for a basketball player vs an elderly patient with osteoporosis.

**Engineering trade-offs matter**: Perfect accuracy isn't always worth 100x the cost. The best solution often combines fast classical methods with targeted LLM enhancement.

**Validation drives decisions**: Without masking experiments and clinical review, you're just guessing whether your imputation helps or hurts downstream analysis.

**Data quality**: The synthetic artefacts can distort the model evaluation. Synthetic patterns can make some models look better than they are and the other way around by hiding failure modes that matter in practice.

---

## Conclusion

For biomechanics research, the choice between classical and LLM-based imputation depends on your specific research context, data characteristics, and quality requirements. Classical methods provide reliable, fast results for most scenarios, while LLM methods excel when rich contextual information can improve accuracy and biological plausibility.

## Decision Framework You Can Use

| Use Case | Method | Why |
|----------|--------|-----|
| **Emergency triage** | Mean/KNN | Speed critical, good enough accuracy |
| **Clinical research** | MICE + LLM validation | Accuracy critical, cost acceptable |
| **Population health** | Classical ensemble | Large scale, cost constraints |
| **Precision medicine** | LLM with rich context | Individual optimization matters |

## Practical Implementation Guide


1. **Assess missing data patterns** - understand why data is missing
2. **Establish classical baseline** - implement and validate traditional methods
3. **Identify high-value cases** - participants where context matters most
4. **Implement LLM enhancement** - for complex cases requiring domain knowledge
5. **Validate results** - ensure biomechanical relationships are preserved
6. **Document methodology** - for research reproducibility

---

## Experiments to take further

### 🧪 Experiment 1: The Context Advantage Test
**Goal**: Prove LLMs actually use context, not just fancy statistical correlation.

**Steps**:
1. Create identical twins in the dataset (same height, weight, age, gender)
2. Give one twin context: "Former basketball player, recent knee surgery"  
3. Give the other: "Office worker, chronic back pain"
4. Mask and then impute the weight of both
5. **Hypothesis**: LLM should give different results; classical methods won't

**What to look for**:
- LLMs produce meaningfully different imputations (>5% difference)
- Classical methods produce identical results
- LLM reasoning mentions context factors

The last cell of the notebook runs this experiment: the two records, the LLM estimate and the classical estimates side by side.

---

### 🧪 Experiment 2: The Failure Point Discovery
**Goal**: Find where each method breaks down spectacularly.

**Ideas**:
1. **Stress test classical methods**:
   - Try KNN with 90% missing data
   - Use regression with completely unrelated variables
   - Apply mean imputation to bimodal distributions

2. **Stress test LLM methods**:
   - Feed contradictory context ("7-year-old weightlifter")
   - Use medical jargon it shouldn't know
   - Request impossible precision ("height to nearest 0.01mm")

**Expected discoveries**:
- Classical methods fail silently (bad imputations, no warnings)
- LLMs fail loudly (weird reasoning, low confidence)
- Each has predictable failure modes you can defend against

---

### 🧪 Experiment 3: The Cost-Accuracy Frontier
**Goal**: Build the optimal hybrid system for different budgets.

**Idea**: a function `optimal_imputation_strategy(budget_per_record, accuracy_target)` that chooses the method for each record so that the accuracy is as high as the budget allows. Budget scenarios to compare:
- $0.001 per record (classical only)
- $0.01 per record (selective LLM use)
- $0.10 per record (full LLM with ensemble)

**Design considerations**:
- Use classical methods for "easy" missing values
- Trigger LLM for high-stakes or complex cases
- Implement quality gates: if classical confidence < threshold → escalate to LLM

**A target**: 95% of full-LLM accuracy at 20% of the cost

---

## Final Engineering Wisdom

**Start simple, measure everything, optimize strategically.**

Your classical methods baseline isn't just for comparison—it's your fallback when LLMs fail, your cost optimization target, and your sanity check.

**Context is power, but validation is truth.**

LLMs can hallucinate convincing but wrong medical reasoning. Never deploy without human expert validation and systematic quality monitoring.

**The best imputation method is the one you don't need ;-).**

Invest in better data collection before fancier imputation. But when you do need imputation, now you know how to do it right.

---

*Go build something that matters. Medical data affects real people, so make sure your missing data handling helps rather than harms.*

In [ ]:
#@title Experiment 1: the context test (identical twins, different notes)
# Two patients with the same height, age and gender. Only the comment differs; the weight is missing for both.
twins = pd.DataFrame([
    dict(patient_id="A", age=45, gender="Male", height_cm=185.0, weight_kg=np.nan,
         comments_english="Former basketball player, recent knee surgery"),
    dict(patient_id="B", age=45, gender="Male", height_cm=185.0, weight_kg=np.nan,
         comments_english="Office worker, chronic back pain"),
])

# The LLM reads the comment.
llm_twins = llm_imputation_complete(twins.copy(), model)

# The classical methods are fitted on the standardised records and see only the numbers.
known = standardized_df[["height_cm", "weight_kg"]].dropna()
reg = LinearRegression().fit(known[["height_cm"]].values, known["weight_kg"].values)
knn = KNNImputer(n_neighbors=5).fit(known.values)
numbers = twins[["height_cm", "weight_kg"]].values
result = pd.DataFrame({
    "comment": twins["comments_english"].values,
    "mean": known["weight_kg"].mean(),
    "regression": reg.predict(twins[["height_cm"]].values),
    "KNN": knn.transform(numbers)[:, 1],
    "LLM": pd.to_numeric(llm_twins["weight_kg"], errors="coerce").values,
}, index=["twin A", "twin B"]).round(1)
display(result)
diff = abs(result.loc["twin A", "LLM"] - result.loc["twin B", "LLM"])
print(f"LLM: the twins differ by {diff:.1f} kg. Classical methods: identical by construction.")
